<div style="background: brown; padding: 34px; border-radius: 16px; border: 4px solid #111111; text-align:center; font-family: Arial, sans-serif;">
<div style="font-size: 38px; font-weight: 800; color: white; text-shadow: 2px 2px 4px rgba(0,0,0,0.55);">BioLingua</div>
<div style="font-size: 21px; font-weight: 600; color: white; margin-top: 8px; text-shadow: 1px 1px 3px rgba(0,0,0,0.55);">One Complete Research Pipeline for Self-Supervised Bird Vocalization Analysis</div>
<div style="font-size: 15px; color: #FFF7E6; margin-top: 12px;">Speech and bird encoders • Acoustic unit discovery • Sequence language models • ABX tests • Behaviour probes • 25 controlled experiments • Paper-ready figures and tables</div>
</div>

## What this notebook is

This is **one single notebook** that consolidates the earlier exploratory runs (ten pilot runs, fine-tuned variants, and supervised and unsupervised pilots) into one reproducible pipeline.
Everything is built from classes and functions, so the same code path is used for every experiment and every number in the paper comes from one place.

**The big question of the paper**

> Can we learn the building blocks of bird sound (acoustic units) from raw audio without labels, model how these units follow each other, and see whether this structure carries information about species and behaviour (alarm, song, contact)?

**What is inside (in order)**

| Part | What happens | Main outputs |
|---|---|---|
| 1 | Setup, colour theme, 900 DPI figure vault, table vault | Global style |
| 2 | Configuration and presets (smoke, pilot, standard, full) | One config object |
| 3 | Data collection from Xeno-Canto (or Kaggle dataset, or synthetic data for a dry run) | Recording catalog |
| 4 | Weak behaviour labels from call type text | Labelled recordings |
| 5 | Audio cleaning and segmentation into a fast memory-mapped store | Segment store |
| 6 | Exploratory data analysis | Corpus, label and sound figures |
| 7 | Encoders: HuBERT, wav2vec2, BirdAVES, MFCC baseline | Encoder benchmark |
| 8 | Contrastive projection adapter with augmentations | Adapted embeddings |
| 9 | Acoustic unit discovery: K-Means, GMM, HDBSCAN, Ward, VQ autoencoder | Token vocabulary |
| 10 | Sequence language model with n-gram and shuffle controls | Perplexity results |
| 11 | ABX discrimination with recording-safe triplets | ABX results |
| 12 | Behaviour probes with recording-level cross-validation and a leakage audit | Behaviour results |
| 13 | Semi-supervised pseudo-labelling, ensembles, frame-level BiLSTM | Extra behaviour results |
| 14 | Syllable discovery pilot (frame-level tokens) | Pilot results |
| 15 | The 20-experiment ablation suite with multiple-comparison correction | Master table |
| 16 | Data scaling study and registry of all earlier runs | Scaling figures |
| 17 | Statistics, claim checker, dashboard, exports | Final tables, zip file |

**Colour code used in every plot**

- Primary colours: **orange** `#FF8C00` and **blue** `#1E90FF`
- Secondary colours: **green** `#32CD32`, **red** `#DC143C`, **purple** `#9370DB`
- Every chart has black outlines. Figures are shown on screen at Full HD size and saved to disk at **900 DPI** (PNG) and as vector PDF.

## How to run this on Kaggle

1. Open the notebook in Kaggle and set **Accelerator = GPU T4 x2** (or P100).
2. Set **Internet = On**.
3. Go to *Add-ons → Secrets* and add a secret named `XENO_CANTO_API_KEY` with your Xeno-Canto key. Never paste the key inside the notebook. If the secret is missing, the notebook asks for it with a hidden input box.
4. In the configuration cell of Part 2 choose one preset and press *Run all*.

| Preset | Data | Purpose | Rough time on a T4 |
|---|---|---|---|
| `smoke` | Synthetic audio and tiny random encoders | Checks that every cell works before you spend GPU hours | 5 to 15 minutes |
| `pilot` | 5 species x 60 recordings | Quick real run | 1 to 2 hours |
| `standard` | 5 species x 200 recordings | Main working run | 4 to 6 hours |
| `full` | 5 species x 300 recordings | Paper-scale run | 7 to 10 hours |

The times are rough guides only. Every heavy step saves its result to disk, so if Kaggle stops the session you can run the notebook again and it continues from the saved files.

## A note on honest evaluation

Earlier notebooks scored behaviour classifiers with random segment-level folds. Segments cut from the same recording are very similar, so the same recording could appear in both train and test folds. That inflates accuracy.
This notebook always splits by **recording**. It also contains a **leakage audit** (Part 12) that runs both ways on the same data, so you can show in the paper how large the inflation was. Because of this, numbers here can be lower than the numbers in the old Run notebooks. The lower numbers are the ones a reviewer can trust.

# Part 1: Setup

## 1.1 Packages, imports and hardware

This cell installs only what is missing, imports everything once, fixes all random seeds and reports the hardware. Nothing else in the notebook installs packages.

In [ ]:
import os, sys, gc, io, re, json, math, time, glob, shutil, random, hashlib, warnings, subprocess, itertools, importlib, zipfile, pickle, getpass, textwrap
from dataclasses import dataclass, field, asdict, replace
from pathlib import Path
from collections import Counter, defaultdict, OrderedDict
from concurrent.futures import ThreadPoolExecutor, as_completed


def ensure_packages(specs):
    missing = []
    for import_name, pip_name in specs:
        try:
            importlib.import_module(import_name)
        except Exception:
            missing.append(pip_name)
    if missing and os.environ.get("BIOLINGUA_SKIP_INSTALL") != "1":
        for pip_name in missing:
            print(f"installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)
    return missing


ensure_packages([
    ("transformers", "transformers"),
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("noisereduce", "noisereduce"),
    ("umap", "umap-learn"),
    ("xgboost", "xgboost"),
    ("seaborn", "seaborn"),
    ("networkx", "networkx"),
    ("tqdm", "tqdm"),
])

import numpy as np
import pandas as pd
import scipy
from scipy import stats as sps
from scipy.cluster import hierarchy as sch
from scipy.spatial.distance import pdist, squareform

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import cm, colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import librosa
import soundfile as sf
import networkx as nx
from tqdm.auto import tqdm
from IPython.display import display, HTML, Markdown

from sklearn.cluster import MiniBatchKMeans, KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.calibration import calibration_curve
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score, accuracy_score, f1_score,
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score,
)

try:
    import xgboost as xgb
except Exception:
    xgb = None

try:
    from sklearn.cluster import HDBSCAN as SkHDBSCAN
except Exception:
    SkHDBSCAN = None

try:
    import hdbscan as hdbscan_lib
except Exception:
    hdbscan_lib = None

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_KAGGLE = Path("/kaggle/working").exists()
CPU_COUNT = os.cpu_count() or 1

print("=" * 72)
print(f"device          : {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"gpu count       : {torch.cuda.device_count()}")
print(f"cpu cores       : {CPU_COUNT}")
print(f"running on      : {'Kaggle' if ON_KAGGLE else 'local or other'}")
print(f"torch / numpy   : {torch.__version__} / {np.__version__}")
print(f"pandas / sklearn: {pd.__version__} / {__import__('sklearn').__version__}")
print("=" * 72)

## 1.2 Colour palette and plot theme

All colours live in one `Palette` class. The theme puts **orange and blue first**, then green, red and purple. Axes, bars, legends and figure borders all get black outlines so every plot looks clean in a paper.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler


class Palette:
    ORANGE = "#FF8C00"
    BLUE = "#1E90FF"
    GREEN = "#32CD32"
    RED = "#DC143C"
    PURPLE = "#9370DB"
    ORANGE_DARK = "#C96F00"
    BLUE_DARK = "#0B5CAD"
    ORANGE_LIGHT = "#FFE2B8"
    BLUE_LIGHT = "#CFE8FF"
    INK = "#111111"
    GRID = "#D5D8DC"
    CATEGORICAL = [ORANGE, BLUE, GREEN, RED, PURPLE]
    BEHAVIOUR = {"alarm": RED, "song": BLUE, "contact": ORANGE}
    BEHAVIOUR_LIGHT = {"alarm": "#F7B3BF", "song": BLUE_LIGHT, "contact": ORANGE_LIGHT}

    @staticmethod
    def diverging():
        return LinearSegmentedColormap.from_list("blue_orange_div", [Palette.BLUE_DARK, Palette.BLUE, "#FFFFFF", Palette.ORANGE, Palette.ORANGE_DARK])

    @staticmethod
    def blues():
        return LinearSegmentedColormap.from_list("bio_blues", ["#FFFFFF", Palette.BLUE_LIGHT, Palette.BLUE, Palette.BLUE_DARK])

    @staticmethod
    def oranges():
        return LinearSegmentedColormap.from_list("bio_oranges", ["#FFFFFF", Palette.ORANGE_LIGHT, Palette.ORANGE, Palette.ORANGE_DARK])

    @staticmethod
    def blue_to_orange():
        return LinearSegmentedColormap.from_list("bio_b2o", [Palette.BLUE_DARK, Palette.BLUE, "#9CCBF5", "#FFD9A0", Palette.ORANGE, Palette.ORANGE_DARK])

    @staticmethod
    def spectrogram():
        return LinearSegmentedColormap.from_list("bio_spec", ["#07132B", Palette.BLUE_DARK, Palette.BLUE, Palette.ORANGE, "#FFF3DC"])

    @staticmethod
    def token_colors(n):
        base = Palette.blue_to_orange()(np.linspace(0.02, 0.98, max(n, 2)))
        order = np.argsort((np.arange(max(n, 2)) * 0.6180339887) % 1.0)
        return [tuple(base[i]) for i in order[:n]]

    @staticmethod
    def species_colors(names):
        return {name: Palette.CATEGORICAL[i % len(Palette.CATEGORICAL)] for i, name in enumerate(names)}


def apply_theme(display_dpi=150, export_dpi=900):
    plt.rcParams.update({
        "figure.dpi": display_dpi,
        "savefig.dpi": export_dpi,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": Palette.INK,
        "axes.linewidth": 1.6,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlecolor": Palette.ORANGE_DARK,
        "axes.titlepad": 10,
        "axes.labelsize": 11.5,
        "axes.labelweight": "bold",
        "axes.labelcolor": Palette.BLUE_DARK,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": cycler(color=Palette.CATEGORICAL),
        "grid.color": Palette.GRID,
        "grid.linestyle": "--",
        "grid.linewidth": 0.7,
        "grid.alpha": 0.9,
        "xtick.color": Palette.INK,
        "ytick.color": Palette.INK,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "xtick.major.width": 1.3,
        "ytick.major.width": 1.3,
        "legend.frameon": True,
        "legend.edgecolor": Palette.INK,
        "legend.framealpha": 0.96,
        "legend.fontsize": 9.5,
        "legend.title_fontsize": 10.5,
        "patch.edgecolor": Palette.INK,
        "patch.linewidth": 1.3,
        "patch.force_edgecolor": True,
        "lines.linewidth": 2.4,
        "lines.markersize": 7,
        "lines.markeredgecolor": Palette.INK,
        "lines.markeredgewidth": 0.9,
        "font.family": "DejaVu Sans",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.max_open_warning": 200,
    })
    sns.set_theme(style="whitegrid", rc={k: plt.rcParams[k] for k in [
        "axes.edgecolor", "axes.linewidth", "grid.color", "grid.linestyle", "axes.facecolor"]})
    plt.rcParams["axes.prop_cycle"] = cycler(color=Palette.CATEGORICAL)
    plt.rcParams["figure.dpi"] = display_dpi
    plt.rcParams["savefig.dpi"] = export_dpi


print("palette ready:", Palette.CATEGORICAL)

## 1.3 Figure vault and table vault

Every figure goes through `FigureVault.finish(...)`. It draws a black border around the figure, saves a **900 DPI PNG** and a **vector PDF**, keeps an index of all figures, and shows the picture in the notebook at a large screen size.

One safety rule: a single PNG is never allowed to exceed 150 million pixels, because bigger images can crash the notebook memory. Normal figures are saved at the full 900 DPI. Only a few very wide figures (the forest plot and the dashboard) are saved at a slightly lower DPI. Their PDF files are vector graphics and have no resolution limit. The real DPI of every PNG is written to the figure index.

Every table goes through `TableVault.show(...)`. It shows an orange and blue styled table, and saves a CSV, a LaTeX file and a table picture for the paper.

In [ ]:
class FigureVault:
    def __init__(self, out_dir, export_dpi=900, display_dpi=150, formats=("png", "pdf")):
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.export_dpi = export_dpi
        self.display_dpi = display_dpi
        self.formats = tuple(formats)
        self.max_pixels = 150_000_000
        self.records = []

    def canvas(self, rows=1, cols=1, panel=(6.2, 4.4), figsize=None, **kwargs):
        if figsize is None:
            figsize = (10.0, 5.625) if rows == 1 and cols == 1 else (panel[0] * cols, panel[1] * rows)
        fig, axes = plt.subplots(rows, cols, figsize=figsize, **kwargs)
        return fig, axes

    def finish(self, fig, name, title=None, caption="", section="", show=True, bottom=0.012):
        if title:
            fig.suptitle(title, fontsize=16, fontweight="bold", color=Palette.ORANGE_DARK, y=0.982)
            fig.tight_layout(rect=(0.012, bottom, 0.988, 0.94))
        else:
            fig.tight_layout(rect=(0.012, bottom, 0.988, 0.988))
        border = mpatches.Rectangle((0.002, 0.002), 0.996, 0.996, transform=fig.transFigure, fill=False,
                                    edgecolor=Palette.INK, linewidth=3.0, zorder=1000)
        fig.add_artist(border)
        index = len(self.records) + 1
        stem = f"fig{index:02d}_{name}"
        width, height = fig.get_size_inches()
        dpi_used = int(min(self.export_dpi, math.floor(math.sqrt(self.max_pixels / max(width * height, 1e-6)))))
        saved = []
        for fmt in self.formats:
            path = self.out_dir / f"{stem}.{fmt}"
            if fmt == "png":
                fig.savefig(path, dpi=dpi_used, facecolor="white")
            else:
                fig.savefig(path, facecolor="white")
            saved.append(str(path))
        self.records.append({"figure": index, "name": name, "title": title or name, "caption": caption,
                             "section": section, "files": saved, "png_dpi": dpi_used})
        if show:
            fig.set_dpi(min(self.display_dpi, 2600.0 / max(width, 1.0)))
            plt.show()
        plt.close(fig)
        return saved[0]

    def index_frame(self):
        rows = [{"figure": r["figure"], "name": r["name"], "title": r["title"], "section": r["section"],
                 "png_dpi": r.get("png_dpi", np.nan), "png": r["files"][0]} for r in self.records]
        return pd.DataFrame(rows)


def panel_label(ax, text, dx=-0.08, dy=1.06):
    ax.text(dx, dy, text, transform=ax.transAxes, fontsize=15, fontweight="bold", color="white",
            ha="center", va="center", bbox=dict(boxstyle="round,pad=0.25", facecolor=Palette.ORANGE, edgecolor=Palette.INK, linewidth=1.4))


def outline_bars(container, color=None, lw=1.4):
    for patch in container:
        patch.set_edgecolor(Palette.INK)
        patch.set_linewidth(lw)
        if color is not None:
            patch.set_facecolor(color)


def annotate_bars(ax, bars, fmt="{:.3f}", fontsize=9, offset=3, horizontal=False):
    for bar in bars:
        if horizontal:
            width = bar.get_width()
            ax.annotate(fmt.format(width), (width, bar.get_y() + bar.get_height() / 2), xytext=(offset, 0),
                        textcoords="offset points", va="center", ha="left", fontsize=fontsize, fontweight="bold")
        else:
            height = bar.get_height()
            ax.annotate(fmt.format(height), (bar.get_x() + bar.get_width() / 2, height), xytext=(0, offset),
                        textcoords="offset points", ha="center", va="bottom", fontsize=fontsize, fontweight="bold")


class TableVault:
    def __init__(self, out_dir, export_dpi=900):
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.export_dpi = export_dpi
        self.records = []

    def _format_frame(self, df, precision):
        formats = {}
        for col in df.columns:
            if pd.api.types.is_float_dtype(df[col]):
                formats[col] = "{:." + str(precision) + "f}"
        return formats

    def styled(self, df, caption="", precision=4):
        styler = df.style.format(self._format_frame(df, precision), na_rep="-").hide(axis="index")
        styler = styler.set_caption(caption)
        styler = styler.set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "15px"), ("font-weight", "bold"),
                                                ("color", Palette.ORANGE_DARK), ("text-align", "left"), ("padding", "6px 2px")]},
            {"selector": "th", "props": [("background-color", Palette.ORANGE), ("color", "white"), ("border", "1.6px solid #111111"),
                                          ("font-weight", "bold"), ("text-align", "center"), ("padding", "6px 10px")]},
            {"selector": "td", "props": [("border", "1.2px solid #111111"), ("padding", "5px 10px"), ("text-align", "center"), ("color", "#111111")]},
            {"selector": "tbody tr:nth-child(even)", "props": [("background-color", Palette.BLUE_LIGHT)]},
            {"selector": "tbody tr:nth-child(odd)", "props": [("background-color", "#FFFFFF")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("border", "2.2px solid #111111"), ("font-size", "12.5px")]},
        ])
        return styler

    def _render_png(self, df, path, caption, precision):
        shown = df.iloc[:28, :12]
        cells = []
        for row in shown.itertuples(index=False):
            cells.append([f"{v:.{precision}f}" if isinstance(v, (float, np.floating)) and np.isfinite(v) else ("-" if isinstance(v, (float, np.floating)) else str(v)) for v in row])
        n_rows, n_cols = shown.shape
        widths = [max(len(str(c)), max((len(r[j]) for r in cells), default=1)) for j, c in enumerate(shown.columns)]
        fig_w = min(max(6.0, 0.115 * sum(widths) + 0.5 * n_cols), 18.0)
        fig_h = 0.46 * (n_rows + 1) + 1.1
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis("off")
        table = ax.table(cellText=cells, colLabels=[str(c) for c in shown.columns], loc="center", cellLoc="center")
        table.auto_set_font_size(False)
        table.set_fontsize(9.5)
        table.auto_set_column_width(list(range(n_cols)))
        for (r, c), cell in table.get_celld().items():
            cell.set_edgecolor(Palette.INK)
            cell.set_linewidth(1.2)
            if r == 0:
                cell.set_facecolor(Palette.ORANGE)
                cell.set_text_props(color="white", fontweight="bold")
            else:
                cell.set_facecolor(Palette.BLUE_LIGHT if r % 2 == 0 else "white")
            cell.set_height(0.46 / fig_h * 1.6)
        ax.set_title(caption, fontsize=13, fontweight="bold", color=Palette.ORANGE_DARK, loc="left", pad=8)
        border = mpatches.Rectangle((0.002, 0.002), 0.996, 0.996, transform=fig.transFigure, fill=False, edgecolor=Palette.INK, linewidth=2.5)
        fig.add_artist(border)
        fig.savefig(path, dpi=self.export_dpi, bbox_inches="tight", facecolor="white")
        plt.close(fig)

    def show(self, df, name, caption="", precision=4, render_png=True, display_table=True):
        index = len(self.records) + 1
        stem = f"table{index:02d}_{name}"
        csv_path = self.out_dir / f"{stem}.csv"
        tex_path = self.out_dir / f"{stem}.tex"
        png_path = self.out_dir / f"{stem}.png"
        df.to_csv(csv_path, index=False)
        try:
            df.to_latex(tex_path, index=False, float_format=lambda v: f"{v:.{precision}f}", caption=caption, label=f"tab:{name}", escape=True)
        except Exception:
            tex_path.write_text(df.to_string(index=False))
        if render_png:
            try:
                self._render_png(df, png_path, caption, precision)
            except Exception as err:
                print("table image skipped:", err)
        if display_table:
            try:
                display(self.styled(df, caption, precision))
            except Exception:
                display(df)
        self.records.append({"table": index, "name": name, "caption": caption, "rows": len(df), "csv": str(csv_path)})
        return csv_path

    def index_frame(self):
        return pd.DataFrame(self.records)


def banner(title, subtitle="", color_a=Palette.ORANGE, color_b=Palette.BLUE):
    html = (
        f'<div style="background: linear-gradient(90deg,{color_a},{color_b}); padding:16px 22px; border-radius:12px; '
        f'border:3px solid #111; margin:10px 0; font-family:Arial,sans-serif;">'
        f'<div style="font-size:21px; font-weight:800; color:white; text-shadow:1px 1px 3px rgba(0,0,0,.6);">{title}</div>'
        f'<div style="font-size:13.5px; color:#FFF8EA; margin-top:4px;">{subtitle}</div></div>'
    )
    display(HTML(html))


def note_box(text, kind="info"):
    colors = {"info": (Palette.BLUE_LIGHT, Palette.BLUE_DARK), "warn": (Palette.ORANGE_LIGHT, Palette.ORANGE_DARK),
              "good": ("#DFF5DC", "#1E7B1A"), "bad": ("#FBD5DB", "#9B0E27")}
    bg, fg = colors.get(kind, colors["info"])
    display(HTML(f'<div style="background:{bg}; border:2px solid #111; border-left:9px solid {fg}; border-radius:8px; '
                 f'padding:10px 14px; margin:8px 0; font-family:Arial,sans-serif; font-size:13.5px; color:#111;">{text}</div>'))


class StageTimer:
    def __init__(self):
        self.rows = []

    def track(self, name):
        return _TimerContext(self, name)

    def frame(self):
        return pd.DataFrame(self.rows)


class _TimerContext:
    def __init__(self, owner, name):
        self.owner = owner
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.owner.rows.append({"stage": self.name, "seconds": round(time.time() - self.t0, 1)})
        return False


TIMER = StageTimer()
print("vaults ready")

# Part 2: Configuration

## 2.1 One config object for the whole notebook

All settings are in `PipelineConfig`. You only need to change **one line**: `PRESET`.
The presets change how much data is used and how many epochs are trained. Everything else is derived from the config, so nothing is hidden inside later cells.

Heavy results are cached on disk. If you change a setting (for example the species list) and want a completely fresh run, run `CACHE.clear()` once or delete the output folder.

In [ ]:
PRESET = os.environ.get("BIOLINGUA_PRESET", "standard")

SPECIES_DEFAULT = ("Turdus merula", "Parus major", "Corvus corone", "Erithacus rubecula", "Fringilla coelebs")


@dataclass
class PipelineConfig:
    preset: str = "standard"
    seed: int = 42
    data_source: str = "auto"
    species: tuple = SPECIES_DEFAULT
    recordings_per_species: int = 200
    quality_grades: tuple = ("A", "B")
    targeted_label_queries: bool = True
    targeted_recordings_per_label: int = 20
    api_url: str = "https://xeno-canto.org/api/3/recordings"
    api_timeout: int = 90
    api_retries: int = 3
    api_pause: float = 0.4
    download_workers: int = 4
    sample_rate: int = 16000
    segment_seconds: float = 3.0
    min_segment_seconds: float = 1.0
    trim_top_db: float = 30.0
    normalize: bool = True
    denoise: bool = True
    max_recording_seconds: float = 120.0
    max_segments_per_recording: int = 40
    preprocess_workers: int = 4
    split_fractions: tuple = (0.70, 0.15, 0.15)
    encoder_candidates: tuple = ("hubert-base", "wav2vec2-base", "wav2vec2-large-lv60", "birdaves-base", "birdaves-large")
    primary_encoder: str = "auto"
    encoder_batch_size: int = 16
    use_fp16: bool = True
    layer_fractions: tuple = (0.25, 0.5, 0.75, 1.0)
    eval_segments_per_species: int = 500
    eval_labelled_per_class: int = 300
    adapter_train_segments: int = 6000
    adapter_val_segments: int = 1200
    adapter_views: int = 3
    adapter_dim: int = 128
    adapter_hidden: int = 256
    adapter_epochs: int = 40
    adapter_patience: int = 6
    adapter_lr: float = 2e-3
    adapter_temperature: float = 0.1
    pca_variance: float = 0.95
    pca_min_components: int = 16
    pca_max_components: int = 128
    k_grid: tuple = (10, 20, 30, 50, 75)
    cluster_seeds: tuple = (42, 43, 44, 45, 46)
    min_clusters: int = 8
    cluster_eval_points: int = 6000
    silhouette_points: int = 3000
    vq_epochs: int = 25
    lm_d_model: int = 192
    lm_heads: int = 6
    lm_layers: int = 3
    lm_ff: int = 768
    lm_dropout: float = 0.15
    lm_epochs: int = 40
    lm_batch: int = 64
    lm_lr: float = 5e-4
    lm_patience: int = 6
    lm_seeds: tuple = (42, 43, 44)
    behaviour_classes: tuple = ("alarm", "song", "contact")
    labelled_segments_per_class: int = 400
    max_segments_per_labelled_recording: int = 6
    cv_folds: int = 5
    cv_repeats: int = 2
    bootstrap_iterations: int = 500
    abx_triplets: int = 5000
    scaling_fractions: tuple = (0.1, 0.2, 0.4, 0.7, 1.0)
    frame_study_segments: int = 600
    frame_steps: int = 25
    pseudo_thresholds: tuple = (0.9, 0.8, 0.7, 0.6)
    ablation_recordings: int = 120
    export_dpi: int = 900
    display_dpi: int = 150
    figure_formats: tuple = ("png", "pdf")
    keep_raw_audio: bool = True

    def summary_frame(self):
        rows = []
        for key, value in asdict(self).items():
            rows.append({"setting": key, "value": str(value)})
        return pd.DataFrame(rows)


PRESET_OVERRIDES = {
    "smoke": dict(
        data_source="synthetic", recordings_per_species=10, targeted_recordings_per_label=0,
        max_recording_seconds=14.0, max_segments_per_recording=6, min_segment_seconds=1.0,
        preprocess_workers=1, denoise=False, split_fractions=(0.6, 0.2, 0.2),
        encoder_candidates=("tiny-a", "tiny-b", "tiny-c"), encoder_batch_size=16, use_fp16=False,
        eval_segments_per_species=40, eval_labelled_per_class=30,
        adapter_train_segments=120, adapter_val_segments=40, adapter_views=2, adapter_epochs=4, adapter_patience=2,
        pca_max_components=24, pca_min_components=8,
        k_grid=(6, 10, 14), cluster_seeds=(42, 43), min_clusters=4, cluster_eval_points=400, silhouette_points=300,
        vq_epochs=3,
        lm_d_model=32, lm_heads=2, lm_layers=1, lm_ff=64, lm_epochs=3, lm_batch=16, lm_patience=2, lm_seeds=(42, 43),
        labelled_segments_per_class=40, max_segments_per_labelled_recording=4, cv_folds=3, cv_repeats=1,
        bootstrap_iterations=60, abx_triplets=400, scaling_fractions=(0.4, 0.7, 1.0),
        frame_study_segments=60, frame_steps=8, pseudo_thresholds=(0.8, 0.6), ablation_recordings=12,
        export_dpi=110, display_dpi=90, figure_formats=("png", "pdf"),
    ),
    "pilot": dict(
        recordings_per_species=60, targeted_recordings_per_label=10, preprocess_workers=min(4, CPU_COUNT),
        encoder_candidates=("hubert-base", "wav2vec2-base", "birdaves-base"),
        eval_segments_per_species=250, eval_labelled_per_class=150,
        adapter_train_segments=2500, adapter_val_segments=500, adapter_epochs=25,
        k_grid=(10, 20, 30, 50), cluster_seeds=(42, 43, 44), cluster_eval_points=4000,
        lm_epochs=25, lm_seeds=(42, 43), labelled_segments_per_class=150,
        bootstrap_iterations=300, abx_triplets=3000, scaling_fractions=(0.25, 0.5, 1.0),
        frame_study_segments=300, ablation_recordings=60,
    ),
    "standard": dict(preprocess_workers=min(4, CPU_COUNT)),
    "full": dict(
        recordings_per_species=300, targeted_recordings_per_label=40, preprocess_workers=min(4, CPU_COUNT),
        eval_segments_per_species=700, adapter_train_segments=9000, adapter_val_segments=1800,
        labelled_segments_per_class=600, bootstrap_iterations=1000, cv_repeats=3, abx_triplets=8000,
        frame_study_segments=900, ablation_recordings=160,
    ),
}

if PRESET not in PRESET_OVERRIDES:
    raise ValueError(f"PRESET must be one of {list(PRESET_OVERRIDES)}; got {PRESET!r}")

CFG = PipelineConfig(preset=PRESET, **PRESET_OVERRIDES[PRESET])
SEED = CFG.seed
set_seed(SEED)
apply_theme(CFG.display_dpi, CFG.export_dpi)


class RunPaths:
    def __init__(self, preset):
        base = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("BIOLINGUA_ROOT", "./biolingua_outputs"))
        self.root = base / f"biolingua_{preset}"
        self.raw = self.root / "data" / "raw"
        self.segments = self.root / "data" / "segments"
        self.embeddings = self.root / "data" / "embeddings"
        self.cache = self.root / "cache"
        self.models = self.root / "models"
        self.results = self.root / "results"
        self.figures = self.root / "figures"
        self.tables = self.root / "tables"
        for path in [self.raw, self.segments, self.embeddings, self.cache, self.models, self.results, self.figures, self.tables]:
            path.mkdir(parents=True, exist_ok=True)


PATHS = RunPaths(CFG.preset)
VAULT = FigureVault(PATHS.figures, CFG.export_dpi, CFG.display_dpi, CFG.figure_formats)
TABLES = TableVault(PATHS.tables, CFG.export_dpi)


class ResultCache:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)

    def path(self, name):
        return self.folder / f"{name}.pkl"

    def has(self, name):
        return self.path(name).exists()

    def save(self, name, value):
        with open(self.path(name), "wb") as handle:
            pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        return value

    def load(self, name):
        with open(self.path(name), "rb") as handle:
            return pickle.load(handle)

    def clear(self):
        for path in self.folder.glob("*.pkl"):
            path.unlink()
        print("cache cleared")

    def get_or_compute(self, name, builder, force=False):
        if self.has(name) and not force:
            try:
                value = self.load(name)
                print(f"[cache] loaded {name}")
                return value
            except Exception as err:
                print(f"[cache] could not read {name}: {err}")
        value = builder()
        self.save(name, value)
        return value


CACHE = ResultCache(PATHS.cache)
RESULTS = {}

banner("Configuration ready", f"preset = {CFG.preset} | recordings per species = {CFG.recordings_per_species} | export DPI = {CFG.export_dpi} | outputs in {PATHS.root}", color_a="brown", color_b="brown")
TABLES.show(CFG.summary_frame(), "configuration", "Table 1. All settings used in this run", render_png=False, display_table=False)
print("config table saved. species:", ", ".join(CFG.species))

# Part 3: Data collection

## 3.1 Where the recordings come from

Three data sources sit behind one class called `CorpusBuilder`:

1. **Xeno-Canto API** (main source). The key is read from Kaggle Secrets, then from an environment variable, and only then from a hidden input box. It is never written into the notebook.
2. **Kaggle dataset fallback** (`rohanrao/xeno-canto-bird-recordings-extended-*`), used only if the API fails.
3. **Synthetic corpus**, used by the `smoke` preset. It creates fake bird sounds so you can test every cell in minutes.

Besides the normal top-quality query, the collector also runs small **label-aware queries** (`type:alarm`, `type:song`, `type:call`). Alarm calls are rare in Xeno-Canto, so without these queries the behaviour classes stay very unbalanced.

In [ ]:
class SecretsResolver:
    def __init__(self, names=("XENO_CANTO_API_KEY", "XC_API_KEY")):
        self.names = names

    def find(self):
        for name in self.names:
            value = os.environ.get(name, "")
            if value:
                return value.strip()
        try:
            from kaggle_secrets import UserSecretsClient
            client = UserSecretsClient()
            for name in self.names:
                try:
                    value = client.get_secret(name)
                    if value:
                        return value.strip()
                except Exception:
                    continue
        except Exception:
            pass
        try:
            from google.colab import userdata
            for name in self.names:
                try:
                    value = userdata.get(name)
                    if value:
                        return value.strip()
                except Exception:
                    continue
        except Exception:
            pass
        return ""

    def require(self):
        key = self.find()
        if not key:
            key = getpass.getpass("Enter your Xeno-Canto API key (input is hidden): ").strip()
        if not key:
            raise RuntimeError("A Xeno-Canto API key is required. Add it as the Kaggle secret XENO_CANTO_API_KEY.")
        return key


def parse_length_seconds(text):
    try:
        parts = [float(p) for p in str(text).split(":")]
    except Exception:
        return np.nan
    seconds = 0.0
    for p in parts:
        seconds = seconds * 60.0 + p
    return seconds


CATALOG_COLUMNS = ["file", "species", "recording_id", "english_name", "call_type", "remarks", "quality",
                   "length_seconds", "date", "country", "latitude", "longitude", "source"]


class XenoCantoCollector:
    def __init__(self, cfg, paths, api_key):
        self.cfg = cfg
        self.paths = paths
        self.api_key = api_key
        self.session = None

    def _session(self):
        if self.session is None:
            import requests
            self.session = requests.Session()
            self.session.headers.update({"User-Agent": "BioLingua-Research/3.0"})
        return self.session

    def _clean(self, text):
        return str(text).replace(self.api_key, "[HIDDEN]") if self.api_key else str(text)

    def _get_json(self, params, label):
        params = dict(params)
        params["key"] = self.api_key
        last = None
        for attempt in range(1, self.cfg.api_retries + 1):
            try:
                response = self._session().get(self.cfg.api_url, params=params, timeout=self.cfg.api_timeout)
                response.raise_for_status()
                return response.json()
            except Exception as err:
                last = err
                if attempt < self.cfg.api_retries:
                    time.sleep(3.0 * attempt)
        print(f"  api listing failed ({label}): {self._clean(last)}")
        return {"recordings": [], "numPages": 0}

    @staticmethod
    def species_query(name):
        parts = name.strip().split(" ", 1)
        return f"gen:{parts[0]} sp:{parts[1]}" if len(parts) > 1 else f"gen:{parts[0]}"

    def list_recordings(self, species, grade, extra="", limit=10 ** 9):
        query = f"{self.species_query(species)} q:{grade}" + (f" {extra}" if extra else "")
        found, page, pages = [], 1, 1
        while page <= pages and len(found) < limit:
            payload = self._get_json({"query": query, "page": page}, f"{species} {grade} {extra} p{page}")
            pages = int(payload.get("numPages", 1) or 1)
            batch = payload.get("recordings", [])
            if not batch:
                break
            found.extend(batch)
            page += 1
            time.sleep(self.cfg.api_pause)
        return found[:limit]

    def _download(self, record, species):
        rec_id = str(record.get("id", ""))
        url = record.get("file") or ""
        if not rec_id or not url:
            return None
        if url.startswith("//"):
            url = "https:" + url
        elif url.startswith("/"):
            url = "https://xeno-canto.org" + url
        target = self.paths.raw / f"{species.replace(' ', '_')}_{rec_id}.mp3"
        if target.exists() and target.stat().st_size > 2048:
            return target
        for attempt in range(1, self.cfg.api_retries + 1):
            try:
                response = self._session().get(url, timeout=max(90, self.cfg.api_timeout))
                response.raise_for_status()
                if len(response.content) < 2048:
                    raise ValueError("file too small")
                tmp = target.with_suffix(".part")
                tmp.write_bytes(response.content)
                tmp.replace(target)
                return target
            except Exception:
                time.sleep(2.0 * attempt)
        return None

    def _row(self, record, species, path):
        return {
            "file": str(path), "species": species, "recording_id": str(record.get("id")),
            "english_name": record.get("en", ""), "call_type": record.get("type") or "",
            "remarks": record.get("rmk", "") or "", "quality": record.get("q", ""),
            "length_seconds": parse_length_seconds(record.get("length", "")), "date": record.get("date", ""),
            "country": record.get("cnt", ""), "latitude": record.get("lat", ""), "longitude": record.get("lon", ""),
            "source": "xeno_canto_api",
        }

    def collect(self):
        rows, seen = [], set()
        for species in self.cfg.species:
            print(f"\ncollecting {species}")
            picked = []
            for grade in self.cfg.quality_grades:
                if len(picked) >= self.cfg.recordings_per_species:
                    break
                need = self.cfg.recordings_per_species - len(picked)
                for record in self.list_recordings(species, grade, limit=need * 2):
                    rid = str(record.get("id"))
                    if rid in seen:
                        continue
                    picked.append(record)
                    seen.add(rid)
                    if len(picked) >= self.cfg.recordings_per_species:
                        break
            if self.cfg.targeted_label_queries and self.cfg.targeted_recordings_per_label > 0:
                for tag in ("alarm", "song", "call"):
                    added = 0
                    for grade in self.cfg.quality_grades:
                        if added >= self.cfg.targeted_recordings_per_label:
                            break
                        for record in self.list_recordings(species, grade, extra=f"type:{tag}", limit=self.cfg.targeted_recordings_per_label * 2):
                            rid = str(record.get("id"))
                            if rid in seen:
                                continue
                            picked.append(record)
                            seen.add(rid)
                            added += 1
                            if added >= self.cfg.targeted_recordings_per_label:
                                break
            print(f"  {len(picked)} recordings listed, downloading ...")
            with ThreadPoolExecutor(max_workers=self.cfg.download_workers) as pool:
                futures = {pool.submit(self._download, rec, species): rec for rec in picked}
                for future in tqdm(as_completed(futures), total=len(futures), desc=species[:18]):
                    path = future.result()
                    if path is not None:
                        rows.append(self._row(futures[future], species, path))
        frame = pd.DataFrame(rows, columns=CATALOG_COLUMNS).drop_duplicates("recording_id").reset_index(drop=True)
        if frame.empty:
            raise RuntimeError("No recordings were downloaded. Check the API key and internet access.")
        return frame


class KaggleDatasetCollector:
    handles = ("rohanrao/xeno-canto-bird-recordings-extended-a-m", "rohanrao/xeno-canto-bird-recordings-extended-n-z")

    def __init__(self, cfg, paths):
        self.cfg = cfg
        self.paths = paths

    def _roots(self):
        roots = []
        for handle in self.handles:
            local = Path("/kaggle/input") / handle.split("/")[1]
            if local.exists():
                roots.append(local)
                continue
            import kagglehub
            roots.append(Path(kagglehub.dataset_download(handle)))
        return roots

    def collect(self):
        roots = self._roots()
        tables, audio = [], {}
        for root in roots:
            for csv_path in root.rglob("train_extended.csv"):
                tables.append(pd.read_csv(csv_path))
            for path in root.rglob("*.mp3"):
                audio[path.name] = path
        master = pd.concat(tables, ignore_index=True)
        master["call_type"] = master["type"].fillna("")
        wanted = list(self.cfg.species)
        available = set(master["sci_name"].unique())
        if not any(s in available for s in wanted):
            text = master["call_type"].str.lower()
            coverage = pd.DataFrame({"alarm": text.str.contains("alarm"), "song": text.str.contains("song"),
                                     "contact": text.str.contains("call|contact")}).groupby(master["sci_name"]).sum()
            wanted = list(coverage.assign(score=coverage.min(axis=1)).sort_values("score", ascending=False).index[:5])
            print("Configured species not in this dataset; using the best label-covered species:", wanted)
        rows = []
        for species in wanted:
            subset = master[master["sci_name"] == species]
            subset = subset[subset["filename"].isin(audio)].head(self.cfg.recordings_per_species)
            for record in subset.itertuples():
                rows.append({"file": str(audio[record.filename]), "species": species, "recording_id": str(record.xc_id),
                             "english_name": getattr(record, "species", ""), "call_type": record.call_type, "remarks": "",
                             "quality": "", "length_seconds": float(getattr(record, "duration", np.nan)),
                             "date": getattr(record, "date", ""), "country": getattr(record, "country", ""),
                             "latitude": getattr(record, "latitude", ""), "longitude": getattr(record, "longitude", ""),
                             "source": "kaggle_dataset"})
        frame = pd.DataFrame(rows, columns=CATALOG_COLUMNS)
        if frame.empty:
            raise RuntimeError("The Kaggle dataset fallback found no usable recordings.")
        return frame


class SyntheticCorpus:
    def __init__(self, cfg, paths):
        self.cfg = cfg
        self.paths = paths

    def _voice(self, rng, species_index, behaviour, seconds, sr):
        t = np.arange(int(seconds * sr)) / sr
        y = np.zeros_like(t)
        base = 1800.0 + 650.0 * species_index
        h2 = 0.15 + 0.12 * species_index
        def tone(start, length, f0, f1):
            i0, i1 = int(start * sr), min(int((start + length) * sr), len(t))
            if i1 <= i0:
                return
            local = np.linspace(0, 1, i1 - i0)
            freq = f0 + (f1 - f0) * local + 60 * np.sin(2 * np.pi * 6 * local)
            phase = 2 * np.pi * np.cumsum(freq) / sr
            env = np.sin(np.pi * local) ** 1.5
            y[i0:i1] += env * (np.sin(phase) + h2 * np.sin(2 * phase))
        cursor = rng.uniform(0.1, 0.5)
        while cursor < seconds - 0.3:
            if behaviour == "song":
                length = rng.uniform(0.35, 0.8)
                tone(cursor, length, base, base * rng.uniform(1.1, 1.5))
                cursor += length + rng.uniform(0.05, 0.3)
            elif behaviour == "alarm":
                for _ in range(rng.integers(5, 9)):
                    tone(cursor, 0.06, base * 1.6, base * 1.9)
                    cursor += 0.11
                cursor += rng.uniform(0.3, 0.7)
            else:
                length = rng.uniform(0.12, 0.25)
                tone(cursor, length, base * 1.3, base * 0.8)
                cursor += length + rng.uniform(0.7, 1.4)
        y += rng.normal(0, 0.03 + 0.03 * rng.random(), size=y.shape)
        return (y / max(np.max(np.abs(y)), 1e-6) * 0.8).astype(np.float32)

    def collect(self):
        rng = np.random.default_rng(self.cfg.seed)
        rows = []
        vocabulary = {"alarm": ["alarm call", "alarm call, call", "scolding"], "song": ["song", "song, territorial song", "subsong"],
                      "contact": ["call", "contact call", "flight call"]}
        for si, species in enumerate(self.cfg.species):
            for r in range(self.cfg.recordings_per_species):
                behaviour = rng.choice(["alarm", "song", "contact"], p=[0.25, 0.4, 0.35])
                seconds = rng.uniform(7.0, self.cfg.max_recording_seconds)
                audio = self._voice(rng, si, behaviour, seconds, self.cfg.sample_rate)
                rec_id = f"{9000000 + si * 1000 + r}"
                path = self.paths.raw / f"{species.replace(' ', '_')}_{rec_id}.wav"
                sf.write(path, audio, self.cfg.sample_rate)
                call_type = rng.choice(vocabulary[behaviour]) if rng.random() > 0.15 else ""
                rows.append({"file": str(path), "species": species, "recording_id": rec_id, "english_name": species,
                             "call_type": call_type, "remarks": "", "quality": rng.choice(["A", "B"]),
                             "length_seconds": seconds, "date": "2026-01-01", "country": "Synthetic",
                             "latitude": "", "longitude": "", "source": "synthetic"})
        return pd.DataFrame(rows, columns=CATALOG_COLUMNS)


class CorpusBuilder:
    def __init__(self, cfg, paths):
        self.cfg = cfg
        self.paths = paths
        self.catalog_path = paths.root / "data" / "catalog.csv"

    def build(self):
        if self.catalog_path.exists():
            frame = pd.read_csv(self.catalog_path, dtype={"recording_id": str})
            frame = frame[frame["file"].map(lambda p: Path(p).exists())].reset_index(drop=True)
            target = self.cfg.recordings_per_species * len(self.cfg.species)
            if len(frame) >= 0.9 * target:
                print(f"[reuse] catalog with {len(frame)} recordings")
                return frame
        source = self.cfg.data_source
        if source == "synthetic":
            frame = SyntheticCorpus(self.cfg, self.paths).collect()
        else:
            frame = None
            if source in ("auto", "xeno_canto_api"):
                try:
                    frame = XenoCantoCollector(self.cfg, self.paths, SecretsResolver().require()).collect()
                except Exception as err:
                    print("Xeno-Canto API path failed:", err)
                    if source == "xeno_canto_api":
                        raise
            if frame is None:
                frame = KaggleDatasetCollector(self.cfg, self.paths).collect()
        frame.to_csv(self.catalog_path, index=False)
        return frame


with TIMER.track("data collection"):
    catalog_raw = CorpusBuilder(CFG, PATHS).build()
print(f"\ncatalog: {len(catalog_raw)} recordings from {catalog_raw['source'].iloc[0]}")
display(catalog_raw.head(5))

# Part 4: Behaviour labels

## 4.1 Weak labels from call type text

Xeno-Canto uploaders describe each recording with free text such as `song`, `alarm call` or `call, flight call`. `BehaviourLabeler` reads this text phrase by phrase and votes for **alarm**, **song** or **contact**.

- If one class gets more votes than the others, the recording gets that label.
- If the text is mixed (for example `song, call`), the recording is marked **ambiguous** and is left out of supervised tests. This is stricter than the old notebooks, which always took the first matching word.
- `strict` mode reads only the call type field. `expanded` mode also reads the uploader remarks.

These labels are **weak** because they come from human text, not from expert annotation. Every behaviour number in this notebook should be described as "weak-label behaviour probing".

In [ ]:
class BehaviourLabeler:
    ALARM = ("alarm", "scold", "distress", "mobbing", "warning", "aggress", "threat")
    SONG = ("song", "subsong", "singing", "sings")
    CONTACT = ("contact", "call", "begging", "chip", "chatter", "chirp", "twitter", "flight")

    @staticmethod
    def _phrases(text):
        return [p.strip() for p in re.split(r"[,;/()]", str(text).lower()) if p.strip()]

    def vote_phrase(self, phrase):
        if any(k in phrase for k in self.ALARM):
            return "alarm"
        if any(k in phrase for k in self.SONG):
            return "song"
        if any(k in phrase for k in self.CONTACT):
            return "contact"
        return None

    def decide(self, text):
        votes = Counter(v for v in (self.vote_phrase(p) for p in self._phrases(text)) if v)
        if not votes:
            return None, 0.0, "none"
        ranked = votes.most_common()
        if len(ranked) > 1 and ranked[0][1] == ranked[1][1]:
            return None, 0.5, "ambiguous"
        return ranked[0][0], ranked[0][1] / sum(votes.values()), "clear"

    def label_catalog(self, catalog):
        out = catalog.copy()
        strict = [self.decide(t) for t in out["call_type"]]
        expanded = [self.decide(f"{t}, {r}") for t, r in zip(out["call_type"], out["remarks"])]
        out["behaviour_strict"] = [s[0] for s in strict]
        out["behaviour_confidence"] = [s[1] for s in strict]
        out["label_status"] = [s[2] for s in strict]
        out["behaviour_expanded"] = [e[0] for e in expanded]
        return out


catalog = BehaviourLabeler().label_catalog(catalog_raw)
label_summary = (catalog.assign(behaviour_strict=catalog["behaviour_strict"].fillna("unlabelled"))
                 .pivot_table(index="species", columns="behaviour_strict", values="recording_id", aggfunc="count", fill_value=0).reset_index())
TABLES.show(label_summary, "label_coverage", "Table 2. Recordings per species and weak behaviour label (strict mode)", precision=0)
print(catalog["label_status"].value_counts().to_dict())

# Part 5: Audio cleaning and the segment store

## 5.1 Cleaning, cutting and storing audio

`AudioPreprocessor` follows the same order as the earlier notebooks: resample, peak-normalise, reduce noise, trim silence at both ends, then cut into fixed segments (3 seconds by default, at least 1 second, padded with zeros). Only the first `max_segments_per_recording` segments are kept, so very long recordings cannot dominate the corpus.

`SegmentStore` saves all segments in **one memory-mapped int16 file**. This is much faster and lighter than writing tens of thousands of small WAV files. Every segment keeps its recording id and its position, so sequences of acoustic units can be rebuilt later.

The split into **train, validation and test** is done by **recording** and stratified by species. No recording is ever shared between splits.

In [ ]:
class AudioPreprocessor:
    def __init__(self, sample_rate=16000, segment_seconds=3.0, min_seconds=1.0, trim_db=30.0, denoise=True,
                 normalize=True, max_seconds=120.0, max_segments=40):
        self.sr = sample_rate
        self.segment_length = int(segment_seconds * sample_rate)
        self.min_length = int(min_seconds * sample_rate)
        self.trim_db = trim_db
        self.denoise = denoise
        self.normalize = normalize
        self.max_seconds = max_seconds
        self.max_segments = max_segments

    @classmethod
    def from_config(cls, cfg, **overrides):
        settings = dict(sample_rate=cfg.sample_rate, segment_seconds=cfg.segment_seconds, min_seconds=cfg.min_segment_seconds,
                        trim_db=cfg.trim_top_db, denoise=cfg.denoise, normalize=cfg.normalize,
                        max_seconds=cfg.max_recording_seconds, max_segments=cfg.max_segments_per_recording)
        settings.update(overrides)
        return cls(**settings)

    def clean(self, wave):
        wave = np.nan_to_num(wave.astype(np.float32))
        if self.normalize:
            wave = wave / (np.max(np.abs(wave)) + 1e-9)
        if self.denoise:
            try:
                import noisereduce as nr
                wave = nr.reduce_noise(y=wave, sr=self.sr).astype(np.float32)
            except Exception:
                pass
        if self.trim_db is not None and self.trim_db > 0:
            trimmed, _ = librosa.effects.trim(wave, top_db=self.trim_db)
            if len(trimmed) >= self.min_length:
                wave = trimmed
        return wave

    def cut(self, wave):
        segments, valid = [], []
        for start in range(0, len(wave), self.segment_length):
            chunk = wave[start:start + self.segment_length]
            if len(chunk) < self.min_length:
                continue
            valid.append(len(chunk))
            if len(chunk) < self.segment_length:
                chunk = np.pad(chunk, (0, self.segment_length - len(chunk)))
            segments.append(chunk)
            if len(segments) >= self.max_segments:
                break
        if not segments:
            return np.zeros((0, self.segment_length), dtype=np.int16), np.zeros(0, dtype=np.int32)
        stacked = np.clip(np.stack(segments), -1.0, 1.0)
        return (stacked * 32767.0).astype(np.int16), np.asarray(valid, dtype=np.int32)

    def process_file(self, path):
        try:
            wave, _ = librosa.load(path, sr=self.sr, mono=True, duration=self.max_seconds)
        except Exception:
            return np.zeros((0, self.segment_length), dtype=np.int16), np.zeros(0, dtype=np.int32)
        if len(wave) < self.min_length:
            return np.zeros((0, self.segment_length), dtype=np.int16), np.zeros(0, dtype=np.int32)
        return self.cut(self.clean(wave))


class RecordingSplitter:
    def __init__(self, fractions=(0.70, 0.15, 0.15), seed=42):
        self.fractions = fractions
        self.seed = seed

    def assign(self, catalog):
        rng = np.random.default_rng(self.seed)
        split = pd.Series("train", index=catalog.index)
        for species, group in catalog.groupby("species"):
            ids = group.index.to_numpy().copy()
            rng.shuffle(ids)
            n_train = int(round(len(ids) * self.fractions[0]))
            n_val = int(round(len(ids) * self.fractions[1]))
            split.loc[ids[n_train:n_train + n_val]] = "val"
            split.loc[ids[n_train + n_val:]] = "test"
        return split


class SegmentStore:
    def __init__(self, folder, segment_length):
        self.folder = Path(folder)
        self.segment_length = segment_length
        self.data_path = self.folder / "segments_int16.dat"
        self.meta_path = self.folder / "segments_meta.csv"
        self.memmap = None
        self.meta = None

    def exists(self):
        return self.data_path.exists() and self.meta_path.exists()

    def build(self, catalog, preprocessor, workers=1, chunk=32):
        from joblib import Parallel, delayed
        rows, cursor = [], 0
        with open(self.data_path, "wb") as handle:
            for start in tqdm(range(0, len(catalog), chunk), desc="segmenting"):
                block = catalog.iloc[start:start + chunk]
                if workers > 1:
                    outputs = Parallel(n_jobs=workers)(delayed(preprocessor.process_file)(f) for f in block["file"])
                else:
                    outputs = [preprocessor.process_file(f) for f in block["file"]]
                for record, (segments, valid) in zip(block.itertuples(), outputs):
                    for i in range(len(segments)):
                        rows.append({"segment_id": cursor, "recording_id": str(record.recording_id), "species": record.species,
                                     "segment_index": i, "valid_length": int(valid[i])})
                        cursor += 1
                    handle.write(segments.tobytes())
        meta = pd.DataFrame(rows)
        meta.to_csv(self.meta_path, index=False)
        return meta

    def open(self):
        self.meta = pd.read_csv(self.meta_path, dtype={"recording_id": str})
        self.memmap = np.memmap(self.data_path, dtype=np.int16, mode="r", shape=(len(self.meta), self.segment_length))
        return self

    def __len__(self):
        return 0 if self.meta is None else len(self.meta)

    def get(self, indices):
        indices = np.asarray(indices, dtype=np.int64)
        waves = self.memmap[indices].astype(np.float32) / 32768.0
        valid = self.meta["valid_length"].to_numpy()[indices]
        return waves, valid


class ArrayWaveSource:
    def __init__(self, waves, valid):
        self.waves = waves
        self.valid = valid

    def __len__(self):
        return len(self.waves)

    def get(self, indices):
        indices = np.asarray(indices, dtype=np.int64)
        return self.waves[indices].astype(np.float32), self.valid[indices]


catalog["split"] = RecordingSplitter(CFG.split_fractions, CFG.seed).assign(catalog)
preprocessor = AudioPreprocessor.from_config(CFG)
store = SegmentStore(PATHS.segments, preprocessor.segment_length)
with TIMER.track("segmentation"):
    if not store.exists():
        store.build(catalog, preprocessor, workers=CFG.preprocess_workers)
    store.open()

seg_meta = store.meta.merge(
    catalog[["recording_id", "call_type", "remarks", "quality", "behaviour_strict", "behaviour_expanded", "behaviour_confidence", "label_status", "split"]],
    on="recording_id", how="left")
seg_meta["seg_pos"] = np.arange(len(seg_meta))
assert (seg_meta["segment_id"].to_numpy() == np.arange(len(seg_meta))).all()
species_names = list(CFG.species)
species_to_index = {s: i for i, s in enumerate(species_names)}
seg_meta["species_index"] = seg_meta["species"].map(species_to_index).astype(int)
catalog.to_csv(PATHS.root / "data" / "catalog_labelled.csv", index=False)
print(f"segments: {len(seg_meta)} from {seg_meta['recording_id'].nunique()} recordings")
print(seg_meta["split"].value_counts().to_dict())

# Part 6: Exploratory data analysis

Before any model is trained, we look at the data. A reviewer will ask: how many recordings per species, how long are they, how many have behaviour labels, and do the sounds look different between classes? The figures below answer these questions.

## 6.1 Corpus overview

In [ ]:
SPECIES_COLORS = Palette.species_colors(species_names)
BEHAVIOURS = list(CFG.behaviour_classes)


def figure_corpus_overview():
    fig, axes = VAULT.canvas(2, 2, panel=(7.2, 4.6))
    ax = axes[0, 0]
    quality = catalog.assign(quality=catalog["quality"].replace("", "n/a")).pivot_table(index="species", columns="quality", values="recording_id", aggfunc="count", fill_value=0)
    bottoms = np.zeros(len(quality))
    for i, grade in enumerate(quality.columns):
        bars = ax.bar(range(len(quality)), quality[grade].to_numpy(), bottom=bottoms, label=f"grade {grade}", color=Palette.CATEGORICAL[i % 5], edgecolor=Palette.INK, linewidth=1.3)
        bottoms += quality[grade].to_numpy()
    ax.set_xticks(range(len(quality)))
    ax.set_xticklabels([s.replace(" ", "\n") for s in quality.index], fontsize=8.5)
    ax.set_ylabel("Recordings")
    ax.set_title("Recordings per species")
    ax.legend(title="Quality")
    panel_label(ax, "A")

    ax = axes[0, 1]
    for species in species_names:
        values = catalog.loc[catalog["species"] == species, "length_seconds"].dropna().clip(upper=300)
        if len(values) > 2:
            sns.kdeplot(values, ax=ax, color=SPECIES_COLORS[species], fill=True, alpha=0.30, linewidth=2.2, label=species, cut=0)
    ax.set_xlabel("Recording length (seconds, clipped at 300)")
    ax.set_ylabel("Density")
    ax.set_title("Recording length by species")
    ax.legend(fontsize=8)
    panel_label(ax, "B")

    ax = axes[1, 0]
    counts = seg_meta.groupby("recording_id").size()
    ax.hist(counts, bins=min(30, max(5, counts.nunique())), color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.3)
    ax.axvline(counts.median(), color=Palette.ORANGE, linewidth=3, label=f"median = {counts.median():.0f}")
    ax.set_xlabel("Segments per recording")
    ax.set_ylabel("Recordings")
    ax.set_title("Segments per recording")
    ax.legend()
    panel_label(ax, "C")

    ax = axes[1, 1]
    funnel = [("all recordings", len(catalog)),
              ("clear strict label", int(catalog["behaviour_strict"].notna().sum())),
              ("clear expanded label", int(catalog["behaviour_expanded"].notna().sum()))]
    for b in BEHAVIOURS:
        funnel.append((f"{b} (strict)", int((catalog["behaviour_strict"] == b).sum())))
    colors = [Palette.BLUE_DARK, Palette.BLUE, Palette.BLUE] + [Palette.BEHAVIOUR[b] for b in BEHAVIOURS]
    bars = ax.barh([f[0] for f in funnel][::-1], [f[1] for f in funnel][::-1], color=colors[::-1], edgecolor=Palette.INK, linewidth=1.3)
    annotate_bars(ax, bars, "{:.0f}", horizontal=True)
    ax.set_xlabel("Recordings")
    ax.set_title("Label availability funnel")
    panel_label(ax, "D")
    return VAULT.finish(fig, "corpus_overview", "Corpus overview", section="EDA")


figure_corpus_overview()

## 6.2 Behaviour labels and the call type vocabulary

Panel A shows how many recordings of each species fall into each weak label class. Panel B shows the most common raw phrases that uploaders wrote. Panel C shows how much the split by recording is balanced.

In [ ]:
def figure_label_coverage():
    fig, axes = VAULT.canvas(1, 3, panel=(7.0, 5.0))
    ax = axes[0]
    table = (catalog.assign(b=catalog["behaviour_strict"].fillna("unlabelled"))
             .pivot_table(index="species", columns="b", values="recording_id", aggfunc="count", fill_value=0))
    order = [b for b in BEHAVIOURS if b in table.columns] + [c for c in table.columns if c not in BEHAVIOURS]
    bottoms = np.zeros(len(table))
    for b in order:
        color = Palette.BEHAVIOUR.get(b, "#B8BEC6")
        ax.bar(range(len(table)), table[b].to_numpy(), bottom=bottoms, color=color, label=b, edgecolor=Palette.INK, linewidth=1.3)
        bottoms += table[b].to_numpy()
    ax.set_xticks(range(len(table)))
    ax.set_xticklabels([s.replace(" ", "\n") for s in table.index], fontsize=8.5)
    ax.set_ylabel("Recordings")
    ax.set_title("Weak labels per species")
    ax.legend()
    panel_label(ax, "A")

    ax = axes[1]
    phrases = Counter()
    for text in catalog["call_type"].fillna(""):
        for p in re.split(r"[,;]", str(text).lower()):
            p = p.strip()
            if p:
                phrases[p] += 1
    top = phrases.most_common(14)[::-1]
    if top:
        labels = [t[0][:26] for t in top]
        colors = [Palette.BEHAVIOUR.get(BehaviourLabeler().vote_phrase(t[0]) or "", "#B8BEC6") for t in top]
        bars = ax.barh(labels, [t[1] for t in top], color=colors, edgecolor=Palette.INK, linewidth=1.3)
        annotate_bars(ax, bars, "{:.0f}", horizontal=True)
    ax.set_xlabel("Recordings")
    ax.set_title("Most common call type phrases")
    panel_label(ax, "B")

    ax = axes[2]
    split_table = catalog.pivot_table(index="species", columns="split", values="recording_id", aggfunc="count", fill_value=0)
    split_table = split_table[[c for c in ["train", "val", "test"] if c in split_table.columns]]
    x = np.arange(len(split_table))
    width = 0.26
    for i, col in enumerate(split_table.columns):
        bars = ax.bar(x + (i - 1) * width, split_table[col], width, label=col, color=[Palette.BLUE, Palette.ORANGE, Palette.GREEN][i], edgecolor=Palette.INK, linewidth=1.3)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace(" ", "\n") for s in split_table.index], fontsize=8.5)
    ax.set_ylabel("Recordings")
    ax.set_title("Recording-level split")
    ax.legend(title="Split")
    panel_label(ax, "C")
    return VAULT.finish(fig, "label_coverage", "Behaviour labels, vocabulary and splits", section="EDA")


figure_label_coverage()

## 6.3 What do the sounds look like?

The figure shows one waveform and one mel spectrogram for each weak behaviour class. The colour map runs from deep blue to orange, so loud time-frequency regions glow orange.

In [ ]:
def mel_db(wave, sr, n_mels=96):
    mel = librosa.feature.melspectrogram(y=wave, sr=sr, n_fft=1024, hop_length=256, n_mels=n_mels, fmin=500, fmax=sr / 2)
    return librosa.power_to_db(mel, ref=np.max)


def pick_gallery_segments(seed=CFG.seed):
    rng = np.random.default_rng(seed)
    picks = {}
    for b in BEHAVIOURS:
        pool = seg_meta[(seg_meta["behaviour_strict"] == b) & (seg_meta["valid_length"] >= 0.9 * preprocessor.segment_length)]
        if len(pool):
            picks[b] = int(rng.choice(pool["seg_pos"].to_numpy()))
    return picks


def figure_gallery():
    picks = pick_gallery_segments()
    if not picks:
        print("no labelled segments for the gallery")
        return
    fig, axes = VAULT.canvas(2, len(picks), panel=(6.0, 3.6), squeeze=False)
    for j, (b, pos) in enumerate(picks.items()):
        wave, valid = store.get([pos])
        wave = wave[0][:valid[0]]
        t = np.arange(len(wave)) / CFG.sample_rate
        ax = axes[0, j]
        ax.plot(t, wave, color=Palette.BEHAVIOUR[b], linewidth=0.8)
        ax.fill_between(t, wave, color=Palette.BEHAVIOUR_LIGHT[b], alpha=0.6)
        ax.set_title(f"{b.capitalize()} | {seg_meta.loc[pos, 'species']}", fontsize=11)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude")
        ax = axes[1, j]
        spec = mel_db(wave, CFG.sample_rate)
        image = ax.imshow(spec, origin="lower", aspect="auto", cmap=Palette.spectrogram(), extent=[0, t[-1], 500, CFG.sample_rate / 2], vmin=-70, vmax=0)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
        ax.grid(False)
        if j == len(picks) - 1:
            cbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)
            cbar.set_label("dB")
            cbar.outline.set_edgecolor(Palette.INK)
    return VAULT.finish(fig, "acoustic_gallery", "Waveforms and mel spectrograms by weak behaviour class", section="EDA")


figure_gallery()

## 6.4 Simple acoustic descriptors

Four classic descriptors are measured on a random sample of segments: spectral centroid (brightness), spectral bandwidth (spread), zero-crossing rate (noisiness) and RMS energy (loudness). If these already differ between species and behaviours, then the learned encoders have something to work with.

In [ ]:
class DescriptorExtractor:
    def __init__(self, sr):
        self.sr = sr

    def describe(self, wave):
        centroid = librosa.feature.spectral_centroid(y=wave, sr=self.sr)[0]
        bandwidth = librosa.feature.spectral_bandwidth(y=wave, sr=self.sr)[0]
        zcr = librosa.feature.zero_crossing_rate(wave)[0]
        rms = librosa.feature.rms(y=wave)[0]
        return {"centroid_hz": float(np.mean(centroid)), "bandwidth_hz": float(np.mean(bandwidth)), "zero_crossing": float(np.mean(zcr)), "rms": float(np.mean(rms))}

    def table(self, positions):
        rows = []
        for pos in positions:
            wave, valid = store.get([pos])
            row = self.describe(wave[0][:valid[0]])
            row["seg_pos"] = int(pos)
            rows.append(row)
        return pd.DataFrame(rows).merge(seg_meta[["seg_pos", "species", "behaviour_strict"]], on="seg_pos")


def figure_descriptors():
    rng = np.random.default_rng(CFG.seed)
    sample = rng.choice(len(seg_meta), size=min(1200, len(seg_meta)), replace=False)
    frame = DescriptorExtractor(CFG.sample_rate).table(sample)
    fig, axes = VAULT.canvas(2, 4, panel=(4.6, 4.2))
    columns = ["centroid_hz", "bandwidth_hz", "zero_crossing", "rms"]
    titles = ["Spectral centroid (Hz)", "Spectral bandwidth (Hz)", "Zero-crossing rate", "RMS energy"]
    for j, (col, title) in enumerate(zip(columns, titles)):
        ax = axes[0, j]
        sns.violinplot(data=frame, x="species", y=col, order=species_names, palette=[SPECIES_COLORS[s] for s in species_names], ax=ax, inner="quartile", linewidth=1.3, cut=0)
        ax.set_xticklabels([s.split(" ")[0] for s in species_names], rotation=25, fontsize=8.5)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title(title, fontsize=11.5)
        ax = axes[1, j]
        labelled = frame[frame["behaviour_strict"].isin(BEHAVIOURS)]
        if len(labelled):
            present = [b for b in BEHAVIOURS if b in set(labelled["behaviour_strict"])]
            sns.boxplot(data=labelled, x="behaviour_strict", y=col, order=present, palette=[Palette.BEHAVIOUR[b] for b in present], ax=ax, linewidth=1.5, fliersize=2)
        ax.set_xlabel("")
        ax.set_ylabel("")
    axes[0, 0].set_ylabel("By species", fontsize=12)
    axes[1, 0].set_ylabel("By weak behaviour", fontsize=12)
    VAULT.finish(fig, "descriptors", "Classic acoustic descriptors", section="EDA")
    return frame


descriptor_frame = figure_descriptors()
stat_rows = []
for col in ["centroid_hz", "bandwidth_hz", "zero_crossing", "rms"]:
    groups = [g[col].to_numpy() for _, g in descriptor_frame.groupby("species") if len(g) > 2]
    if len(groups) > 1:
        h, p = sps.kruskal(*groups)
        stat_rows.append({"descriptor": col, "kruskal_H_species": h, "p_value": p})
TABLES.show(pd.DataFrame(stat_rows), "descriptor_tests", "Table 3. Do simple descriptors differ across species? (Kruskal-Wallis)")

## 6.5 Dataset summary tables

In [ ]:
per_species = (catalog.groupby("species").agg(recordings=("recording_id", "count"),
                                              median_length_s=("length_seconds", "median"),
                                              labelled_strict=("behaviour_strict", lambda s: int(s.notna().sum())),
                                              labelled_expanded=("behaviour_expanded", lambda s: int(s.notna().sum()))).reset_index())
per_species["segments"] = per_species["species"].map(seg_meta.groupby("species").size()).fillna(0).astype(int)
totals = {"species": "TOTAL", "recordings": int(per_species["recordings"].sum()), "median_length_s": float(catalog["length_seconds"].median()),
          "labelled_strict": int(per_species["labelled_strict"].sum()), "labelled_expanded": int(per_species["labelled_expanded"].sum()),
          "segments": int(per_species["segments"].sum())}
per_species = pd.concat([per_species, pd.DataFrame([totals])], ignore_index=True)
TABLES.show(per_species, "dataset_summary", "Table 4. Dataset summary", precision=1)
RESULTS["dataset"] = totals

# Part 7: Encoders

## 7.1 Encoder classes

Every encoder sits behind the same tiny interface: give it a batch of waveforms, get back the hidden states of all transformer layers. Three families are supported:

- **Hugging Face** models: HuBERT base, wav2vec2 base, wav2vec2 large (trained on human speech).
- **BirdAVES** models: wav2vec2-style models trained on bird and animal sound (loaded with `torchaudio`, with the `aves` package as backup).
- **Tiny random encoders**, only for the `smoke` preset.

`pool_hidden` turns the frame sequence into one vector per segment (mean, mean plus standard deviation, or mean plus max) and ignores the zero-padded tail of short segments.

`EmbeddingExtractor` runs a whole set of segments through an encoder in batches and can return **several layers and several pooling types in a single pass**. This makes the layer and pooling experiments cheap.

In [ ]:
@dataclass(frozen=True)
class EncoderSpec:
    name: str
    family: str
    source: str
    dim: int
    n_layers: int
    config_url: str = ""
    weights_url: str = ""
    seed: int = 0


AVES_BASE_URL = "https://storage.googleapis.com/esp-public-files/birdaves/"

ENCODER_CATALOG = {
    "hubert-base": EncoderSpec("hubert-base", "hf", "facebook/hubert-base-ls960", 768, 12),
    "wav2vec2-base": EncoderSpec("wav2vec2-base", "hf", "facebook/wav2vec2-base-960h", 768, 12),
    "wav2vec2-large-lv60": EncoderSpec("wav2vec2-large-lv60", "hf", "facebook/wav2vec2-large-960h-lv60-self", 1024, 24),
    "birdaves-base": EncoderSpec("birdaves-base", "aves", "birdaves-biox-base", 768, 12,
                                 AVES_BASE_URL + "birdaves-biox-base.torchaudio.model_config.json",
                                 AVES_BASE_URL + "birdaves-biox-base.torchaudio.pt"),
    "birdaves-large": EncoderSpec("birdaves-large", "aves", "birdaves-biox-large", 1024, 24,
                                  AVES_BASE_URL + "birdaves-biox-large.torchaudio.model_config.json",
                                  AVES_BASE_URL + "birdaves-biox-large.torchaudio.pt"),
    "tiny-a": EncoderSpec("tiny-a", "synthetic", "tiny", 32, 3, seed=1),
    "tiny-b": EncoderSpec("tiny-b", "synthetic", "tiny", 48, 2, seed=2),
    "tiny-c": EncoderSpec("tiny-c", "synthetic", "tiny", 24, 4, seed=3),
}

FRAME_STRIDE = 320


class EncoderBase:
    normalise_input = False

    def __init__(self, spec):
        self.spec = spec
        self.name = spec.name
        self.dim = spec.dim
        self.n_layers = spec.n_layers

    def hidden_states(self, waves):
        raise NotImplementedError

    def release(self):
        for attr in ("model", "wrapper"):
            if hasattr(self, attr):
                delattr(self, attr)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


class HuggingFaceEncoder(EncoderBase):
    def __init__(self, spec):
        super().__init__(spec)
        from transformers import AutoModel, AutoFeatureExtractor
        self.model = AutoModel.from_pretrained(spec.source).to(DEVICE).eval()
        try:
            extractor = AutoFeatureExtractor.from_pretrained(spec.source)
            self.normalise_input = bool(getattr(extractor, "do_normalize", False))
        except Exception:
            self.normalise_input = False
        self.n_layers = int(getattr(self.model.config, "num_hidden_layers", spec.n_layers))
        self.dim = int(self.model.config.hidden_size)

    def hidden_states(self, waves):
        output = self.model(waves, output_hidden_states=True)
        return list(output.hidden_states[1:])


class AvesEncoder(EncoderBase):
    def __init__(self, spec, cache_dir):
        super().__init__(spec)
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        config_path = self._fetch(spec.config_url, cache_dir / f"{spec.source}.model_config.json")
        weights_path = self._fetch(spec.weights_url, cache_dir / f"{spec.source}.torchaudio.pt")
        self.mode = "torchaudio"
        try:
            import torchaudio
            with open(config_path) as handle:
                config = json.load(handle)
            config.pop("aux_num_out", None)
            model = torchaudio.models.wav2vec2_model(**config, aux_num_out=None)
            model.load_state_dict(torch.load(weights_path, map_location="cpu"))
            self.model = model.to(DEVICE).eval()
        except Exception as err:
            print(f"  torchaudio loader failed for {spec.name} ({err}); trying the aves package")
            ensure_packages([("aves", "esp-aves")])
            from aves import load_feature_extractor
            self.wrapper = load_feature_extractor(config_path=str(config_path), model_path=str(weights_path), device=str(DEVICE), for_inference=True)
            self.mode = "aves_package"

    @staticmethod
    def _fetch(url, target):
        import requests
        if target.exists() and target.stat().st_size > 0:
            return target
        print(f"  downloading {target.name} ...")
        response = requests.get(url, timeout=300, stream=True)
        response.raise_for_status()
        with open(target, "wb") as handle:
            for chunk in response.iter_content(chunk_size=1 << 20):
                handle.write(chunk)
        return target

    def hidden_states(self, waves):
        if self.mode == "torchaudio":
            features, _ = self.model.extract_features(waves)
            return list(features)
        outputs = []
        for wave in waves:
            feats = self.wrapper.extract_features(wave, layers=-1)
            outputs.append(feats.reshape(-1, feats.shape[-1]))
        return [torch.stack(outputs)]


class TinySyntheticEncoder(EncoderBase):
    def __init__(self, spec):
        super().__init__(spec)
        from transformers import HubertConfig, HubertModel
        torch.manual_seed(1000 + spec.seed)
        config = HubertConfig(hidden_size=spec.dim, num_hidden_layers=spec.n_layers, num_attention_heads=2, intermediate_size=spec.dim * 2,
                              conv_dim=(16,) * 7, conv_stride=(5, 2, 2, 2, 2, 2, 2), conv_kernel=(10, 3, 3, 3, 3, 2, 2),
                              num_feat_extract_layers=7, feat_extract_norm="group", num_conv_pos_embeddings=16, num_conv_pos_embedding_groups=2)
        self.model = HubertModel(config).to(DEVICE).eval()
        self.normalise_input = True

    def hidden_states(self, waves):
        output = self.model(waves, output_hidden_states=True)
        return list(output.hidden_states[1:])


def build_encoder(name, paths):
    spec = ENCODER_CATALOG[name]
    if spec.family == "hf":
        return HuggingFaceEncoder(spec)
    if spec.family == "aves":
        return AvesEncoder(spec, paths.models / "aves_cache")
    return TinySyntheticEncoder(spec)


def pool_hidden(hidden, n_valid, mode):
    batch, steps, dim = hidden.shape
    positions = torch.arange(steps, device=hidden.device)[None, :]
    mask = (positions < n_valid[:, None]).unsqueeze(-1).float()
    h = hidden.float()
    denom = mask.sum(1).clamp_min(1.0)
    mean = (h * mask).sum(1) / denom
    if mode == "mean":
        return mean
    if mode == "mean_std":
        variance = (((h - mean[:, None, :]) ** 2) * mask).sum(1) / denom
        return torch.cat([mean, torch.sqrt(variance + 1e-6)], dim=1)
    if mode == "mean_max":
        peak = h.masked_fill(mask == 0, -1e9).max(1).values
        return torch.cat([mean, peak], dim=1)
    raise ValueError(mode)


class EmbeddingExtractor:
    def __init__(self, encoder, batch_size=16, use_fp16=True):
        self.encoder = encoder
        self.batch_size = batch_size
        self.use_amp = bool(use_fp16 and DEVICE.type == "cuda")

    def resolve_layer(self, layer, n_available):
        if layer in (-1, "last", None):
            return n_available - 1
        return int(min(max(layer, 1), n_available) - 1)

    def _prepare(self, waves, valid):
        tensor = torch.from_numpy(waves).to(DEVICE)
        lengths = torch.from_numpy(np.asarray(valid)).to(DEVICE)
        if self.encoder.normalise_input:
            positions = torch.arange(tensor.shape[1], device=DEVICE)[None, :]
            mask = (positions < lengths[:, None]).float()
            mean = (tensor * mask).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp_min(1)
            var = (((tensor - mean) ** 2) * mask).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp_min(1)
            tensor = (tensor - mean) / torch.sqrt(var + 1e-7) * mask
        frames = torch.clamp(lengths // FRAME_STRIDE, min=1)
        return tensor, frames

    @torch.no_grad()
    def extract(self, source, indices, layers=(-1,), poolings=("mean_std",), progress=True, desc="embedding"):
        indices = np.asarray(indices, dtype=np.int64)
        keys = [(l, p) for l in layers for p in poolings]
        collected = {k: [] for k in keys}
        starts = range(0, len(indices), self.batch_size)
        iterator = tqdm(starts, desc=f"{desc} [{self.encoder.name}]", leave=False) if progress and len(indices) > self.batch_size * 3 else starts
        for start in iterator:
            waves, valid = source.get(indices[start:start + self.batch_size])
            tensor, frames = self._prepare(waves, valid)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=self.use_amp):
                states = self.encoder.hidden_states(tensor)
            for layer in layers:
                hidden = states[self.resolve_layer(layer, len(states))]
                steps = hidden.shape[1]
                n_valid = torch.clamp(frames, max=steps)
                for pooling in poolings:
                    collected[(layer, pooling)].append(pool_hidden(hidden, n_valid, pooling).cpu().numpy())
        return {k: np.nan_to_num(np.concatenate(v, axis=0)).astype(np.float32) for k, v in collected.items()}

    @torch.no_grad()
    def frame_grid(self, source, indices, layer=-1, steps=25):
        indices = np.asarray(indices, dtype=np.int64)
        grids = []
        for start in range(0, len(indices), self.batch_size):
            waves, valid = source.get(indices[start:start + self.batch_size])
            tensor, _ = self._prepare(waves, valid)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=self.use_amp):
                states = self.encoder.hidden_states(tensor)
            hidden = states[self.resolve_layer(layer, len(states))].float()
            pooled = F.adaptive_avg_pool1d(hidden.transpose(1, 2), steps).transpose(1, 2)
            grids.append(pooled.cpu().numpy().astype(np.float16))
        return np.concatenate(grids, axis=0)

    @torch.no_grad()
    def frames_raw(self, source, indices, layer=-1):
        indices = np.asarray(indices, dtype=np.int64)
        outputs = []
        for start in range(0, len(indices), self.batch_size):
            waves, valid = source.get(indices[start:start + self.batch_size])
            tensor, frames = self._prepare(waves, valid)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=self.use_amp):
                states = self.encoder.hidden_states(tensor)
            hidden = states[self.resolve_layer(layer, len(states))].float().cpu().numpy()
            for row, count in zip(hidden, frames.cpu().numpy()):
                outputs.append(row[:int(min(count, row.shape[0]))].astype(np.float16))
        return outputs


class EncoderPool:
    def __init__(self, paths):
        self.paths = paths
        self.loaded = {}
        self.failed = {}

    def get(self, name):
        if name in self.failed:
            return None
        if name not in self.loaded:
            for other in list(self.loaded):
                self.loaded[other].release()
                del self.loaded[other]
            try:
                self.loaded[name] = build_encoder(name, self.paths)
            except Exception as err:
                self.failed[name] = str(err)
                print(f"  encoder {name} is not available: {str(err)[:160]}")
                return None
        return self.loaded[name]

    def extractor(self, name, cfg):
        encoder = self.get(name)
        return None if encoder is None else EmbeddingExtractor(encoder, cfg.encoder_batch_size, cfg.use_fp16)

    def layer_ids(self, name, fractions):
        encoder = self.get(name)
        n = encoder.n_layers
        ids = sorted({max(1, int(round(f * n))) for f in fractions})
        return ids


ENCODERS = EncoderPool(PATHS)
probe_extractor = ENCODERS.extractor(CFG.encoder_candidates[0], CFG)
demo = probe_extractor.extract(store, np.arange(min(20, len(store))), layers=(1, -1), poolings=("mean", "mean_std", "mean_max"), progress=False)
print({str(k): v.shape for k, v in demo.items()})

# Part 8: Evaluation toolkit

## 8.1 Geometry, clustering scores, ABX and grouped bootstrap

These classes are used by every experiment later, so all numbers are computed in exactly the same way.

- `EmbeddingSpace`: standardise then PCA, with a minimum and maximum number of components and a check for collapsed embeddings.
- `ClusterScorer`: silhouette, Davies-Bouldin and Calinski-Harabasz (internal scores), plus NMI and ARI against species or behaviour (external scores, used only for evaluation), plus bootstrap intervals over points.
- `ABXEvaluator`: makes triplets (A, B, X) where A and X share a species **but come from different recordings** and B is another species. The score is how often X is closer to A than to B. Chance is 0.5.
- `GroupedBootstrap`: confidence intervals for classification metrics that resample **recordings** instead of segments, because segments from one recording are not independent.

In [ ]:
def effective_rank(X, max_rows=3000, seed=0):
    X = np.asarray(X, dtype=np.float64)
    if len(X) > max_rows:
        X = X[np.random.default_rng(seed).choice(len(X), max_rows, replace=False)]
    X = X - X.mean(0, keepdims=True)
    singular = np.linalg.svd(X, compute_uv=False)
    energy = singular ** 2
    total = energy.sum()
    if total <= 0:
        return 0.0
    p = energy / total
    p = p[p > 1e-12]
    return float(np.exp(-(p * np.log(p)).sum()))


def diversity_report(X, name="embeddings"):
    X = np.nan_to_num(np.asarray(X, dtype=np.float32))
    std = X.std(0)
    report = {"name": name, "rows": int(X.shape[0]), "dims": int(X.shape[1]), "active_dims": int((std > 1e-6).sum()),
              "effective_rank": effective_rank(X)}
    if X.shape[0] > 5 and X.shape[1] > 1:
        Xs = StandardScaler().fit_transform(X)
        pca = PCA(n_components=min(10, Xs.shape[0] - 1, Xs.shape[1]), random_state=0).fit(Xs)
        ratios = pca.explained_variance_ratio_
        report["top1_variance"] = float(ratios[0])
        report["top2_variance"] = float(ratios[:2].sum())
    return report


class EmbeddingSpace:
    def __init__(self, variance=0.95, min_components=16, max_components=128, whiten=False, use_pca=True, seed=42):
        self.variance = variance
        self.min_components = min_components
        self.max_components = max_components
        self.whiten = whiten
        self.use_pca = use_pca
        self.seed = seed
        self.scaler = StandardScaler()
        self.pca = None

    @classmethod
    def from_config(cls, cfg, **overrides):
        settings = dict(variance=cfg.pca_variance, min_components=cfg.pca_min_components, max_components=cfg.pca_max_components, seed=cfg.seed)
        settings.update(overrides)
        return cls(**settings)

    def fit(self, X):
        X = np.nan_to_num(np.asarray(X, dtype=np.float32))
        scaled = self.scaler.fit_transform(X)
        if not self.use_pca:
            return self
        ceiling = int(min(self.max_components, scaled.shape[0] - 1, scaled.shape[1]))
        probe = PCA(n_components=ceiling, random_state=self.seed).fit(scaled)
        cumulative = np.cumsum(probe.explained_variance_ratio_)
        wanted = int(np.searchsorted(cumulative, self.variance) + 1)
        n = int(min(max(wanted, min(self.min_components, ceiling)), ceiling))
        self.pca = PCA(n_components=n, whiten=self.whiten, random_state=self.seed).fit(scaled)
        return self

    def transform(self, X):
        X = np.nan_to_num(np.asarray(X, dtype=np.float32))
        scaled = self.scaler.transform(X)
        return np.ascontiguousarray(self.pca.transform(scaled) if self.pca is not None else scaled, dtype=np.float32)

    def fit_transform(self, X):
        return self.fit(X).transform(X)


class ClusterScorer:
    def __init__(self, sample_points=3000, seed=42):
        self.sample_points = sample_points
        self.seed = seed

    def internal(self, X, labels):
        labels = np.asarray(labels)
        usable = labels >= 0
        if usable.sum() < 10 or len(np.unique(labels[usable])) < 2:
            return {"silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan}
        Xu, lu = X[usable], labels[usable]
        n = len(Xu)
        size = min(self.sample_points, n)
        if len(np.unique(lu)) >= size:
            return {"silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan}
        try:
            sil = float(silhouette_score(Xu, lu, sample_size=size if size < n else None, random_state=self.seed))
        except Exception:
            sil = np.nan
        return {"silhouette": sil, "davies_bouldin": float(davies_bouldin_score(Xu, lu)), "calinski_harabasz": float(calinski_harabasz_score(Xu, lu))}

    @staticmethod
    def external(labels, reference):
        labels = np.asarray(labels)
        reference = np.asarray(reference)
        usable = (labels >= 0) & pd.notna(reference)
        if usable.sum() < 5:
            return {"nmi": np.nan, "ari": np.nan, "purity": np.nan}
        l, r = labels[usable], reference[usable]
        table = pd.crosstab(l, r)
        return {"nmi": float(normalized_mutual_info_score(r, l)), "ari": float(adjusted_rand_score(r, l)),
                "purity": float(table.max(axis=1).sum() / table.values.sum())}

    def bootstrap(self, X, labels, n_boot=200, sample_size=1200, seed=None):
        rng = np.random.default_rng(self.seed if seed is None else seed)
        labels = np.asarray(labels)
        usable = np.where(labels >= 0)[0]
        if len(usable) < 30:
            return {m: (np.nan, np.nan, np.nan) for m in ["silhouette", "davies_bouldin", "calinski_harabasz"]}
        draws = {"silhouette": [], "davies_bouldin": [], "calinski_harabasz": []}
        size = min(sample_size, len(usable))
        for _ in range(n_boot):
            pick = rng.choice(usable, size, replace=True)
            lab = labels[pick]
            if len(np.unique(lab)) < 2 or len(np.unique(lab)) >= size:
                continue
            try:
                draws["silhouette"].append(silhouette_score(X[pick], lab))
                draws["davies_bouldin"].append(davies_bouldin_score(X[pick], lab))
                draws["calinski_harabasz"].append(calinski_harabasz_score(X[pick], lab))
            except Exception:
                continue
        return {m: ((float(np.mean(v)), float(np.quantile(v, 0.025)), float(np.quantile(v, 0.975))) if len(v) > 5 else (np.nan, np.nan, np.nan)) for m, v in draws.items()}


class ABXEvaluator:
    def __init__(self, labels, groups, seed=42):
        self.labels = np.asarray(labels)
        self.groups = np.asarray(groups)
        self.seed = seed
        self.index = {c: np.where(self.labels == c)[0] for c in np.unique(self.labels)}
        self.classes = [c for c, idx in self.index.items() if len(np.unique(self.groups[idx])) >= 2]

    def triplets(self, n, across_groups=True, seed=None):
        rng = np.random.default_rng(self.seed if seed is None else seed)
        if len(self.classes) < 2:
            return np.zeros((0, 3), dtype=np.int64)
        out, attempts = [], 0
        while len(out) < n and attempts < 60:
            attempts += 1
            need = n - len(out)
            pair_a = rng.integers(0, len(self.classes), size=need)
            pair_b = (pair_a + rng.integers(1, len(self.classes), size=need)) % len(self.classes)
            for ia, ib in zip(pair_a, pair_b):
                pool_a = self.index[self.classes[ia]]
                a = int(rng.choice(pool_a))
                if across_groups:
                    candidates = pool_a[self.groups[pool_a] != self.groups[a]]
                    if len(candidates) == 0:
                        continue
                    x = int(rng.choice(candidates))
                else:
                    candidates = pool_a[pool_a != a]
                    if len(candidates) == 0:
                        continue
                    x = int(rng.choice(candidates))
                b = int(rng.choice(self.index[self.classes[ib]]))
                out.append((a, b, x))
        return np.asarray(out, dtype=np.int64)

    @staticmethod
    def correct(X, triplets, metric="cosine"):
        X = np.asarray(X, dtype=np.float32)
        a, b, x = triplets[:, 0], triplets[:, 1], triplets[:, 2]
        if metric == "cosine":
            Z = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
            d_ax = 1.0 - np.sum(Z[a] * Z[x], axis=1)
            d_bx = 1.0 - np.sum(Z[b] * Z[x], axis=1)
        else:
            d_ax = np.linalg.norm(X[a] - X[x], axis=1)
            d_bx = np.linalg.norm(X[b] - X[x], axis=1)
        return np.where(d_ax < d_bx, 1.0, np.where(d_ax == d_bx, 0.5, 0.0))

    def score(self, X, triplets, n_boot=300, seed=0, metric="cosine"):
        hits = self.correct(X, triplets, metric)
        if len(hits) == 0:
            return np.nan, np.nan, np.nan
        rng = np.random.default_rng(seed)
        boots = [hits[rng.integers(0, len(hits), len(hits))].mean() for _ in range(n_boot)]
        return float(hits.mean()), float(np.quantile(boots, 0.025)), float(np.quantile(boots, 0.975))


def confusion_metrics(cm):
    cm = np.asarray(cm, dtype=np.float64)
    total = cm.sum()
    if total <= 0:
        return {k: np.nan for k in ["accuracy", "macro_f1", "balanced_accuracy", "kappa", "mcc"]}
    true_counts, pred_counts, diag = cm.sum(1), cm.sum(0), np.diag(cm)
    accuracy = diag.sum() / total
    denominator = 2 * diag + (pred_counts - diag) + (true_counts - diag)
    f1 = np.where(denominator > 0, 2 * diag / np.maximum(denominator, 1e-12), 0.0)
    present = true_counts > 0
    recall = np.where(true_counts > 0, diag / np.maximum(true_counts, 1e-12), 0.0)
    pe = (true_counts * pred_counts).sum() / total ** 2
    kappa = (accuracy - pe) / (1 - pe) if pe < 1 else 0.0
    s = total
    denom_mcc = math.sqrt(max((s ** 2 - (pred_counts ** 2).sum()) * (s ** 2 - (true_counts ** 2).sum()), 0.0))
    mcc = (diag.sum() * s - (pred_counts * true_counts).sum()) / denom_mcc if denom_mcc > 0 else 0.0
    return {"accuracy": float(accuracy), "macro_f1": float(f1[present].mean()), "balanced_accuracy": float(recall[present].mean()),
            "kappa": float(kappa), "mcc": float(mcc)}


class GroupedBootstrap:
    METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "kappa", "mcc"]

    def __init__(self, y_true, predictions, groups, n_classes):
        y_true = np.asarray(y_true)
        groups = np.asarray(groups)
        if isinstance(predictions, np.ndarray) and predictions.ndim == 1:
            predictions = [predictions]
        self.unique_groups = np.unique(groups)
        self.matrices = np.zeros((len(self.unique_groups), n_classes, n_classes))
        lookup = {g: i for i, g in enumerate(self.unique_groups)}
        gi = np.array([lookup[g] for g in groups])
        for pred in predictions:
            np.add.at(self.matrices, (gi, y_true, np.asarray(pred)), 1.0 / len(predictions))

    def point(self):
        return confusion_metrics(self.matrices.sum(0))

    def interval(self, n_boot=500, seed=0):
        rng = np.random.default_rng(seed)
        G = len(self.unique_groups)
        draws = {m: [] for m in self.METRICS}
        for _ in range(n_boot):
            weights = rng.multinomial(G, np.full(G, 1.0 / G))
            cm = np.tensordot(weights, self.matrices, axes=1)
            result = confusion_metrics(cm)
            for m in self.METRICS:
                draws[m].append(result[m])
        return {m: (float(np.nanquantile(v, 0.025)), float(np.nanquantile(v, 0.975))) for m, v in draws.items()}

    def summary(self, n_boot=500, seed=0):
        point = self.point()
        ci = self.interval(n_boot, seed)
        out = {}
        for m in self.METRICS:
            out[m] = point[m]
            out[f"{m}_lo"], out[f"{m}_hi"] = ci[m]
        return out


def paired_difference(y_true, preds_a, preds_b, groups, n_classes, metric="macro_f1", n_boot=500, seed=0):
    A = GroupedBootstrap(y_true, preds_a, groups, n_classes)
    B = GroupedBootstrap(y_true, preds_b, groups, n_classes)
    rng = np.random.default_rng(seed)
    G = len(A.unique_groups)
    diffs = []
    for _ in range(n_boot):
        w = rng.multinomial(G, np.full(G, 1.0 / G))
        diffs.append(confusion_metrics(np.tensordot(w, A.matrices, axes=1))[metric] - confusion_metrics(np.tensordot(w, B.matrices, axes=1))[metric])
    diffs = np.asarray(diffs)
    point = confusion_metrics(A.matrices.sum(0))[metric] - confusion_metrics(B.matrices.sum(0))[metric]
    p_value = float(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()) + 1.0 / n_boot))
    return {"diff": float(point), "lo": float(np.quantile(diffs, 0.025)), "hi": float(np.quantile(diffs, 0.975)), "p": p_value}

## 8.2 Probes, grouped cross-validation and the MFCC baseline

A **probe** is a simple classifier trained on top of frozen embeddings. If a linear probe can read species or behaviour from the embedding, the embedding contains that information.

- `make_probe` builds nine classifiers with the same interface: logistic regression, SVM, random forest, extra trees, k-nearest neighbours, sklearn MLP, XGBoost, and a PyTorch deep MLP with batch norm (the "deep adapter" from Run 6).
- `CrossValidator` repeats stratified **group** k-fold, where the group is the recording id. It can also run the old **naive** segment-level k-fold, which is used only for the leakage audit.
- `MFCCBaseline` computes the classic hand-made features (20 MFCC plus deltas, mean and standard deviation), the reference that every learned encoder must beat.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import train_test_split


class TorchMLPClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden=(256, 128), dropout=0.25, epochs=80, lr=1e-3, batch_size=64, patience=10, weight_decay=1e-4, seed=42):
        self.hidden = hidden
        self.dropout = dropout
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.patience = patience
        self.weight_decay = weight_decay
        self.seed = seed

    def _build(self, d_in, n_out):
        layers, prev = [], d_in
        for h in self.hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(self.dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_out))
        return nn.Sequential(*layers)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        X = np.asarray(X, dtype=np.float32)
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        try:
            Xtr, Xva, ytr, yva = train_test_split(X, y_idx, test_size=0.15, stratify=y_idx, random_state=self.seed)
        except ValueError:
            Xtr, Xva, ytr, yva = X, X, y_idx, y_idx
        self.net_ = self._build(X.shape[1], len(self.classes_)).to(DEVICE)
        optimizer = torch.optim.AdamW(self.net_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        weights = torch.tensor(len(ytr) / (len(self.classes_) * np.maximum(np.bincount(ytr, minlength=len(self.classes_)), 1)), dtype=torch.float32, device=DEVICE)
        loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)), batch_size=self.batch_size, shuffle=True, drop_last=len(Xtr) > self.batch_size)
        Xva_t, yva_t = torch.tensor(Xva).to(DEVICE), torch.tensor(yva).to(DEVICE)
        best, best_state, wait = float("inf"), None, 0
        for _ in range(self.epochs):
            self.net_.train()
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                F.cross_entropy(self.net_(xb), yb, weight=weights).backward()
                optimizer.step()
            self.net_.eval()
            with torch.no_grad():
                val = F.cross_entropy(self.net_(Xva_t), yva_t).item()
            if val < best - 1e-4:
                best, wait = val, 0
                best_state = {k: v.detach().clone() for k, v in self.net_.state_dict().items()}
            else:
                wait += 1
                if wait >= self.patience:
                    break
        if best_state is not None:
            self.net_.load_state_dict(best_state)
        return self

    def predict_proba(self, X):
        self.net_.eval()
        with torch.no_grad():
            logits = self.net_(torch.tensor(np.asarray(X, dtype=np.float32)).to(DEVICE))
        return torch.softmax(logits, dim=1).cpu().numpy()

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]


class SoftVoteEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, members=("logreg", "svm_rbf", "random_forest", "mlp", "knn"), seed=42):
        self.members = members
        self.seed = seed

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.models_ = [make_probe(name, self.seed).fit(X, y) for name in self.members]
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models_], axis=0)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]


PROBE_NAMES = ["logreg", "svm_rbf", "random_forest", "extra_trees", "knn", "mlp", "xgboost", "torch_mlp", "ensemble"]


def make_probe(name, seed=42, n_jobs=-1):
    if name == "logreg":
        model = LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5)
    elif name == "svm_rbf":
        model = SVC(kernel="rbf", C=3.0, class_weight="balanced", probability=True, random_state=seed)
    elif name == "random_forest":
        model = RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", n_jobs=n_jobs, random_state=seed)
    elif name == "extra_trees":
        model = ExtraTreesClassifier(n_estimators=400, class_weight="balanced_subsample", n_jobs=n_jobs, random_state=seed)
    elif name == "knn":
        model = KNeighborsClassifier(n_neighbors=15, weights="distance", metric="cosine")
    elif name == "mlp":
        model = MLPClassifier(hidden_layer_sizes=(256, 128), early_stopping=True, n_iter_no_change=10, max_iter=400, random_state=seed)
    elif name == "xgboost":
        if xgb is None:
            raise RuntimeError("xgboost is not installed")
        model = xgb.XGBClassifier(n_estimators=250, max_depth=6, learning_rate=0.06, subsample=0.85, colsample_bytree=0.85,
                                  random_state=seed, eval_metric="mlogloss", n_jobs=n_jobs, verbosity=0)
    elif name == "torch_mlp":
        model = TorchMLPClassifier(seed=seed)
    elif name == "ensemble":
        return SoftVoteEnsemble(seed=seed)
    else:
        raise ValueError(name)
    if name in ("random_forest", "extra_trees", "xgboost"):
        return model
    return make_pipeline(StandardScaler(), model)


def available_probes():
    return [n for n in PROBE_NAMES if not (n == "xgboost" and xgb is None)]


class CrossValidator:
    def __init__(self, folds=5, repeats=2, seed=42, grouped=True, n_boot=500):
        self.folds = folds
        self.repeats = repeats
        self.seed = seed
        self.grouped = grouped
        self.n_boot = n_boot

    def _splits(self, y, groups, repeat):
        seed = self.seed + 101 * repeat
        if self.grouped:
            per_class_groups = min(len(np.unique(groups[y == c])) for c in np.unique(y))
            n_splits = int(max(2, min(self.folds, per_class_groups)))
            splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            return list(splitter.split(np.zeros(len(y)), y, groups))
        n_splits = int(max(2, min(self.folds, np.bincount(y).min())))
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(splitter.split(np.zeros(len(y)), y))

    def run(self, X, y, groups, probe_name, n_classes=None, sample_weight_fn=None, fit_hook=None):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y)
        groups = np.asarray(groups)
        n_classes = n_classes or int(y.max() + 1)
        predictions, probabilities, fold_scores = [], [], []
        for repeat in range(self.repeats):
            oof = np.full(len(y), -1, dtype=int)
            proba = np.zeros((len(y), n_classes), dtype=np.float32)
            for train_idx, test_idx in self._splits(y, groups, repeat):
                if fit_hook is not None:
                    model, Xtr, ytr = fit_hook(probe_name, train_idx, self.seed + repeat)
                else:
                    model, Xtr, ytr = make_probe(probe_name, self.seed + repeat), X[train_idx], y[train_idx]
                model.fit(Xtr, ytr)
                oof[test_idx] = model.predict(X[test_idx])
                if hasattr(model, "predict_proba"):
                    p = model.predict_proba(X[test_idx])
                    classes = getattr(model, "classes_", None)
                    if classes is None and hasattr(model, "steps"):
                        classes = model.steps[-1][1].classes_
                    proba[np.ix_(test_idx, np.asarray(classes, dtype=int))] = p
                fold_scores.append(float(accuracy_score(y[test_idx], oof[test_idx])))
            predictions.append(oof)
            probabilities.append(proba)
        boot = GroupedBootstrap(y, predictions, groups, n_classes)
        summary = boot.summary(self.n_boot, seed=self.seed)
        summary["fold_scores"] = fold_scores
        summary["predictions"] = predictions
        summary["probabilities"] = np.mean(probabilities, axis=0)
        summary["probe"] = probe_name
        summary["cv"] = "grouped" if self.grouped else "naive"
        return summary


class MFCCBaseline:
    def __init__(self, sr=16000, n_mfcc=20):
        self.sr = sr
        self.n_mfcc = n_mfcc

    def one(self, wave):
        mfcc = librosa.feature.mfcc(y=wave, sr=self.sr, n_mfcc=self.n_mfcc, n_fft=1024, hop_length=320)
        delta = librosa.feature.delta(mfcc)
        return np.concatenate([mfcc.mean(1), mfcc.std(1), delta.mean(1), delta.std(1)])

    def extract(self, source, indices, batch=64):
        indices = np.asarray(indices)
        out = []
        for start in range(0, len(indices), batch):
            waves, valid = source.get(indices[start:start + batch])
            for w, v in zip(waves, valid):
                out.append(self.one(w[:int(v)]))
        return np.nan_to_num(np.asarray(out, dtype=np.float32))


class LabelledSetBuilder:
    def __init__(self, seg_meta, cfg, mode="strict"):
        self.meta = seg_meta
        self.cfg = cfg
        self.column = "behaviour_strict" if mode == "strict" else "behaviour_expanded"
        self.classes = list(cfg.behaviour_classes)

    def build(self, per_class=None, per_recording=None, seed=42, splits=None):
        per_class = per_class or self.cfg.labelled_segments_per_class
        per_recording = per_recording or self.cfg.max_segments_per_labelled_recording
        rng = np.random.default_rng(seed)
        frame = self.meta[self.meta[self.column].isin(self.classes)]
        if splits is not None:
            frame = frame[frame["split"].isin(splits)]
        picked = []
        for rid, group in frame.groupby("recording_id"):
            take = group.sample(n=min(per_recording, len(group)), random_state=int(rng.integers(0, 2 ** 31 - 1)))
            picked.append(take)
        if not picked:
            return frame.iloc[0:0].assign(label=[])
        frame = pd.concat(picked)
        counts = frame[self.column].value_counts()
        floor = int(min(per_class, counts.reindex(self.classes).fillna(0).min()))
        if floor < 2:
            return frame.iloc[0:0].assign(label=[])
        balanced = [g.sample(n=floor, random_state=int(rng.integers(0, 2 ** 31 - 1))) for _, g in frame.groupby(self.column)]
        out = pd.concat(balanced).sort_values("seg_pos").reset_index(drop=True)
        out["label"] = out[self.column].map({c: i for i, c in enumerate(self.classes)}).astype(int)
        return out


labelled_strict = LabelledSetBuilder(seg_meta, CFG, "strict").build(seed=CFG.seed)
print("labelled segments:", len(labelled_strict), labelled_strict[["behaviour_strict"]].value_counts().to_dict() if len(labelled_strict) else "none")
print("labelled recordings:", labelled_strict["recording_id"].nunique() if len(labelled_strict) else 0)

## 8.3 Encoder benchmark

The main question of this part: **which encoder, which layer and which pooling give the best bird embedding?**

A fixed **development subset** is drawn from the train and validation recordings (the test recordings stay untouched): a species-balanced random sample of segments plus all the labelled segments. Each candidate encoder embeds this subset once, at several layers and with three pooling types. Then every combination is scored with four label-light measures:

| Measure | What it tells us |
|---|---|
| ABX accuracy | Are two calls of the same species closer than calls of different species? (recording-safe) |
| Species probe (balanced accuracy) | Can a linear classifier read the species from the embedding? |
| Silhouette (K-Means, K = 20) | How compact and separated are the clusters, without labels? |
| Behaviour probe (macro-F1) | Can a linear classifier read weak behaviour labels? |

The **primary encoder** for the rest of the notebook is picked by a rank-average of ABX, species probe and silhouette (behaviour is shown, but it has few labels, so it only gets half a weight).

In [ ]:
class DevelopmentSubset:
    def __init__(self, seg_meta, cfg, seed=42):
        rng = np.random.default_rng(seed)
        dev = seg_meta[seg_meta["split"].isin(["train", "val"])]
        parts = []
        for species, group in dev.groupby("species"):
            n = min(cfg.eval_segments_per_species, len(group))
            parts.append(group.sample(n=n, random_state=int(rng.integers(0, 2 ** 31 - 1))))
        labelled = LabelledSetBuilder(seg_meta, cfg, "strict").build(per_class=cfg.eval_labelled_per_class, seed=seed, splits=["train", "val"])
        if len(labelled):
            parts.append(labelled.drop(columns=["label"], errors="ignore"))
        merged = pd.concat(parts).drop_duplicates("seg_pos").sort_values("seg_pos").reset_index(drop=True)
        self.meta = merged
        self.positions = merged["seg_pos"].to_numpy()
        self.species = merged["species_index"].to_numpy()
        self.groups = merged["recording_id"].to_numpy()
        self.behaviour = merged["behaviour_strict"].map({c: i for i, c in enumerate(cfg.behaviour_classes)}).to_numpy()
        self.behaviour_mask = ~pd.isna(self.behaviour)

    def __len__(self):
        return len(self.positions)


DEV = DevelopmentSubset(seg_meta, CFG, CFG.seed)
print(f"development subset: {len(DEV)} segments | labelled: {int(DEV.behaviour_mask.sum())} | recordings: {len(np.unique(DEV.groups))}")


class SubsetScorer:
    def __init__(self, dev, cfg):
        self.dev = dev
        self.cfg = cfg
        self.abx = ABXEvaluator(dev.species, dev.groups, cfg.seed)
        n_trip = int(min(cfg.abx_triplets, 4000))
        self.triplets = self.abx.triplets(n_trip, across_groups=True, seed=cfg.seed)
        self.scorer = ClusterScorer(cfg.silhouette_points, cfg.seed)

    def species_probe(self, X, folds=3):
        cv = CrossValidator(folds=folds, repeats=1, seed=self.cfg.seed, grouped=True, n_boot=50)
        res = cv.run(X, self.dev.species, self.dev.groups, "logreg", n_classes=len(species_names))
        return res["balanced_accuracy"]

    def behaviour_probe(self, X, folds=3):
        mask = self.dev.behaviour_mask
        if mask.sum() < 12:
            return np.nan
        y = self.dev.behaviour[mask].astype(int)
        g = self.dev.groups[mask]
        if len(np.unique(y)) < 2 or min(len(np.unique(g[y == c])) for c in np.unique(y)) < 2:
            return np.nan
        cv = CrossValidator(folds=folds, repeats=1, seed=self.cfg.seed, grouped=True, n_boot=50)
        return cv.run(X[mask], y, g, "logreg", n_classes=len(self.cfg.behaviour_classes))["macro_f1"]

    def score(self, raw, name, layer, pooling, with_behaviour=True):
        space = EmbeddingSpace.from_config(self.cfg).fit(raw)
        X = space.transform(raw)
        abx, abx_lo, abx_hi = self.abx.score(X, self.triplets, n_boot=200, seed=self.cfg.seed)
        k = int(min(20, max(3, len(X) // 15)))
        labels = KMeans(n_clusters=k, n_init=3, random_state=self.cfg.seed).fit_predict(X)
        internal = self.scorer.internal(X, labels)
        external = self.scorer.external(labels, self.dev.species)
        return {"encoder": name, "layer": layer, "pooling": pooling, "dim": raw.shape[1], "pca_dims": X.shape[1],
                "abx": abx, "abx_lo": abx_lo, "abx_hi": abx_hi, "species_bacc": self.species_probe(X),
                "behaviour_f1": self.behaviour_probe(X) if with_behaviour else np.nan,
                "silhouette": internal["silhouette"], "davies_bouldin": internal["davies_bouldin"],
                "calinski_harabasz": internal["calinski_harabasz"], "species_nmi": external["nmi"],
                "effective_rank": effective_rank(X)}


class EncoderBenchmark:
    POOLINGS = ("mean", "mean_std", "mean_max")

    def __init__(self, cfg, paths, pool, dev, store):
        self.cfg = cfg
        self.paths = paths
        self.pool = pool
        self.dev = dev
        self.store = store
        self.scorer = SubsetScorer(dev, cfg)
        self.bank = {}

    def embed(self, name):
        extractor = self.pool.extractor(name, self.cfg)
        if extractor is None:
            return None
        layers = self.pool.layer_ids(name, self.cfg.layer_fractions)
        return extractor.extract(self.store, self.dev.positions, layers=tuple(layers), poolings=self.POOLINGS, desc="benchmark")

    def run(self):
        rows = []
        for name in self.cfg.encoder_candidates:
            key = f"bench_{self.cfg.preset}_{name}"
            def build(name=name):
                embeddings = self.embed(name)
                if embeddings is None:
                    return None
                return embeddings
            embeddings = CACHE.get_or_compute(key + "_emb", build)
            if embeddings is None:
                continue
            self.bank[name] = embeddings
            for (layer, pooling), raw in tqdm(embeddings.items(), desc=f"scoring {name}", leave=False):
                rows.append(self.scorer.score(raw, name, layer, pooling))
        mfcc_raw = CACHE.get_or_compute(f"bench_{self.cfg.preset}_mfcc", lambda: MFCCBaseline(self.cfg.sample_rate).extract(self.store, self.dev.positions))
        self.bank["mfcc"] = {("-", "mean_std"): mfcc_raw}
        rows.append(self.scorer.score(mfcc_raw, "mfcc", "-", "mean_std"))
        frame = pd.DataFrame(rows)
        return frame

    @staticmethod
    def add_composite(frame):
        out = frame.copy()
        for col in ["abx", "species_bacc", "silhouette", "behaviour_f1"]:
            out[f"rank_{col}"] = out[col].rank(pct=True)
        out["composite"] = (out["rank_abx"] + out["rank_species_bacc"] + out["rank_silhouette"] + 0.5 * out["rank_behaviour_f1"].fillna(out["rank_behaviour_f1"].mean() if out["rank_behaviour_f1"].notna().any() else 0.5)) / 3.5
        return out


with TIMER.track("encoder benchmark"):
    BENCH = EncoderBenchmark(CFG, PATHS, ENCODERS, DEV, store)
    bench_frame = EncoderBenchmark.add_composite(BENCH.run())

learned = bench_frame[bench_frame["encoder"] != "mfcc"]
best_row = learned.sort_values("composite", ascending=False).iloc[0]
PRIMARY = {"encoder": str(best_row["encoder"]), "layer": int(best_row["layer"]), "pooling": str(best_row["pooling"])}
if CFG.primary_encoder != "auto":
    forced = learned[learned["encoder"] == CFG.primary_encoder].sort_values("composite", ascending=False)
    if len(forced):
        best_row = forced.iloc[0]
        PRIMARY = {"encoder": str(best_row["encoder"]), "layer": int(best_row["layer"]), "pooling": str(best_row["pooling"])}
print("PRIMARY encoder configuration:", PRIMARY)
show_cols = ["encoder", "layer", "pooling", "dim", "abx", "species_bacc", "behaviour_f1", "silhouette", "davies_bouldin", "effective_rank", "composite"]
TABLES.show(bench_frame.sort_values("composite", ascending=False)[show_cols].head(25), "encoder_benchmark", "Table 5. Encoder, layer and pooling benchmark (top 25 by composite score)", precision=3)
RESULTS["primary"] = PRIMARY

## 8.4 Encoder benchmark figures

The first figure compares the best configuration of every encoder against the MFCC baseline. The radar chart shows all measures together. The layer-wise figure shows how quality changes from early to late layers. The pooling figure compares mean, mean plus std, and mean plus max. The last figure shows how much variance each embedding keeps in its first principal components.

In [ ]:
FAMILY_COLOR = {"hf": Palette.BLUE, "aves": Palette.ORANGE, "synthetic": Palette.PURPLE, "mfcc": Palette.GREEN}


def encoder_family(name):
    return "mfcc" if name == "mfcc" else ENCODER_CATALOG[name].family


def best_per_encoder(frame):
    idx = frame.groupby("encoder")["composite"].idxmax()
    return frame.loc[idx].sort_values("composite", ascending=False).reset_index(drop=True)


def figure_encoder_comparison(frame):
    best = best_per_encoder(frame)
    fig, axes = VAULT.canvas(1, 4, panel=(5.2, 5.0))
    specs = [("abx", "ABX accuracy", True), ("species_bacc", "Species probe (balanced acc.)", False),
             ("silhouette", "Silhouette (K = 20)", False), ("behaviour_f1", "Behaviour probe (macro-F1)", False)]
    order = best["encoder"].tolist()[::-1]
    for ax, (col, title, has_ci) in zip(axes, specs):
        values = [float(best.loc[best["encoder"] == e, col].iloc[0]) for e in order]
        colors = [FAMILY_COLOR[encoder_family(e)] for e in order]
        bars = ax.barh(order, np.nan_to_num(values), color=colors, edgecolor=Palette.INK, linewidth=1.4)
        if has_ci:
            lo = [float(best.loc[best["encoder"] == e, "abx_lo"].iloc[0]) for e in order]
            hi = [float(best.loc[best["encoder"] == e, "abx_hi"].iloc[0]) for e in order]
            ax.errorbar(values, range(len(order)), xerr=[np.clip(np.array(values) - np.array(lo), 0, None), np.clip(np.array(hi) - np.array(values), 0, None)], fmt="none", ecolor=Palette.INK, capsize=4, linewidth=1.6)
            ax.axvline(0.5, color=Palette.RED, linestyle="--", linewidth=2)
        annotate_bars(ax, bars, "{:.3f}", horizontal=True, fontsize=8.5, offset=14 if has_ci else 3)
        ax.set_title(title, fontsize=11.5)
        ax.set_xlim(0, max(np.nanmax(values) * 1.28, 0.1))
        if ax is not axes[0]:
            ax.set_yticklabels([])
    handles = [mpatches.Patch(facecolor=c, edgecolor=Palette.INK, label=l) for l, c in [("Speech encoders", Palette.BLUE), ("BirdAVES", Palette.ORANGE), ("Tiny random (smoke)", Palette.PURPLE), ("MFCC baseline", Palette.GREEN)]]
    fig.legend(handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 0.012))
    return VAULT.finish(fig, "encoder_comparison", "Encoder comparison at each encoder's best layer and pooling", section="Encoders", bottom=0.09)


def figure_radar(frame):
    best = best_per_encoder(frame)
    metrics = ["abx", "species_bacc", "silhouette", "behaviour_f1", "inv_dbi", "effective_rank"]
    labels = ["ABX", "Species probe", "Silhouette", "Behaviour F1", "1 / Davies-Bouldin", "Effective rank"]
    data = best.assign(inv_dbi=1.0 / best["davies_bouldin"])[metrics].astype(float)
    data = data.fillna(data.mean())
    scaled = (data - data.min()) / (data.max() - data.min() + 1e-9)
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]
    fig = plt.figure(figsize=(9.5, 8.0))
    ax = fig.add_subplot(111, polar=True)
    for i, name in enumerate(best["encoder"]):
        values = scaled.iloc[i].tolist() + scaled.iloc[i].tolist()[:1]
        color = Palette.CATEGORICAL[i % 5]
        ax.plot(angles, values, color=color, linewidth=2.6, label=name)
        ax.fill(angles, values, color=color, alpha=0.14)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=11, fontweight="bold", color=Palette.BLUE_DARK)
    ax.set_yticklabels([])
    ax.spines["polar"].set_edgecolor(Palette.INK)
    ax.spines["polar"].set_linewidth(2)
    ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.12))
    return VAULT.finish(fig, "encoder_radar", "Encoder profile (each axis scaled from worst to best)", section="Encoders")


def figure_layerwise(frame):
    learned_rows = frame[(frame["encoder"] != "mfcc") & (frame["pooling"] == "mean_std")]
    fig, axes = VAULT.canvas(1, 3, panel=(6.2, 4.8))
    for ax, col, title in zip(axes, ["abx", "species_bacc", "silhouette"], ["ABX accuracy", "Species probe", "Silhouette"]):
        for i, (name, group) in enumerate(learned_rows.groupby("encoder", sort=False)):
            group = group.sort_values("layer")
            depth = group["layer"].to_numpy() / group["layer"].max()
            ax.plot(depth, group[col], marker="o", color=Palette.CATEGORICAL[i % 5], label=name)
        mfcc_value = frame.loc[frame["encoder"] == "mfcc", col]
        if len(mfcc_value):
            ax.axhline(float(mfcc_value.iloc[0]), color=Palette.GREEN, linestyle="--", linewidth=2.4, label="MFCC")
        ax.set_xlabel("Relative depth (0 = first layer, 1 = last layer)")
        ax.set_title(title)
    axes[0].axhline(0.5, color=Palette.RED, linestyle=":", linewidth=2)
    axes[0].legend(fontsize=8)
    return VAULT.finish(fig, "layerwise", "Layer-wise quality of each encoder (mean + std pooling)", section="Encoders")


def figure_pooling(frame):
    learned_rows = frame[frame["encoder"] != "mfcc"]
    best = learned_rows.loc[learned_rows.groupby(["encoder", "pooling"])["composite"].idxmax()]
    fig, axes = VAULT.canvas(1, 3, panel=(6.0, 4.6))
    for ax, col, title in zip(axes, ["abx", "species_bacc", "silhouette"], ["ABX accuracy", "Species probe", "Silhouette"]):
        table = best.pivot(index="encoder", columns="pooling", values=col)
        x = np.arange(len(table))
        for j, pooling in enumerate(table.columns):
            ax.bar(x + (j - 1) * 0.27, table[pooling], 0.27, label=pooling, color=[Palette.BLUE, Palette.ORANGE, Palette.GREEN][j % 3], edgecolor=Palette.INK, linewidth=1.3)
        ax.set_xticks(x)
        ax.set_xticklabels(table.index, rotation=25, ha="right", fontsize=9)
        ax.set_title(title)
    axes[0].legend(title="Pooling")
    return VAULT.finish(fig, "pooling", "Pooling comparison at the best layer of each encoder", section="Encoders")


def figure_variance_spectrum(frame, bank):
    best = best_per_encoder(frame)
    fig, axes = VAULT.canvas(1, 2, panel=(7.0, 5.0))
    for i, row in best.iterrows():
        key = ("-", "mean_std") if row["encoder"] == "mfcc" else (int(row["layer"]), row["pooling"])
        raw = bank[row["encoder"]][key]
        Xs = StandardScaler().fit_transform(raw)
        n = int(min(40, Xs.shape[0] - 1, Xs.shape[1]))
        ratio = PCA(n_components=n, random_state=0).fit(Xs).explained_variance_ratio_
        axes[0].plot(np.arange(1, n + 1), np.cumsum(ratio), color=Palette.CATEGORICAL[i % 5], label=row["encoder"])
    axes[0].set_xlabel("Principal components")
    axes[0].set_ylabel("Cumulative explained variance")
    axes[0].set_title("How compact is each embedding?")
    axes[0].legend(fontsize=8)
    bars = axes[1].barh(best["encoder"][::-1], best["effective_rank"][::-1], color=[FAMILY_COLOR[encoder_family(e)] for e in best["encoder"][::-1]], edgecolor=Palette.INK, linewidth=1.4)
    annotate_bars(axes[1], bars, "{:.1f}", horizontal=True)
    axes[1].set_xlabel("Effective rank")
    axes[1].set_title("Effective rank (higher = richer)")
    return VAULT.finish(fig, "variance_spectrum", "Embedding richness", section="Encoders")


figure_encoder_comparison(bench_frame)
figure_radar(bench_frame)
figure_layerwise(bench_frame)
figure_pooling(bench_frame)
figure_variance_spectrum(bench_frame, BENCH.bank)

# Part 9: Held-out evaluation subset

## 9.1 One fixed subset for all later experiments

From here on, every comparison uses the **same fixed subset** so that numbers are comparable:

- **Held-out part**: a species-balanced sample of segments from the **test recordings**. ABX and clustering scores are computed here.
- **Labelled part**: all weak-labelled segments (balanced, capped per recording) from **all recordings**. Behaviour probes are cross-validated here with recording-level folds, and no label is ever used to train an encoder, an adapter, or a clusterer.

`SuiteScorer.evaluate(raw_features)` turns any feature matrix from this subset into one row of metrics with confidence intervals. The 20-experiment suite in Part 15 is built on this class.

In [ ]:
class SuiteSubset:
    def __init__(self, seg_meta, cfg, seed=42):
        rng = np.random.default_rng(seed)
        held_pool = seg_meta[seg_meta["split"] == "test"]
        parts = []
        for species, group in held_pool.groupby("species"):
            parts.append(group.sample(n=min(cfg.eval_segments_per_species, len(group)), random_state=int(rng.integers(0, 2 ** 31 - 1))))
        held = pd.concat(parts)
        labelled = LabelledSetBuilder(seg_meta, cfg, "strict").build(per_class=cfg.labelled_segments_per_class, seed=seed)
        self.labelled_positions = labelled["seg_pos"].to_numpy() if len(labelled) else np.zeros(0, dtype=int)
        self.labelled_labels = labelled["label"].to_numpy() if len(labelled) else np.zeros(0, dtype=int)
        union = sorted(set(held["seg_pos"]).union(self.labelled_positions.tolist()))
        self.positions = np.asarray(union, dtype=np.int64)
        self.meta = seg_meta.iloc[self.positions].reset_index(drop=True)
        held_set = set(held["seg_pos"].tolist())
        lab_set = dict(zip(self.labelled_positions.tolist(), self.labelled_labels.tolist()))
        self.is_held = np.array([p in held_set for p in self.positions])
        self.is_labelled = np.array([p in lab_set for p in self.positions])
        self.behaviour = np.array([lab_set.get(p, -1) for p in self.positions])
        self.species = self.meta["species_index"].to_numpy()
        self.groups = self.meta["recording_id"].to_numpy()

    def __len__(self):
        return len(self.positions)


SUITE = SuiteSubset(seg_meta, CFG, CFG.seed)
print(f"suite subset: {len(SUITE)} segments | held-out: {int(SUITE.is_held.sum())} | labelled: {int(SUITE.is_labelled.sum())} | "
      f"labelled recordings: {len(np.unique(SUITE.groups[SUITE.is_labelled]))}")


class SuiteScorer:
    def __init__(self, suite, cfg, k=None):
        self.suite = suite
        self.cfg = cfg
        held_idx = np.where(suite.is_held)[0]
        self.held_idx = held_idx
        self.abx = ABXEvaluator(suite.species[held_idx], suite.groups[held_idx], cfg.seed)
        self.triplets = self.abx.triplets(int(cfg.abx_triplets), across_groups=True, seed=cfg.seed)
        self.k = k or int(min(30, max(3, len(held_idx) // 15)))
        self.cluster = ClusterScorer(cfg.silhouette_points, cfg.seed)
        self.lab_idx = np.where(suite.is_labelled)[0]
        self.details = {}

    def cluster_labels(self, Xh, method, param, k):
        if method == "kmeans":
            return KMeans(n_clusters=k, n_init=3, random_state=self.cfg.seed).fit_predict(Xh)
        model = make_units(method, self.cfg.vq_epochs).fit(Xh, int(param if method == "hdbscan" else k), self.cfg.seed)
        return model.labels_ if method == "hdbscan" else model.predict(Xh)

    def evaluate(self, raw, tag, extra=None, with_behaviour=True, probe="logreg", n_boot=None, k=None, cluster_method="kmeans", cluster_param=None, space_kwargs=None):
        n_boot = n_boot or self.cfg.bootstrap_iterations
        space = EmbeddingSpace.from_config(self.cfg, **(space_kwargs or {})).fit(raw)
        X = space.transform(raw)
        Xh = X[self.held_idx]
        hits = ABXEvaluator.correct(Xh, self.triplets)
        abx, abx_lo, abx_hi = self.abx.score(Xh, self.triplets, n_boot=min(n_boot, 300), seed=self.cfg.seed)
        k = int(min(k or self.k, max(2, len(Xh) // 10)))
        row = {"variant": tag, "dims": int(raw.shape[1]), "pca_dims": int(X.shape[1]), "clusters": k}
        try:
            labels = self.cluster_labels(Xh, cluster_method, cluster_param, k)
            internal = self.cluster.internal(Xh, labels)
            boot = self.cluster.bootstrap(Xh, labels, n_boot=min(n_boot, 100), sample_size=min(1200, len(Xh)), seed=self.cfg.seed)
            external = self.cluster.external(labels, self.suite.species[self.held_idx])
            row["noise_fraction"] = float((labels < 0).mean())
            row["clusters"] = int(len(np.unique(labels[labels >= 0])))
        except Exception as err:
            internal = {"silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan}
            boot = {m: (np.nan, np.nan, np.nan) for m in internal}
            external = {"nmi": np.nan}
            row["noise_fraction"] = np.nan
        row.update({"abx": abx, "abx_lo": abx_lo, "abx_hi": abx_hi,
                    "silhouette": internal["silhouette"], "silhouette_lo": boot["silhouette"][1], "silhouette_hi": boot["silhouette"][2],
                    "davies_bouldin": internal["davies_bouldin"], "davies_bouldin_lo": boot["davies_bouldin"][1], "davies_bouldin_hi": boot["davies_bouldin"][2],
                    "calinski_harabasz": internal["calinski_harabasz"], "species_nmi": external["nmi"], "effective_rank": effective_rank(X)})
        predictions = None
        if with_behaviour and len(self.lab_idx) >= 12:
            y = self.suite.behaviour[self.lab_idx]
            g = self.suite.groups[self.lab_idx]
            if len(np.unique(y)) >= 2 and min(len(np.unique(g[y == c])) for c in np.unique(y)) >= 2:
                cv = CrossValidator(self.cfg.cv_folds, 1, self.cfg.seed, True, n_boot=min(n_boot, 200))
                res = cv.run(X[self.lab_idx], y, g, probe, n_classes=len(self.cfg.behaviour_classes))
                predictions = res["predictions"]
                row.update({"behaviour_acc": res["accuracy"], "behaviour_acc_lo": res["accuracy_lo"], "behaviour_acc_hi": res["accuracy_hi"],
                            "behaviour_f1": res["macro_f1"], "behaviour_f1_lo": res["macro_f1_lo"], "behaviour_f1_hi": res["macro_f1_hi"]})
        self.details[tag] = {"hits": hits, "predictions": predictions}
        if extra:
            row.update(extra)
        return row


SCORER = SuiteScorer(SUITE, CFG)
print("triplets:", len(SCORER.triplets), "| K used for clustering scores:", SCORER.k)

## 9.2 Full-corpus embeddings with the primary encoder

The primary encoder (chosen in Part 8) now embeds **every segment** once. The result is saved to disk, so the next run of the notebook loads it instantly.

In [ ]:
def full_embeddings(primary):
    path = PATHS.embeddings / f"raw_{primary['encoder']}_L{primary['layer']}_{primary['pooling']}.npy"
    if path.exists():
        data = np.load(path)
        if len(data) == len(seg_meta):
            print(f"[reuse] {path.name} {data.shape}")
            return data
    extractor = ENCODERS.extractor(primary["encoder"], CFG)
    output = extractor.extract(store, np.arange(len(seg_meta)), layers=(primary["layer"],), poolings=(primary["pooling"],), desc="full corpus")
    data = output[(primary["layer"], primary["pooling"])]
    np.save(path, data)
    return data


with TIMER.track("full embeddings"):
    RAW_FULL = full_embeddings(PRIMARY)


def get_primary_extractor():
    return ENCODERS.extractor(PRIMARY["encoder"], CFG)


print("raw embeddings:", RAW_FULL.shape, "| effective rank:", round(effective_rank(RAW_FULL), 2))

# Part 10: Contrastive projection adapter

## 10.1 Why an adapter?

Speech and bird encoders are frozen. A small **projection adapter** on top of them can be trained without any labels so that two different views of the same 3-second segment (with noise, speed change, masking, filtering) land close together, while different segments are pushed apart. This is the SimCLR idea (NT-Xent loss) plus a small uniformity term that fights collapse.

To keep it fast, the encoder runs **once per view** and the cached features are reused for all epochs. `AugmentationBank` holds the seven audio augmentations. `ContrastiveAdapterTrainer` trains the head and reports collapse diagnostics.

In [ ]:
class AugmentationBank:
    ALL = ("gain", "noise", "shift", "timemask", "speed", "bandpass", "mixbg")

    def __init__(self, sr):
        self.sr = sr

    def gain(self, y, valid, rng):
        return y * 10 ** (rng.uniform(-6, 6) / 20)

    def noise(self, y, valid, rng):
        power = np.mean(y[:valid] ** 2) + 1e-9
        snr = 10 ** (rng.uniform(8, 28) / 10)
        return y + rng.normal(0, np.sqrt(power / snr), size=y.shape).astype(np.float32) * (np.arange(len(y)) < valid)

    def shift(self, y, valid, rng):
        out = y.copy()
        out[:valid] = np.roll(y[:valid], int(rng.integers(-self.sr // 3, self.sr // 3)))
        return out

    def timemask(self, y, valid, rng):
        out = y.copy()
        for _ in range(int(rng.integers(1, 4))):
            width = int(rng.uniform(0.03, 0.15) * self.sr)
            start = int(rng.integers(0, max(valid - width, 1)))
            out[start:start + width] = 0.0
        return out

    def speed(self, y, valid, rng):
        factor = rng.uniform(0.88, 1.12)
        new_len = max(int(valid / factor), 8)
        resampled = np.interp(np.linspace(0, valid - 1, new_len), np.arange(valid), y[:valid]).astype(np.float32)
        out = np.zeros_like(y)
        keep = min(len(resampled), len(y))
        out[:keep] = resampled[:keep]
        return out

    def bandpass(self, y, valid, rng):
        spectrum = np.fft.rfft(y[:valid])
        freqs = np.fft.rfftfreq(valid, 1.0 / self.sr)
        low, high = rng.uniform(0, 1500), rng.uniform(5500, self.sr / 2 - 100)
        spectrum[(freqs < low) | (freqs > high)] *= 0.05
        out = y.copy()
        out[:valid] = np.fft.irfft(spectrum, n=valid).astype(np.float32)
        return out

    def mixbg(self, y, valid, rng):
        noise = rng.normal(0, 1, size=y.shape).astype(np.float32)
        kernel = np.ones(int(rng.integers(3, 40))) / 20.0
        colored = np.convolve(noise, kernel, mode="same").astype(np.float32)
        level = np.std(y[:valid]) * rng.uniform(0.05, 0.4) / (np.std(colored) + 1e-9)
        return y + colored * level * (np.arange(len(y)) < valid)

    def apply(self, y, valid, ops, rng, prob=0.6):
        out = y.astype(np.float32).copy()
        order = list(ops)
        rng.shuffle(order)
        for op in order:
            if rng.random() < prob:
                out = getattr(self, op)(out, valid, rng)
        peak = np.max(np.abs(out[:valid])) + 1e-9
        return np.clip(out / peak * 0.95, -1, 1).astype(np.float32)

    def batch(self, waves, valid, ops, rng):
        return np.stack([self.apply(w, int(v), ops, rng) for w, v in zip(waves, valid)])


class ViewBuilder:
    def __init__(self, extractor, layer, pooling, bank, source, chunk=256):
        self.extractor = extractor
        self.layer = layer
        self.pooling = pooling
        self.bank = bank
        self.source = source
        self.chunk = chunk

    def build(self, positions, n_views, ops, seed):
        rng = np.random.default_rng(seed)
        positions = np.asarray(positions)
        outputs = []
        for start in tqdm(range(0, len(positions), self.chunk), desc="augmented views", leave=False):
            waves, valid = self.source.get(positions[start:start + self.chunk])
            per_view = []
            for v in range(n_views):
                w = waves if v == 0 else self.bank.batch(waves, valid, ops, rng)
                feats = self.extractor.extract(ArrayWaveSource(w, valid), np.arange(len(w)), layers=(self.layer,), poolings=(self.pooling,), progress=False)
                per_view.append(feats[(self.layer, self.pooling)])
            outputs.append(np.stack(per_view, axis=1))
        return np.concatenate(outputs, axis=0)


class ProjectionHead(nn.Module):
    def __init__(self, d_in, d_out, hidden, depth=2):
        super().__init__()
        if depth <= 1:
            self.net = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, d_out))
        elif depth == 2:
            self.net = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, hidden), nn.GELU(), nn.Linear(hidden, d_out))
        else:
            self.net = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, d_out))

    def forward(self, x):
        return self.net(x)


def nt_xent(z1, z2, temperature):
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    logits = z1 @ z2.T / temperature
    labels = torch.arange(len(z1), device=z1.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))


def uniformity(z, t=2.0):
    z = F.normalize(z, dim=1)
    if len(z) < 3:
        return z.new_tensor(0.0)
    return torch.log(torch.exp(-t * torch.pdist(z).pow(2)).mean().clamp_min(1e-12))


class ContrastiveAdapterTrainer:
    def __init__(self, d_in, cfg, depth=2, out_dim=None, hidden=None, epochs=None, temperature=None, lr=None, seed=42):
        self.cfg = cfg
        self.depth = depth
        self.out_dim = out_dim or cfg.adapter_dim
        self.hidden = hidden or cfg.adapter_hidden
        self.epochs = epochs or cfg.adapter_epochs
        self.temperature = temperature or cfg.adapter_temperature
        self.lr = lr or cfg.adapter_lr
        self.seed = seed
        self.head = ProjectionHead(d_in, self.out_dim, self.hidden, depth).to(DEVICE)
        self.mu = None
        self.sd = None
        self.history = []

    def _norm(self, x):
        return (x - self.mu) / self.sd

    def fit(self, train_views, val_views, batch_size=256, verbose=True):
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        tv = torch.tensor(train_views, dtype=torch.float32, device=DEVICE)
        vv = torch.tensor(val_views, dtype=torch.float32, device=DEVICE)
        self.mu = tv[:, 0].mean(0, keepdim=True)
        self.sd = tv[:, 0].std(0, keepdim=True).clamp_min(1e-4)
        n, n_views = tv.shape[0], tv.shape[1]
        optimizer = torch.optim.AdamW(self.head.parameters(), lr=self.lr, weight_decay=1e-3)
        steps_per_epoch = max(1, int(math.ceil(n / batch_size)))
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=self.lr, total_steps=self.epochs * steps_per_epoch)
        best, best_state, wait = float("inf"), None, 0
        for epoch in range(self.epochs):
            self.head.train()
            order = torch.randperm(n, device=DEVICE)
            losses = []
            for s in range(0, n, batch_size):
                idx = order[s:s + batch_size]
                if len(idx) < 8:
                    continue
                i, j = rng.choice(n_views, size=2, replace=False)
                z1, z2 = self.head(self._norm(tv[idx, i])), self.head(self._norm(tv[idx, j]))
                loss = nt_xent(z1, z2, self.temperature) + 0.03 * 0.5 * (uniformity(z1) + uniformity(z2))
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.head.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                losses.append(loss.item())
            self.head.eval()
            with torch.no_grad():
                a, b = self.head(self._norm(vv[:, 0])), self.head(self._norm(vv[:, min(1, n_views - 1)]))
                val = nt_xent(a[:512], b[:512], self.temperature).item()
            self.history.append({"epoch": epoch + 1, "train_loss": float(np.mean(losses)) if losses else np.nan, "val_loss": val})
            if verbose and (epoch % 5 == 0 or epoch == self.epochs - 1):
                print(f"  adapter epoch {epoch + 1:>3}/{self.epochs} train={self.history[-1]['train_loss']:.4f} val={val:.4f}")
            if val < best - 1e-4:
                best, wait = val, 0
                best_state = {k: v.detach().clone() for k, v in self.head.state_dict().items()}
            else:
                wait += 1
                if wait >= self.cfg.adapter_patience:
                    break
        if best_state is not None:
            self.head.load_state_dict(best_state)
        return pd.DataFrame(self.history)

    @torch.no_grad()
    def transform(self, X, batch=4096):
        self.head.eval()
        outputs = []
        for s in range(0, len(X), batch):
            x = torch.tensor(np.asarray(X[s:s + batch], dtype=np.float32), device=DEVICE)
            outputs.append(self.head(self._norm(x)).cpu().numpy())
        return np.concatenate(outputs, axis=0)


AUGMENTER = AugmentationBank(CFG.sample_rate)


def fresh_view_builder():
    return ViewBuilder(get_primary_extractor(), PRIMARY["layer"], PRIMARY["pooling"], AUGMENTER, store)


VIEW_BUILDER = fresh_view_builder()


def sample_split_positions(split, limit, seed):
    pool = seg_meta[seg_meta["split"] == split]["seg_pos"].to_numpy()
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(pool, size=min(limit, len(pool)), replace=False))


TRAIN_POS = sample_split_positions("train", CFG.adapter_train_segments, CFG.seed)
VAL_POS = sample_split_positions("val", CFG.adapter_val_segments, CFG.seed + 1)
with TIMER.track("adapter views"):
    TRAIN_VIEWS = CACHE.get_or_compute("adapter_train_views", lambda: fresh_view_builder().build(TRAIN_POS, CFG.adapter_views, AugmentationBank.ALL, CFG.seed))
    VAL_VIEWS = CACHE.get_or_compute("adapter_val_views", lambda: fresh_view_builder().build(VAL_POS, CFG.adapter_views, AugmentationBank.ALL, CFG.seed + 1))
print("views:", TRAIN_VIEWS.shape, VAL_VIEWS.shape)

with TIMER.track("adapter training"):
    ADAPTER = ContrastiveAdapterTrainer(RAW_FULL.shape[1], CFG, seed=CFG.seed)
    adapter_history = ADAPTER.fit(TRAIN_VIEWS, VAL_VIEWS)
ADAPTED_FULL = ADAPTER.transform(RAW_FULL)
adapter_report = diversity_report(ADAPTED_FULL[::max(1, len(ADAPTED_FULL) // 4000)], "adapted")
raw_report = diversity_report(RAW_FULL[::max(1, len(RAW_FULL) // 4000)], "raw")
ADAPTER_COLLAPSED = adapter_report.get("top1_variance", 0) > 0.85 or adapter_report["effective_rank"] < 3
print("adapter diversity:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in adapter_report.items()})
if ADAPTER_COLLAPSED:
    note_box("The adapter collapsed, so the notebook falls back to raw encoder features for the rest of the run.", "bad")
FEATURES_FULL = RAW_FULL if ADAPTER_COLLAPSED else ADAPTED_FULL
FEATURE_NAME = "raw encoder" if ADAPTER_COLLAPSED else "adapted embeddings"

## 10.2 Does the adapter help? Raw features against adapted features against MFCC

The same held-out subset, the same triplets and the same folds are used for all three feature types. Error bars are 95% bootstrap intervals. The bootstrap for the clustering scores resamples points. The bootstrap for behaviour resamples **recordings**.

In [ ]:
SUITE_MFCC = CACHE.get_or_compute("suite_mfcc", lambda: MFCCBaseline(CFG.sample_rate).extract(store, SUITE.positions))
SUITE_RAW = RAW_FULL[SUITE.positions]
SUITE_ADAPTED = ADAPTED_FULL[SUITE.positions]

with TIMER.track("adapter evaluation"):
    comparison_rows = [SCORER.evaluate(SUITE_MFCC, "MFCC baseline"), SCORER.evaluate(SUITE_RAW, "Raw encoder"), SCORER.evaluate(SUITE_ADAPTED, "Adapted")]
adapter_comparison = pd.DataFrame(comparison_rows)
TABLES.show(adapter_comparison[["variant", "abx", "abx_lo", "abx_hi", "silhouette", "davies_bouldin", "species_nmi", "behaviour_acc", "behaviour_f1", "effective_rank"]],
            "adapter_comparison", "Table 6. MFCC, raw encoder and adapted embeddings on the held-out subset", precision=3)
RESULTS["adapter_comparison"] = adapter_comparison


def bar_with_ci(ax, frame, metric, variants, colors, lo=None, hi=None, title=None, chance=None):
    values = np.array([float(frame.loc[frame["variant"] == v, metric].iloc[0]) if metric in frame else np.nan for v in variants])
    bars = ax.bar(range(len(variants)), np.nan_to_num(values), color=colors, edgecolor=Palette.INK, linewidth=1.5)
    if lo and hi and lo in frame and hi in frame:
        lows = np.array([float(frame.loc[frame["variant"] == v, lo].iloc[0]) for v in variants])
        highs = np.array([float(frame.loc[frame["variant"] == v, hi].iloc[0]) for v in variants])
        ax.errorbar(range(len(variants)), values, yerr=[np.clip(np.nan_to_num(values - lows), 0, None), np.clip(np.nan_to_num(highs - values), 0, None)], fmt="none", ecolor=Palette.INK, capsize=5, linewidth=1.8)
    annotate_bars(ax, bars, "{:.3f}", fontsize=9, offset=6)
    ax.set_xticks(range(len(variants)))
    ax.set_xticklabels(variants, rotation=18, ha="right", fontsize=9)
    if chance is not None:
        ax.axhline(chance, color=Palette.RED, linestyle="--", linewidth=2, label="chance")
    if title:
        ax.set_title(title)


def figure_adapter():
    variants = ["MFCC baseline", "Raw encoder", "Adapted"]
    colors = [Palette.GREEN, Palette.BLUE, Palette.ORANGE]
    fig, axes = VAULT.canvas(1, 4, panel=(5.2, 4.8))
    history = adapter_history
    ax = axes[0]
    ax.plot(history["epoch"], history["train_loss"], color=Palette.BLUE, marker="o", label="train")
    ax.plot(history["epoch"], history["val_loss"], color=Palette.ORANGE, marker="s", label="validation")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NT-Xent loss")
    ax.set_title("Adapter training")
    ax.legend()
    panel_label(ax, "A")
    bar_with_ci(axes[1], adapter_comparison, "abx", variants, colors, "abx_lo", "abx_hi", "ABX accuracy", chance=0.5)
    panel_label(axes[1], "B")
    bar_with_ci(axes[2], adapter_comparison, "silhouette", variants, colors, "silhouette_lo", "silhouette_hi", "Silhouette")
    panel_label(axes[2], "C")
    bar_with_ci(axes[3], adapter_comparison, "behaviour_f1", variants, colors, "behaviour_f1_lo", "behaviour_f1_hi", "Behaviour macro-F1", chance=1 / 3)
    panel_label(axes[3], "D")
    return VAULT.finish(fig, "adapter_effect", "Contrastive adapter: training curve and effect", section="Adapter")


figure_adapter()

## 10.3 Embedding maps (UMAP)

A 2-D map is not a proof, but reviewers like to see it. The same points are drawn for the raw and the adapted features, coloured by species and by weak behaviour. Grey points have no behaviour label. If UMAP is not available, the code falls back to PCA followed by t-SNE.

In [ ]:
def project_2d(X, seed=42, n_neighbors=25, min_dist=0.15):
    X = StandardScaler().fit_transform(np.nan_to_num(X))
    if X.shape[1] > 50:
        X = PCA(n_components=50, random_state=seed).fit_transform(X)
    try:
        import umap
        return umap.UMAP(n_components=2, n_neighbors=min(n_neighbors, max(2, len(X) - 1)), min_dist=min_dist, random_state=seed).fit_transform(X)
    except Exception:
        return TSNE(n_components=2, perplexity=min(30, max(5, len(X) // 4)), random_state=seed, init="pca").fit_transform(X)


def scatter_species(ax, coords, species_idx, size=9):
    for i, s in enumerate(species_names):
        m = species_idx == i
        ax.scatter(coords[m, 0], coords[m, 1], s=size, color=SPECIES_COLORS[s], edgecolors=Palette.INK, linewidths=0.25, alpha=0.85, label=s)


def scatter_behaviour(ax, coords, labels, size=9):
    grey = labels < 0
    ax.scatter(coords[grey, 0], coords[grey, 1], s=size * 0.7, color="#C9CDD2", edgecolors="none", alpha=0.5, label="unlabelled")
    for i, b in enumerate(BEHAVIOURS):
        m = labels == i
        ax.scatter(coords[m, 0], coords[m, 1], s=size * 1.3, color=Palette.BEHAVIOUR[b], edgecolors=Palette.INK, linewidths=0.3, alpha=0.9, label=b)


def figure_maps():
    limit = min(len(SUITE), 4000)
    pick = np.sort(np.random.default_rng(CFG.seed).choice(len(SUITE), limit, replace=False))
    maps = {"Raw encoder": project_2d(SUITE_RAW[pick], CFG.seed), "Adapted": project_2d(SUITE_ADAPTED[pick], CFG.seed)}
    fig, axes = VAULT.canvas(2, 2, panel=(7.0, 5.6))
    for r, (name, coords) in enumerate(maps.items()):
        scatter_species(axes[r, 0], coords, SUITE.species[pick])
        axes[r, 0].set_title(f"{name}: by species")
        scatter_behaviour(axes[r, 1], coords, SUITE.behaviour[pick])
        axes[r, 1].set_title(f"{name}: by weak behaviour")
        for c in range(2):
            axes[r, c].set_xlabel("UMAP-1")
            axes[r, c].set_ylabel("UMAP-2")
            axes[r, c].grid(False)
    axes[0, 0].legend(fontsize=8, markerscale=1.6, loc="best")
    axes[0, 1].legend(fontsize=8, markerscale=1.6, loc="best")
    VAULT.finish(fig, "embedding_maps", "Embedding maps: raw encoder versus adapter", section="Adapter")
    return maps, pick


with TIMER.track("umap maps"):
    MAP_COORDS, MAP_PICK = figure_maps()

# Part 11: Acoustic unit discovery

## 11.1 Turning embeddings into a vocabulary of sound units

Each 3-second segment becomes one **token**. The vocabulary is learned only from **training recordings**, then applied to validation and test recordings. Five methods are compared with the same scores:

| Method | Idea |
|---|---|
| K-Means | Hard clusters around K centres |
| GMM (diagonal) | Soft clusters with one variance per dimension |
| Ward hierarchical | Bottom-up merging, centres used to label new points |
| HDBSCAN | Density based, finds its own number of clusters, marks noise |
| VQ autoencoder | A small neural network with a learned codebook (a real vector-quantised model, not just K-Means) |

The choice of the vocabulary is made by a rank-average of silhouette (higher is better), Davies-Bouldin (lower is better) and Calinski-Harabasz (higher is better), computed as the **mean over five seeds**, with a penalty for unstable results. Only methods that can assign a token to every new segment are eligible for the final vocabulary. HDBSCAN is reported for comparison.

In [ ]:
class VQAutoencoder(nn.Module):
    def __init__(self, d_in, n_codes, code_dim=32, hidden=128, decay=0.99):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Linear(hidden, code_dim))
        self.decoder = nn.Sequential(nn.Linear(code_dim, hidden), nn.GELU(), nn.Linear(hidden, d_in))
        self.register_buffer("codebook", torch.randn(n_codes, code_dim) * 0.5)
        self.register_buffer("counts", torch.ones(n_codes))
        self.register_buffer("sums", self.codebook.clone())
        self.decay = decay
        self.n_codes = n_codes

    def quantize(self, z):
        distances = (z.pow(2).sum(1, keepdim=True) - 2 * z @ self.codebook.T + self.codebook.pow(2).sum(1)[None, :])
        indices = distances.argmin(1)
        return self.codebook[indices], indices

    def forward(self, x):
        z = self.encoder(x)
        q, indices = self.quantize(z)
        commit = F.mse_loss(z, q.detach())
        straight = z + (q - z).detach()
        recon = F.mse_loss(self.decoder(straight), x)
        return recon + 0.25 * commit, indices, z

    @torch.no_grad()
    def ema_update(self, z, indices):
        onehot = F.one_hot(indices, self.n_codes).float()
        self.counts.mul_(self.decay).add_(onehot.sum(0), alpha=1 - self.decay)
        self.sums.mul_(self.decay).add_(onehot.T @ z, alpha=1 - self.decay)
        total = self.counts.sum()
        smoothed = (self.counts + 1e-5) / (total + self.n_codes * 1e-5) * total
        self.codebook.copy_(self.sums / smoothed.unsqueeze(1))
        dead = self.counts < 0.02
        if dead.any() and len(z) > 0:
            pick = z[torch.randint(0, len(z), (int(dead.sum()),), device=z.device)]
            self.codebook[dead] = pick
            self.sums[dead] = pick
            self.counts[dead] = 1.0


class UnitModel:
    name = "base"

    def fit(self, X, k, seed):
        raise NotImplementedError

    def predict(self, X):
        raise NotImplementedError

    def centers(self):
        return self.centers_


class KMeansUnits(UnitModel):
    name = "kmeans"

    def fit(self, X, k, seed):
        self.model = MiniBatchKMeans(n_clusters=k, n_init=3, batch_size=2048, random_state=seed).fit(X)
        self.centers_ = self.model.cluster_centers_
        self.labels_ = self.model.labels_
        return self

    def predict(self, X):
        return self.model.predict(X)


class GMMUnits(UnitModel):
    name = "gmm_diag"

    def __init__(self, covariance="diag"):
        self.covariance = covariance
        self.name = f"gmm_{covariance}"

    def fit(self, X, k, seed):
        self.model = GaussianMixture(n_components=k, covariance_type=self.covariance, reg_covar=1e-4, n_init=1, max_iter=150, random_state=seed).fit(X)
        self.centers_ = self.model.means_
        self.labels_ = self.model.predict(X)
        return self

    def predict(self, X):
        return self.model.predict(X)


class NearestCenterMixin:
    def assign(self, X):
        d = ((X[:, None, :] - self.centers_[None, :, :]) ** 2).sum(-1) if len(X) * len(self.centers_) < 4_000_000 else None
        if d is None:
            out = []
            for s in range(0, len(X), 2000):
                block = X[s:s + 2000]
                out.append(((block[:, None, :] - self.centers_[None, :, :]) ** 2).sum(-1).argmin(1))
            return np.concatenate(out)
        return d.argmin(1)


class WardUnits(UnitModel, NearestCenterMixin):
    name = "ward"

    def fit(self, X, k, seed):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X), size=min(2500, len(X)), replace=False)
        sub = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X[idx])
        self.centers_ = np.stack([X[idx][sub == c].mean(0) for c in range(k)])
        self.labels_ = self.assign(X)
        return self

    def predict(self, X):
        return self.assign(X)


class HDBSCANUnits(UnitModel, NearestCenterMixin):
    name = "hdbscan"

    def fit(self, X, k, seed):
        size = int(k)
        if hdbscan_lib is not None:
            labels = hdbscan_lib.HDBSCAN(min_cluster_size=size, min_samples=max(2, size // 4)).fit_predict(X)
        elif SkHDBSCAN is not None:
            labels = SkHDBSCAN(min_cluster_size=size, min_samples=max(2, size // 4)).fit_predict(X)
        else:
            raise RuntimeError("no HDBSCAN implementation available")
        found = [c for c in np.unique(labels) if c >= 0]
        if len(found) < 2:
            raise RuntimeError("HDBSCAN found fewer than 2 clusters")
        self.centers_ = np.stack([X[labels == c].mean(0) for c in found])
        self.labels_ = labels
        return self

    def predict(self, X):
        return self.assign(X)


class VQUnits(UnitModel):
    name = "vq"

    def __init__(self, epochs=25):
        self.epochs = epochs

    def fit(self, X, k, seed):
        torch.manual_seed(seed)
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        self.model = VQAutoencoder(X.shape[1], k).to(DEVICE)
        with torch.no_grad():
            init = self.model.encoder(Xt[torch.randperm(len(Xt), device=DEVICE)[:k]])
            if len(init) == k:
                self.model.codebook.copy_(init)
                self.model.sums.copy_(init)
        optimizer = torch.optim.AdamW(list(self.model.encoder.parameters()) + list(self.model.decoder.parameters()), lr=2e-3)
        for _ in range(self.epochs):
            order = torch.randperm(len(Xt), device=DEVICE)
            for s in range(0, len(Xt), 256):
                batch = Xt[order[s:s + 256]]
                loss, indices, z = self.model(batch)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                self.model.ema_update(z.detach(), indices)
        self.model.eval()
        self.labels_ = self.predict(X)
        self.centers_ = np.stack([X[self.labels_ == c].mean(0) if (self.labels_ == c).any() else np.zeros(X.shape[1]) for c in range(k)])
        return self

    @torch.no_grad()
    def predict(self, X):
        out = []
        for s in range(0, len(X), 4096):
            z = self.model.encoder(torch.tensor(X[s:s + 4096], dtype=torch.float32, device=DEVICE))
            out.append(self.model.quantize(z)[1].cpu().numpy())
        return np.concatenate(out)


def make_units(method, epochs=25):
    if method == "kmeans":
        return KMeansUnits()
    if method == "gmm_diag":
        return GMMUnits("diag")
    if method == "gmm_full":
        return GMMUnits("full")
    if method == "ward":
        return WardUnits()
    if method == "hdbscan":
        return HDBSCANUnits()
    if method == "vq":
        return VQUnits(epochs)
    raise ValueError(method)


class UnitSweep:
    def __init__(self, cfg, X_fit, species_fit):
        self.cfg = cfg
        rng = np.random.default_rng(cfg.seed)
        size = min(cfg.cluster_eval_points, len(X_fit))
        self.pick = np.sort(rng.choice(len(X_fit), size=size, replace=False))
        self.X = X_fit[self.pick]
        self.species = species_fit[self.pick]
        self.scorer = ClusterScorer(cfg.silhouette_points, cfg.seed)

    def run(self, methods, k_grid=None, seeds=None):
        rows, labels_by = [], {}
        k_grid = k_grid or self.cfg.k_grid
        seeds = seeds or self.cfg.cluster_seeds
        for method in methods:
            grid = self.hdbscan_grid() if method == "hdbscan" else k_grid
            for k in grid:
                if method != "hdbscan" and k >= len(self.X) // 3:
                    continue
                for seed in seeds if method != "hdbscan" else seeds[:1]:
                    try:
                        model = make_units(method, self.cfg.vq_epochs).fit(self.X, int(k), seed)
                    except Exception as err:
                        rows.append({"method": method, "param": k, "seed": seed, "error": str(err)[:80]})
                        continue
                    labels = model.labels_ if method == "hdbscan" else model.predict(self.X)
                    internal = self.scorer.internal(self.X, labels)
                    external = self.scorer.external(labels, self.species)
                    used = int(len(np.unique(labels[labels >= 0])))
                    rows.append({"method": method, "param": int(k), "seed": seed, "n_units": used, "noise_fraction": float((labels < 0).mean()),
                                 **internal, **external})
                    labels_by[(method, int(k), seed)] = labels
        return pd.DataFrame(rows), labels_by

    def hdbscan_grid(self):
        base = max(10, len(self.X) // 300)
        return [base, base * 2, base * 4]

    @staticmethod
    def stability(labels_by, method, k):
        keys = [key for key in labels_by if key[0] == method and key[1] == k]
        if len(keys) < 2:
            return np.nan
        scores = [adjusted_rand_score(labels_by[a], labels_by[b]) for a, b in itertools.combinations(keys, 2)]
        return float(np.mean(scores))

    @staticmethod
    def summarise(frame, labels_by):
        ok = frame[frame.get("silhouette").notna()] if "silhouette" in frame else frame.iloc[0:0]
        grouped = ok.groupby(["method", "param"]).agg(units=("n_units", "mean"), silhouette=("silhouette", "mean"), silhouette_sd=("silhouette", "std"),
                                                       davies_bouldin=("davies_bouldin", "mean"), calinski_harabasz=("calinski_harabasz", "mean"),
                                                       species_nmi=("nmi", "mean"), noise_fraction=("noise_fraction", "mean")).reset_index()
        grouped["stability_ari"] = [UnitSweep.stability(labels_by, m, k) for m, k in zip(grouped["method"], grouped["param"])]
        return grouped


TRAIN_MASK = (seg_meta["split"] == "train").to_numpy()
TOKEN_SPACE = EmbeddingSpace.from_config(CFG).fit(FEATURES_FULL[TRAIN_MASK])
Z_ALL = TOKEN_SPACE.transform(FEATURES_FULL)
print("token space:", Z_ALL.shape, "| PCA components:", Z_ALL.shape[1])

METHODS_SWEEP = ["kmeans", "gmm_diag", "ward", "vq"] + (["hdbscan"] if (hdbscan_lib is not None or SkHDBSCAN is not None) else [])
sweeper = UnitSweep(CFG, Z_ALL[TRAIN_MASK], seg_meta.loc[TRAIN_MASK, "species_index"].to_numpy())
with TIMER.track("unit sweep"):
    sweep_frame, sweep_labels = CACHE.get_or_compute("unit_sweep", lambda: sweeper.run(METHODS_SWEEP))
sweep_summary = UnitSweep.summarise(sweep_frame, sweep_labels)
print(sweep_summary.groupby("method").size().to_dict())

## 11.2 Choosing the vocabulary

In [ ]:
def select_vocabulary(summary, min_units):
    pool = summary[(summary["method"] != "hdbscan") & (summary["units"] >= min_units)].copy()
    if pool.empty:
        pool = summary[summary["method"] != "hdbscan"].copy()
    pool["r_sil"] = pool["silhouette"].rank(pct=True)
    pool["r_dbi"] = (-pool["davies_bouldin"]).rank(pct=True)
    pool["r_ch"] = pool["calinski_harabasz"].rank(pct=True)
    pool["r_stab"] = pool["stability_ari"].rank(pct=True).fillna(0.5)
    pool["selection_score"] = 0.3 * pool["r_sil"] + 0.2 * pool["r_dbi"] + 0.2 * pool["r_ch"] + 0.3 * pool["r_stab"]
    return pool.sort_values("selection_score", ascending=False).reset_index(drop=True)


ranked_vocab = select_vocabulary(sweep_summary, CFG.min_clusters)
VOCAB = {"method": str(ranked_vocab.loc[0, "method"]), "k": int(ranked_vocab.loc[0, "param"])}
print("selected vocabulary:", VOCAB)
TABLES.show(ranked_vocab[["method", "param", "units", "silhouette", "silhouette_sd", "davies_bouldin", "calinski_harabasz", "species_nmi", "stability_ari", "selection_score"]].head(12),
            "vocabulary_selection", "Table 7. Vocabulary candidates ranked by the selection score (means over seeds)", precision=3)

with TIMER.track("final vocabulary"):
    FINAL_UNITS = CACHE.get_or_compute("final_units", lambda: make_units(VOCAB["method"], CFG.vq_epochs).fit(Z_ALL[TRAIN_MASK], VOCAB["k"], CFG.seed))
TOKENS = FINAL_UNITS.predict(Z_ALL).astype(int) + 1
VOCAB_SIZE = int(VOCAB["k"]) + 1
seg_meta["token"] = TOKENS
print(f"tokens assigned to {len(TOKENS)} segments | vocabulary size (with pad) = {VOCAB_SIZE}")


def figure_sweep():
    fig, axes = VAULT.canvas(1, 4, panel=(5.4, 4.6))
    metrics = [("silhouette", "Silhouette (higher is better)"), ("davies_bouldin", "Davies-Bouldin (lower is better)"),
               ("calinski_harabasz", "Calinski-Harabasz (higher is better)"), ("stability_ari", "Seed stability (ARI)")]
    colors = {"kmeans": Palette.BLUE, "gmm_diag": Palette.ORANGE, "ward": Palette.GREEN, "vq": Palette.PURPLE}
    for ax, (col, title) in zip(axes, metrics):
        for method, color in colors.items():
            part = sweep_summary[sweep_summary["method"] == method].sort_values("param")
            if part.empty:
                continue
            ax.plot(part["param"], part[col], marker="o", color=color, label=method)
            if col == "silhouette":
                ax.fill_between(part["param"], part[col] - part["silhouette_sd"].fillna(0), part[col] + part["silhouette_sd"].fillna(0), color=color, alpha=0.16)
        ax.axvline(VOCAB["k"], color=Palette.RED, linestyle="--", linewidth=2)
        ax.set_xlabel("Number of units (K)")
        ax.set_title(title, fontsize=11)
    axes[0].legend(fontsize=8)
    return VAULT.finish(fig, "k_sweep", "Vocabulary size sweep (mean over seeds, red line = selected K)", section="Units")


def figure_method_comparison():
    at_k = sweep_summary.loc[(sweep_summary["param"] == VOCAB["k"]) | (sweep_summary["method"] == "hdbscan")]
    at_k = at_k.sort_values("method").drop_duplicates("method")
    fig, axes = VAULT.canvas(1, 4, panel=(5.2, 4.6))
    specs = [("silhouette", "Silhouette"), ("davies_bouldin", "Davies-Bouldin"), ("species_nmi", "NMI with species"), ("stability_ari", "Seed stability (ARI)")]
    palette = {"kmeans": Palette.BLUE, "gmm_diag": Palette.ORANGE, "ward": Palette.GREEN, "vq": Palette.PURPLE, "hdbscan": Palette.RED}
    for ax, (col, title) in zip(axes, specs):
        bars = ax.bar(at_k["method"], np.nan_to_num(at_k[col]), color=[palette.get(m, Palette.BLUE) for m in at_k["method"]], edgecolor=Palette.INK, linewidth=1.4)
        annotate_bars(ax, bars, "{:.3f}", fontsize=8.5)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=25)
    return VAULT.finish(fig, "method_comparison", f"Unit discovery methods at K = {VOCAB['k']} (HDBSCAN finds its own K)", section="Units")


def figure_token_diagnostics():
    fig, axes = VAULT.canvas(2, 2, panel=(7.0, 4.8))
    counts = np.bincount(TOKENS, minlength=VOCAB_SIZE)[1:]
    order = np.sort(counts)[::-1]
    ax = axes[0, 0]
    ax.loglog(np.arange(1, len(order) + 1), np.maximum(order, 1), marker="o", color=Palette.ORANGE)
    ranks = np.arange(1, len(order) + 1)
    slope, intercept = np.polyfit(np.log(ranks), np.log(np.maximum(order, 1)), 1)
    ax.loglog(ranks, np.exp(intercept) * ranks ** slope, color=Palette.BLUE, linestyle="--", label=f"power law slope = {slope:.2f}")
    ax.set_xlabel("Token rank")
    ax.set_ylabel("Segments")
    ax.set_title("Rank-frequency (Zipf) plot")
    ax.legend()
    panel_label(ax, "A")
    ax = axes[0, 1]
    for split, color in zip(["train", "val", "test"], [Palette.BLUE, Palette.ORANGE, Palette.GREEN]):
        c = np.bincount(TOKENS[(seg_meta["split"] == split).to_numpy()], minlength=VOCAB_SIZE)[1:].astype(float)
        ax.plot(np.arange(1, len(c) + 1), c / max(c.sum(), 1), marker="o", markersize=4, color=color, label=split)
    ax.set_xlabel("Token id")
    ax.set_ylabel("Share of segments")
    ax.set_title("Token usage by split")
    ax.legend()
    panel_label(ax, "B")
    ax = axes[1, 0]
    usage = pd.crosstab(seg_meta["species"], seg_meta["token"], normalize="index").reindex(species_names).fillna(0)
    sns.heatmap(usage, cmap=Palette.blue_to_orange(), ax=ax, linewidths=0.4, linecolor="#333333", cbar_kws={"label": "share within species"})
    ax.set_xlabel("Token id")
    ax.set_ylabel("")
    ax.set_yticklabels([s.split(" ")[0] for s in usage.index], rotation=0)
    ax.set_title("Token usage by species")
    ax.grid(False)
    panel_label(ax, "C")
    ax = axes[1, 1]
    entropies = []
    for s in species_names:
        p = np.bincount(TOKENS[(seg_meta["species"] == s).to_numpy()], minlength=VOCAB_SIZE)[1:].astype(float)
        p = p / max(p.sum(), 1)
        p = p[p > 0]
        entropies.append(float(-(p * np.log2(p)).sum() / np.log2(VOCAB["k"])))
    bars = ax.bar([s.split(" ")[0] for s in species_names], entropies, color=[SPECIES_COLORS[s] for s in species_names], edgecolor=Palette.INK, linewidth=1.4)
    annotate_bars(ax, bars, "{:.2f}")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Normalised entropy")
    ax.set_title("Vocabulary use per species")
    panel_label(ax, "D")
    VAULT.finish(fig, "token_diagnostics", "Token statistics", section="Units")
    return {"zipf_slope": float(slope), "usage_entropy": dict(zip(species_names, entropies))}


def figure_knife_and_dendrogram():
    sample = np.sort(np.random.default_rng(CFG.seed).choice(len(Z_ALL), size=min(3000, len(Z_ALL)), replace=False))
    labels = TOKENS[sample] - 1
    fig, axes = VAULT.canvas(1, 2, panel=(8.0, 6.0))
    ax = axes[0]
    if len(np.unique(labels)) > 1:
        values = silhouette_samples(Z_ALL[sample], labels)
        colors = Palette.token_colors(VOCAB["k"])
        y0 = 0
        for c in np.argsort([-values[labels == c].mean() if (labels == c).any() else 0 for c in range(VOCAB["k"])]):
            v = np.sort(values[labels == c])
            if len(v) == 0:
                continue
            ax.fill_betweenx(np.arange(y0, y0 + len(v)), 0, v, color=colors[c], edgecolor=Palette.INK, linewidth=0.3)
            y0 += len(v) + max(2, len(sample) // 200)
        ax.axvline(values.mean(), color=Palette.RED, linestyle="--", linewidth=2.2, label=f"mean = {values.mean():.3f}")
        ax.set_xlabel("Silhouette value")
        ax.set_yticks([])
        ax.set_title("Silhouette profile per token")
        ax.legend()
    ax = axes[1]
    centers = FINAL_UNITS.centers()
    if len(centers) > 2:
        link = sch.linkage(centers, method="ward")
        sch.dendrogram(link, ax=ax, labels=[str(i + 1) for i in range(len(centers))], color_threshold=0.6 * link[:, 2].max(), above_threshold_color=Palette.INK)
        for line in ax.collections:
            line.set_linewidth(2.2)
        ax.set_xlabel("Token id")
        ax.set_ylabel("Ward distance")
        ax.set_title("How tokens group together")
    return VAULT.finish(fig, "knife_dendrogram", "Cluster quality and token hierarchy", section="Units")


def figure_stability_heatmap():
    keys = sorted([k for k in sweep_labels if k[0] == VOCAB["method"] and k[1] == VOCAB["k"]], key=lambda x: x[2])
    if len(keys) < 2:
        return None
    n = len(keys)
    matrix = np.ones((n, n))
    for i, j in itertools.combinations(range(n), 2):
        matrix[i, j] = matrix[j, i] = adjusted_rand_score(sweep_labels[keys[i]], sweep_labels[keys[j]])
    fig, ax = VAULT.canvas(1, 1, figsize=(7.5, 6.2))
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap=Palette.blues(), vmin=0, vmax=1, ax=ax, linewidths=1.2, linecolor=Palette.INK, cbar_kws={"label": "ARI"},
                xticklabels=[f"seed {k[2]}" for k in keys], yticklabels=[f"seed {k[2]}" for k in keys])
    ax.grid(False)
    return VAULT.finish(fig, "stability_heatmap", f"Seed-to-seed agreement ({VOCAB['method']}, K = {VOCAB['k']})", section="Units")


figure_sweep()
figure_method_comparison()
TOKEN_STATS = figure_token_diagnostics()
figure_knife_and_dendrogram()
figure_stability_heatmap()
RESULTS["vocab"] = {**VOCAB, **TOKEN_STATS}

# Part 12: Sequence language model

## 12.1 Do acoustic units follow rules?

Each recording is now a **sentence of tokens** (one token per 3-second segment, in time order). A small Transformer language model reads the tokens from left to right and predicts the next one. **Perplexity** measures how surprised the model is: a perfect guess gives 1, a blind guess gives the vocabulary size.

Perplexity alone can fool us, because neighbouring segments of one call often get the same token, which makes prediction easy without any real "grammar". So five controls are reported next to the Transformer:

| Model | Question it answers |
|---|---|
| Uniform | What if we guessed blindly? |
| Unigram | Does knowing token frequencies help? |
| Bigram and trigram | How far can simple count models go? |
| Transformer on **shuffled** order | Is the gain coming from temporal order at all? |
| Transformer on **run-collapsed** sequences (repeats removed) | Is the gain more than "the same token repeats"? |

Training uses train recordings, early stopping uses validation recordings, and the final numbers come from **test recordings**. Confidence intervals resample recordings.

In [ ]:
class SequenceBuilder:
    def __init__(self, seg_meta, tokens, max_len):
        frame = seg_meta[["recording_id", "segment_index", "split", "species_index", "behaviour_strict"]].copy()
        frame["token"] = tokens
        self.frame = frame.sort_values(["recording_id", "segment_index"])
        self.max_len = max_len

    def sequences(self, split=None):
        out = {}
        for rid, group in self.frame.groupby("recording_id", sort=False):
            if split is not None and group["split"].iloc[0] != split:
                continue
            out[rid] = group["token"].to_numpy()[: self.max_len]
        return out

    @staticmethod
    def shuffled(seqs, seed):
        rng = np.random.default_rng(seed)
        return {k: rng.permutation(v) for k, v in seqs.items()}

    @staticmethod
    def run_collapsed(seqs):
        out = {}
        for k, v in seqs.items():
            keep = np.concatenate([[True], v[1:] != v[:-1]]) if len(v) else np.zeros(0, dtype=bool)
            out[k] = v[keep]
        return out

    @staticmethod
    def usable(seqs, min_len=2):
        return {k: v for k, v in seqs.items() if len(v) >= min_len}


class BirdLanguageModel(nn.Module):
    def __init__(self, n_tokens, d_model, heads, layers, ff, dropout, max_len):
        super().__init__()
        self.n_tokens = n_tokens
        self.bos = n_tokens + 1
        self.embed = nn.Embedding(n_tokens + 2, d_model, padding_idx=0)
        self.position = nn.Embedding(max_len + 2, d_model)
        block = nn.TransformerEncoderLayer(d_model, heads, ff, dropout, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_tokens)

    def hidden(self, x):
        steps = x.shape[1]
        positions = torch.arange(steps, device=x.device)[None, :]
        h = self.embed(x) + self.position(positions)
        causal = torch.triu(torch.ones(steps, steps, dtype=torch.bool, device=x.device), diagonal=1)
        h = self.encoder(h, mask=causal, src_key_padding_mask=(x == 0))
        return self.norm(h)

    def forward(self, x):
        return self.head(self.hidden(x))


def pack_sequences(seqs, bos, max_len):
    keys = list(seqs)
    x = np.zeros((len(keys), max_len + 1), dtype=np.int64)
    for i, k in enumerate(keys):
        v = seqs[k][:max_len]
        x[i, 0] = bos
        x[i, 1:1 + len(v)] = v
    return keys, torch.tensor(x)


class LMTrainer:
    def __init__(self, cfg, n_tokens, max_len, d_model=None, heads=None, layers=None, ff=None, epochs=None):
        self.cfg = cfg
        self.n_tokens = n_tokens
        self.max_len = max_len
        self.d_model = d_model or cfg.lm_d_model
        self.heads = heads or cfg.lm_heads
        self.layers = layers or cfg.lm_layers
        self.ff = ff or cfg.lm_ff
        self.epochs = epochs or cfg.lm_epochs

    def build(self):
        d = self.d_model - (self.d_model % self.heads)
        return BirdLanguageModel(self.n_tokens, d, self.heads, self.layers, self.ff, self.cfg.lm_dropout, self.max_len).to(DEVICE)

    def _loss(self, model, x, reduce=True):
        logits = model(x[:, :-1])
        target = x[:, 1:]
        mask = target > 0
        nll = F.cross_entropy(logits.reshape(-1, self.n_tokens), (target - 1).clamp_min(0).reshape(-1), reduction="none").reshape(target.shape)
        nll = nll * mask
        return (nll.sum() / mask.sum().clamp_min(1)) if reduce else (nll, mask)

    def fit(self, train_seqs, val_seqs, seed):
        torch.manual_seed(seed)
        model = self.build()
        _, xt = pack_sequences(train_seqs, model.bos, self.max_len)
        _, xv = pack_sequences(val_seqs, model.bos, self.max_len)
        xt, xv = xt.to(DEVICE), xv.to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=self.cfg.lm_lr, weight_decay=0.01)
        steps = max(1, int(math.ceil(len(xt) / self.cfg.lm_batch)))
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=self.cfg.lm_lr, total_steps=self.epochs * steps)
        history, best, best_state, wait = [], float("inf"), None, 0
        for epoch in range(self.epochs):
            model.train()
            order = torch.randperm(len(xt), device=DEVICE)
            losses = []
            for s in range(0, len(xt), self.cfg.lm_batch):
                batch = xt[order[s:s + self.cfg.lm_batch]]
                loss = self._loss(model, batch)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                losses.append(loss.item())
            model.eval()
            with torch.no_grad():
                val_loss = self._loss(model, xv).item() if len(xv) else float("nan")
            history.append({"epoch": epoch + 1, "train_loss": float(np.mean(losses)), "val_loss": val_loss, "train_ppl": math.exp(np.mean(losses)), "val_ppl": math.exp(val_loss) if val_loss == val_loss else np.nan})
            if val_loss < best - 1e-4:
                best, wait = val_loss, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= self.cfg.lm_patience:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        return model, pd.DataFrame(history)

    @torch.no_grad()
    def per_recording_nll(self, model, seqs):
        keys, x = pack_sequences(seqs, model.bos, self.max_len)
        model.eval()
        x = x.to(DEVICE)
        nll, mask = self._loss(model, x, reduce=False)
        return keys, nll.cpu().numpy(), mask.cpu().numpy()


class NGramLM:
    def __init__(self, order, n_tokens, alpha=0.1):
        self.order = order
        self.n_tokens = n_tokens
        self.alpha = alpha
        self.bos = n_tokens + 1
        self.counts = defaultdict(Counter)
        self.totals = Counter()

    def _contexts(self, seq):
        padded = [self.bos] * (self.order - 1) + list(seq)
        for i in range(self.order - 1, len(padded)):
            yield tuple(padded[i - self.order + 1:i]), padded[i]

    def fit(self, seqs):
        for seq in seqs.values():
            for context, token in self._contexts(seq):
                self.counts[context][token] += 1
                self.totals[context] += 1
        return self

    def per_recording_nll(self, seqs):
        keys, out = [], []
        for k, seq in seqs.items():
            values = []
            for context, token in self._contexts(seq):
                p = (self.counts[context][token] + self.alpha) / (self.totals[context] + self.alpha * self.n_tokens)
                values.append(-math.log(p))
            keys.append(k)
            out.append(np.asarray(values))
        return keys, out


def bootstrap_perplexity(nll_sums, counts, n_boot=500, seed=0):
    nll_sums, counts = np.asarray(nll_sums, dtype=float), np.asarray(counts, dtype=float)
    point = math.exp(nll_sums.sum() / max(counts.sum(), 1))
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(nll_sums), len(nll_sums))
        draws.append(math.exp(nll_sums[pick].sum() / max(counts[pick].sum(), 1)))
    return point, float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))


BUILDER = SequenceBuilder(seg_meta, TOKENS, CFG.max_segments_per_recording)
SEQ = {s: SequenceBuilder.usable(BUILDER.sequences(s)) for s in ["train", "val", "test"]}
print({s: len(v) for s, v in SEQ.items()}, "| mean length:", round(float(np.mean([len(v) for v in SEQ["train"].values()])), 1))

## 12.2 Train the language model and all controls

In [ ]:
class LMExperiment:
    def __init__(self, cfg, n_tokens, max_len):
        self.cfg = cfg
        self.n_tokens = n_tokens
        self.max_len = max_len

    def run_transformer(self, train, val, test, label, seeds, **kwargs):
        trainer = LMTrainer(self.cfg, self.n_tokens, self.max_len, **kwargs)
        rows, histories, models = [], [], []
        for seed in seeds:
            model, history = trainer.fit(train, val, seed)
            keys, nll, mask = trainer.per_recording_nll(model, test)
            point, lo, hi = bootstrap_perplexity(nll.sum(1), mask.sum(1), self.cfg.bootstrap_iterations, seed)
            _, nll_v, mask_v = trainer.per_recording_nll(model, val)
            rows.append({"model": label, "seed": seed, "test_ppl": point, "test_ppl_lo": lo, "test_ppl_hi": hi,
                         "val_ppl": math.exp(nll_v.sum() / max(mask_v.sum(), 1)), "epochs_run": len(history), "parameters": sum(p.numel() for p in model.parameters())})
            histories.append(history.assign(seed=seed, model=label))
            models.append(model)
        return rows, histories, models, trainer

    def run_ngram(self, order, train, val, test, label):
        model = NGramLM(order, self.n_tokens).fit(train)
        keys, per = model.per_recording_nll(test)
        sums = np.array([p.sum() for p in per])
        counts = np.array([len(p) for p in per])
        point, lo, hi = bootstrap_perplexity(sums, counts, self.cfg.bootstrap_iterations, self.cfg.seed)
        keys_v, per_v = model.per_recording_nll(val)
        val_ppl = math.exp(sum(p.sum() for p in per_v) / max(sum(len(p) for p in per_v), 1))
        return {"model": label, "seed": -1, "test_ppl": point, "test_ppl_lo": lo, "test_ppl_hi": hi, "val_ppl": val_ppl, "epochs_run": 0, "parameters": 0}, per


K_TOKENS = VOCAB["k"]
LMX = LMExperiment(CFG, K_TOKENS, CFG.max_segments_per_recording)
lm_rows, lm_histories = [], []
with TIMER.track("language model"):
    rows, histories, LM_MODELS, LM_TRAINER = LMX.run_transformer(SEQ["train"], SEQ["val"], SEQ["test"], "Transformer (real order)", CFG.lm_seeds)
    lm_rows += rows
    lm_histories += histories
    total_test_tokens = sum(len(v) for v in SEQ["test"].values())
    uniform = {"model": "Uniform guess", "seed": -1, "test_ppl": float(K_TOKENS), "test_ppl_lo": float(K_TOKENS), "test_ppl_hi": float(K_TOKENS), "val_ppl": float(K_TOKENS), "epochs_run": 0, "parameters": 0}
    lm_rows.append(uniform)
    NGRAM_PER = {}
    for order, label in [(1, "Unigram"), (2, "Bigram"), (3, "Trigram")]:
        row, per = LMX.run_ngram(order, SEQ["train"], SEQ["val"], SEQ["test"], label)
        lm_rows.append(row)
        NGRAM_PER[label] = per
    shuffled = {s: SequenceBuilder.shuffled(SEQ[s], CFG.seed + i) for i, s in enumerate(["train", "val", "test"])}
    rows, histories, LM_SHUFFLED_MODELS, _ = LMX.run_transformer(shuffled["train"], shuffled["val"], shuffled["test"], "Transformer (shuffled order)", CFG.lm_seeds)
    lm_rows += rows
    lm_histories += histories
    row, _ = LMX.run_ngram(2, shuffled["train"], shuffled["val"], shuffled["test"], "Bigram (shuffled order)")
    lm_rows.append(row)
    collapsed = {s: SequenceBuilder.usable(SequenceBuilder.run_collapsed(SEQ[s])) for s in ["train", "val", "test"]}
    if all(len(v) > 3 for v in collapsed.values()):
        rows, histories, _, _ = LMX.run_transformer(collapsed["train"], collapsed["val"], collapsed["test"], "Transformer (repeats removed)", CFG.lm_seeds)
        lm_rows += rows
        lm_histories += histories
        row, _ = LMX.run_ngram(1, collapsed["train"], collapsed["val"], collapsed["test"], "Unigram (repeats removed)")
        lm_rows.append(row)
        row, _ = LMX.run_ngram(2, collapsed["train"], collapsed["val"], collapsed["test"], "Bigram (repeats removed)")
        lm_rows.append(row)

lm_frame = pd.DataFrame(lm_rows)
lm_histories = pd.concat(lm_histories, ignore_index=True)
lm_table = (lm_frame.groupby("model", sort=False).agg(test_ppl=("test_ppl", "mean"), ppl_lo=("test_ppl_lo", "mean"), ppl_hi=("test_ppl_hi", "mean"),
                                                        val_ppl=("val_ppl", "mean"), seeds=("seed", "count"), parameters=("parameters", "max")).reset_index())
TABLES.show(lm_table, "language_model", "Table 8. Next-token perplexity on test recordings (95% bootstrap over recordings; lower is better)", precision=2)
RESULTS["lm"] = lm_table

## 12.3 Language model figures

In [ ]:
def figure_lm():
    fig, axes = VAULT.canvas(1, 3, panel=(7.0, 5.0))
    ax = axes[0]
    for label, color in [("Transformer (real order)", Palette.BLUE), ("Transformer (shuffled order)", Palette.ORANGE), ("Transformer (repeats removed)", Palette.GREEN)]:
        part = lm_histories[lm_histories["model"] == label]
        if part.empty:
            continue
        stats = part.groupby("epoch")[["train_ppl", "val_ppl"]].agg(["mean", "std"]).fillna(0)
        ax.plot(stats.index, stats[("val_ppl", "mean")], color=color, label=f"{label} (val)")
        ax.fill_between(stats.index, stats[("val_ppl", "mean")] - stats[("val_ppl", "std")], stats[("val_ppl", "mean")] + stats[("val_ppl", "std")], color=color, alpha=0.18)
        ax.plot(stats.index, stats[("train_ppl", "mean")], color=color, linestyle="--", alpha=0.7)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Perplexity")
    ax.set_title("Learning curves (solid = validation, dashed = train)")
    ax.legend(fontsize=7.5)
    panel_label(ax, "A")

    ax = axes[1]
    table = lm_table.copy()
    order = table.sort_values("test_ppl", ascending=False)
    colors = []
    for m in order["model"]:
        if "Transformer" in m and "real" in m:
            colors.append(Palette.BLUE)
        elif "Transformer" in m:
            colors.append(Palette.ORANGE)
        elif "Uniform" in m:
            colors.append(Palette.RED)
        else:
            colors.append(Palette.PURPLE if "repeats" in m or "shuffled" in m else Palette.GREEN)
    bars = ax.barh(order["model"], order["test_ppl"], color=colors, edgecolor=Palette.INK, linewidth=1.4)
    ax.errorbar(order["test_ppl"], range(len(order)), xerr=[np.clip(order["test_ppl"] - order["ppl_lo"], 0, None), np.clip(order["ppl_hi"] - order["test_ppl"], 0, None)], fmt="none", ecolor=Palette.INK, capsize=4)
    annotate_bars(ax, bars, "{:.2f}", horizontal=True, fontsize=8.5, offset=12)
    ax.set_xlabel("Test perplexity (lower is better)")
    ax.set_title("Transformer against controls")
    ax.tick_params(axis="y", labelsize=8)
    panel_label(ax, "B")

    ax = axes[2]
    trainer = LM_TRAINER
    def by_position(model, seqs):
        _, nll, mask = trainer.per_recording_nll(model, seqs)
        L = nll.shape[1]
        return np.array([nll[:, j][mask[:, j]].mean() if mask[:, j].any() else np.nan for j in range(L)])
    real = np.nanmean([by_position(m, SEQ["test"]) for m in LM_MODELS], axis=0)
    shuf = np.nanmean([by_position(m, shuffled["test"]) for m in LM_SHUFFLED_MODELS], axis=0)
    x = np.arange(1, len(real) + 1)
    ax.plot(x, real, color=Palette.BLUE, marker="o", label="real order")
    ax.plot(x, shuf, color=Palette.ORANGE, marker="s", label="shuffled order")
    ax.axhline(math.log(K_TOKENS), color=Palette.RED, linestyle="--", linewidth=2, label="uniform guess")
    ax.set_xlabel("Position in recording")
    ax.set_ylabel("Mean negative log-likelihood (nats)")
    ax.set_title("Context helps more later in a recording")
    ax.legend()
    panel_label(ax, "C")
    return VAULT.finish(fig, "language_model", "Sequence language model with controls", section="Language model")


figure_lm()

real_ppl = lm_frame.loc[lm_frame["model"] == "Transformer (real order)", "test_ppl"]
shuf_ppl = lm_frame.loc[lm_frame["model"] == "Transformer (shuffled order)", "test_ppl"]
if len(real_ppl) > 1 and len(shuf_ppl) > 1:
    order_effect = float(shuf_ppl.mean() - real_ppl.mean())
    t_stat, p_val = sps.ttest_rel(shuf_ppl.to_numpy()[:len(real_ppl)], real_ppl.to_numpy()[:len(shuf_ppl)])
    RESULTS["lm_order_effect"] = {"ppl_gain_from_order": order_effect, "paired_t_p": float(p_val)}
    note_box(f"Perplexity gain from real temporal order: <b>{order_effect:.3f}</b> (paired t-test over seeds, p = {p_val:.3g}). Shuffled training data is the control for 'no temporal structure'.",
             "good" if order_effect > 0 and p_val < 0.05 else "warn")

# Part 13: What do the tokens mean?

## 13.1 Acoustic unit gallery, transitions and sequences

Discovering tokens is only useful if we can look at them. Four views are shown:

1. **Unit gallery**: the average mel spectrogram of the 15 segments closest to the centre of each of the most used tokens.
2. **Transition matrix and network**: how often token *i* is followed by token *j* inside a recording. Bright diagonal cells mean a token tends to repeat.
3. **Sequence barcodes**: recordings drawn as coloured strips, grouped by weak behaviour.
4. **Behaviour association**: which tokens are more common in alarm, song or contact recordings, tested with a **recording-level permutation test** (labels are shuffled between recordings, not between segments, to avoid fake significance).

In [ ]:
def figure_unit_gallery(n_show=12):
    counts = np.bincount(TOKENS, minlength=VOCAB_SIZE)
    top_tokens = [t for t in np.argsort(-counts) if t > 0][:n_show]
    centers = FINAL_UNITS.centers()
    cols = 4
    rows = int(math.ceil(len(top_tokens) / cols))
    fig = plt.figure(figsize=(4.6 * cols, 3.3 * rows))
    grid = GridSpec(rows, cols + 1, figure=fig, width_ratios=[1] * cols + [0.06], wspace=0.35, hspace=0.5)
    last_image = None
    for idx, token in enumerate(top_tokens):
        r, c = divmod(idx, cols)
        ax = fig.add_subplot(grid[r, c])
        members = np.where(TOKENS == token)[0]
        dist = ((Z_ALL[members] - centers[token - 1]) ** 2).sum(1)
        chosen = members[np.argsort(dist)[:15]]
        waves, valid = store.get(chosen)
        mels = []
        for w, v in zip(waves, valid):
            m = librosa.feature.melspectrogram(y=w[:int(v)], sr=CFG.sample_rate, n_fft=1024, hop_length=256, n_mels=64, fmin=500, fmax=CFG.sample_rate / 2)
            width = 180
            if m.shape[1] < width:
                m = np.pad(m, ((0, 0), (0, width - m.shape[1])), constant_values=1e-10)
            mels.append(m[:, :width])
        avg = librosa.power_to_db(np.mean(mels, axis=0), ref=np.max)
        last_image = ax.imshow(avg, origin="lower", aspect="auto", cmap=Palette.spectrogram(), vmin=-60, vmax=0, extent=[0, 3, 500, CFG.sample_rate / 2])
        ax.set_title(f"token {token}  (n = {counts[token]})", fontsize=10.5)
        ax.set_xticks([0, 1.5, 3])
        ax.grid(False)
    for idx in range(len(top_tokens), rows * cols):
        r, c = divmod(idx, cols)
        fig.add_subplot(grid[r, c]).axis("off")
    if last_image is not None:
        cax = fig.add_subplot(grid[:, cols])
        cbar = fig.colorbar(last_image, cax=cax)
        cbar.set_label("dB")
        cbar.outline.set_edgecolor(Palette.INK)
    fig.supxlabel("Time (s)", fontweight="bold", color=Palette.BLUE_DARK)
    fig.supylabel("Frequency (Hz)", fontweight="bold", color=Palette.BLUE_DARK)
    return VAULT.finish(fig, "unit_gallery", "Average spectrogram of the most typical segments of each token", section="Interpretation")

def transition_counts(seqs_by_split):
    K = VOCAB["k"]
    counts = np.zeros((K, K))
    for seqs in seqs_by_split:
        for v in seqs.values():
            for a, b in zip(v[:-1], v[1:]):
                counts[a - 1, b - 1] += 1
    return counts


def figure_transitions():
    counts = transition_counts([SEQ["train"], SEQ["val"], SEQ["test"]])
    probs = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    fig, axes = VAULT.canvas(1, 2, panel=(8.2, 6.6))
    ax = axes[0]
    sns.heatmap(probs, cmap=Palette.blue_to_orange(), ax=ax, linewidths=0.3, linecolor="#444444", cbar_kws={"label": "P(next | current)"},
                xticklabels=[str(i + 1) for i in range(VOCAB["k"])], yticklabels=[str(i + 1) for i in range(VOCAB["k"])])
    ax.set_xlabel("Next token")
    ax.set_ylabel("Current token")
    ax.set_title("Transition probabilities")
    ax.grid(False)
    panel_label(ax, "A")
    ax = axes[1]
    G = nx.DiGraph()
    usage = counts.sum(1) + counts.sum(0)
    for i in range(VOCAB["k"]):
        G.add_node(i + 1, size=usage[i])
    flat = [(probs[i, j], i, j) for i in range(VOCAB["k"]) for j in range(VOCAB["k"]) if i != j and counts[i, j] > 0]
    for weight, i, j in sorted(flat, reverse=True)[: max(12, VOCAB["k"] * 2)]:
        G.add_edge(i + 1, j + 1, weight=weight)
    pos = nx.spring_layout(G, seed=CFG.seed, k=1.4 / math.sqrt(max(G.number_of_nodes(), 1)))
    sizes = np.array([G.nodes[n]["size"] for n in G.nodes])
    sizes = 250 + 1700 * sizes / max(sizes.max(), 1)
    weights = np.array([G[u][v]["weight"] for u, v in G.edges]) if G.number_of_edges() else np.array([])
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=[Palette.token_colors(VOCAB["k"])[n - 1] for n in G.nodes], edgecolors=Palette.INK, linewidths=1.6, ax=ax)
    if len(weights):
        nx.draw_networkx_edges(G, pos, width=1 + 7 * weights / weights.max(), edge_color=weights, edge_cmap=Palette.blue_to_orange(), arrows=True, arrowsize=14, connectionstyle="arc3,rad=0.15", ax=ax, alpha=0.9)
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold", font_color=Palette.INK, ax=ax)
    ax.set_title("Strongest token-to-token transitions (self-loops hidden)")
    ax.axis("off")
    panel_label(ax, "B", dx=0.0)
    VAULT.finish(fig, "transitions", "Token transition structure", section="Interpretation")
    diag = float(np.trace(counts) / max(counts.sum(), 1))
    return {"self_transition_share": diag}


def figure_barcodes(per_class=10):
    colors = Palette.token_colors(VOCAB["k"])
    cmap = mcolors.ListedColormap([(1, 1, 1, 1)] + colors)
    fig, axes = VAULT.canvas(1, len(BEHAVIOURS), panel=(6.2, 5.2))
    rng = np.random.default_rng(CFG.seed)
    labels = catalog.set_index("recording_id")["behaviour_strict"]
    all_seqs = {**SEQ["train"], **SEQ["val"], **SEQ["test"]}
    for ax, b in zip(np.atleast_1d(axes), BEHAVIOURS):
        ids = [r for r in all_seqs if labels.get(r) == b]
        ids = list(rng.choice(ids, size=min(per_class, len(ids)), replace=False)) if ids else []
        width = CFG.max_segments_per_recording
        grid = np.zeros((max(len(ids), 1), width))
        for row, rid in enumerate(ids):
            v = all_seqs[rid][:width]
            grid[row, :len(v)] = v
        ax.imshow(grid, aspect="auto", cmap=cmap, vmin=0, vmax=VOCAB["k"], interpolation="nearest")
        ax.set_title(f"{b.capitalize()} recordings", color=Palette.BEHAVIOUR[b], fontsize=13)
        ax.set_xlabel("Segment position")
        ax.set_yticks([])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_edgecolor(Palette.BEHAVIOUR[b])
            spine.set_linewidth(3)
    return VAULT.finish(fig, "barcodes", "Token barcodes: each row is one recording, each colour one token", section="Interpretation")


def recording_histograms():
    rows, labels, species = [], [], []
    all_seqs = {**SEQ["train"], **SEQ["val"], **SEQ["test"]}
    lab = catalog.set_index("recording_id")["behaviour_strict"]
    spc = catalog.set_index("recording_id")["species"]
    ids = []
    for rid, v in all_seqs.items():
        if lab.get(rid) in BEHAVIOURS:
            ids.append(rid)
            rows.append(np.bincount(v, minlength=VOCAB_SIZE)[1:])
            labels.append(BEHAVIOURS.index(lab[rid]))
            species.append(species_names.index(spc[rid]))
    return np.asarray(ids), np.asarray(rows, dtype=float), np.asarray(labels), np.asarray(species)


def cramers_v_from_table(table):
    total = table.sum()
    expected = np.outer(table.sum(1), table.sum(0)) / max(total, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = np.nansum((table - expected) ** 2 / np.where(expected > 0, expected, np.nan))
    r, c = table.shape
    return math.sqrt(chi2 / (total * max(min(r, c) - 1, 1))) if total > 0 else 0.0


def association_test(hist, labels, n_perm=2000, seed=0):
    rng = np.random.default_rng(seed)
    C = len(BEHAVIOURS)
    def table(lbl):
        return np.stack([hist[lbl == c].sum(0) for c in range(C)])
    observed = cramers_v_from_table(table(labels))
    null = np.array([cramers_v_from_table(table(rng.permutation(labels))) for _ in range(n_perm)])
    return observed, float((1 + (null >= observed).sum()) / (1 + n_perm)), null


def figure_association():
    ids, hist, labels, species = recording_histograms()
    if len(ids) < 10 or len(np.unique(labels)) < 2:
        print("not enough labelled recordings for the association test")
        return None
    observed, p_value, null = association_test(hist, labels, n_perm=1000 if CFG.preset != "smoke" else 200, seed=CFG.seed)
    C = len(BEHAVIOURS)
    table = np.stack([hist[labels == c].sum(0) for c in range(C)])
    expected = np.outer(table.sum(1), table.sum(0)) / table.sum()
    residual = (table - expected) / np.sqrt(np.maximum(expected, 1e-9))
    fig, axes = VAULT.canvas(1, 3, panel=(7.0, 4.8))
    ax = axes[0]
    sns.heatmap(residual, cmap=Palette.diverging(), center=0, ax=ax, linewidths=0.5, linecolor="#333333", cbar_kws={"label": "Pearson residual"},
                xticklabels=[str(i + 1) for i in range(VOCAB["k"])], yticklabels=BEHAVIOURS)
    ax.set_xlabel("Token id")
    ax.set_title("Token use above or below expectation")
    ax.grid(False)
    panel_label(ax, "A")
    ax = axes[1]
    ax.hist(null, bins=30, color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.2, label="permuted labels")
    ax.axvline(observed, color=Palette.ORANGE, linewidth=3.5, label=f"observed V = {observed:.3f}")
    ax.set_xlabel("Cramer's V")
    ax.set_ylabel("Permutations")
    ax.set_title(f"Recording-level permutation test (p = {p_value:.3f})")
    ax.legend()
    panel_label(ax, "B")
    ax = axes[2]
    rng = np.random.default_rng(CFG.seed)
    overall = hist.sum(0) / hist.sum()
    base = {}
    for c, b in enumerate(BEHAVIOURS):
        mask = labels == c
        if mask.sum() < 2:
            continue
        draws = []
        for _ in range(200):
            pick = rng.choice(np.where(mask)[0], size=mask.sum(), replace=True)
            share = hist[pick].sum(0) / max(hist[pick].sum(), 1)
            draws.append(np.log2((share + 1e-3) / (overall + 1e-3)))
        draws = np.asarray(draws)
        base[b] = (draws.mean(0), np.quantile(draws, 0.025, axis=0), np.quantile(draws, 0.975, axis=0))
    y = 0
    ticks, names = [], []
    for b, (mean, lo, hi) in base.items():
        top = np.argsort(-np.abs(mean))[:4]
        for t in top[::-1]:
            ax.errorbar(mean[t], y, xerr=[[max(mean[t] - lo[t], 0)], [max(hi[t] - mean[t], 0)]], fmt="o", color=Palette.BEHAVIOUR[b], ecolor=Palette.INK, capsize=3, markersize=8)
            ticks.append(y)
            names.append(f"{b}: token {t + 1}")
            y += 1
        y += 0.6
    ax.axvline(0, color=Palette.INK, linewidth=1.5)
    ax.set_yticks(ticks)
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel("log2 enrichment (95% bootstrap over recordings)")
    ax.set_title("Most enriched tokens per behaviour")
    panel_label(ax, "C")
    VAULT.finish(fig, "association", "Do tokens carry behaviour information?", section="Interpretation")
    return {"cramers_v": float(observed), "permutation_p": p_value, "recordings": int(len(ids))}


with TIMER.track("interpretation"):
    figure_unit_gallery()
    RESULTS["transitions"] = figure_transitions()
    figure_barcodes()
    RESULTS["association"] = figure_association()
print(RESULTS["transitions"], RESULTS["association"])

# Part 14: ABX discrimination

## 14.1 A label-free test of "same species sounds closer"

ABX is a standard test from speech science. Take a segment **A** and a segment **X** from the same species, and a segment **B** from another species. The representation passes if X is closer to A than to B. Chance is 0.5.

Three points make this version safe:

- A and X always come from **different recordings**, otherwise the test only measures that neighbouring segments of one recording look alike.
- The evaluation uses **test recordings only**.
- The confidence interval resamples triplets. A "same recording" run is shown next to the safe run, so you can see how much easier the unsafe test is.

Here all encoders are re-embedded on the held-out subset, each at its own best layer and pooling from Part 8. This gives the held-out version of the encoder comparison.

In [ ]:
def embed_suite_for_encoder(name, layer, pooling):
    extractor = ENCODERS.extractor(name, CFG)
    if extractor is None:
        return None
    key = f"suite_{CFG.preset}_{name}_{layer}_{pooling}"
    return CACHE.get_or_compute(key, lambda: extractor.extract(store, SUITE.positions, layers=(layer,), poolings=(pooling,), desc="held-out")[(layer, pooling)])


with TIMER.track("held-out encoder comparison"):
    best_by_encoder = best_per_encoder(bench_frame)
    heldout_rows, SUITE_BANK = [], {"MFCC baseline": SUITE_MFCC}
    heldout_rows.append(SCORER.evaluate(SUITE_MFCC, "MFCC baseline"))
    for row in best_by_encoder.itertuples():
        if row.encoder == "mfcc":
            continue
        features = embed_suite_for_encoder(row.encoder, int(row.layer), row.pooling)
        if features is None:
            continue
        SUITE_BANK[row.encoder] = features
        heldout_rows.append(SCORER.evaluate(features, row.encoder, extra={"layer": int(row.layer), "pooling": row.pooling}))
    SUITE_BANK["Adapted (primary + adapter)"] = SUITE_ADAPTED
    heldout_rows.append(SCORER.evaluate(SUITE_ADAPTED, "Adapted (primary + adapter)", extra={"layer": PRIMARY["layer"], "pooling": PRIMARY["pooling"]}))
heldout_frame = pd.DataFrame(heldout_rows)
TABLES.show(heldout_frame[["variant", "abx", "abx_lo", "abx_hi", "silhouette", "davies_bouldin", "species_nmi", "behaviour_acc", "behaviour_f1", "effective_rank"]],
            "heldout_encoders", "Table 9. Held-out comparison of all encoders (test recordings for ABX and clustering; recording-level CV for behaviour)", precision=3)
RESULTS["heldout"] = heldout_frame


def pair_triplets(abx, ci, cj, n, rng):
    pool_i, pool_j = abx.index[ci], abx.index[cj]
    out = []
    for _ in range(n * 3):
        a = int(rng.choice(pool_i))
        candidates = pool_i[abx.groups[pool_i] != abx.groups[a]]
        if len(candidates) == 0:
            continue
        out.append((a, int(rng.choice(pool_j)), int(rng.choice(candidates))))
        if len(out) >= n:
            break
    return np.asarray(out, dtype=np.int64)


def figure_abx():
    held = SCORER.held_idx
    fig, axes = VAULT.canvas(2, 2, panel=(7.2, 5.0))
    ax = axes[0, 0]
    order = heldout_frame.sort_values("abx")
    colors = [Palette.GREEN if v == "MFCC baseline" else (Palette.ORANGE if "Adapted" in v else (Palette.BLUE if ENCODER_CATALOG.get(v) is None or ENCODER_CATALOG[v].family != "aves" else Palette.PURPLE)) for v in order["variant"]]
    bars = ax.barh(order["variant"], order["abx"], color=colors, edgecolor=Palette.INK, linewidth=1.4)
    ax.errorbar(order["abx"], range(len(order)), xerr=[np.clip(order["abx"] - order["abx_lo"], 0, None), np.clip(order["abx_hi"] - order["abx"], 0, None)], fmt="none", ecolor=Palette.INK, capsize=4, linewidth=1.6)
    ax.axvline(0.5, color=Palette.RED, linestyle="--", linewidth=2.2, label="chance")
    annotate_bars(ax, bars, "{:.3f}", horizontal=True, fontsize=8.5, offset=12)
    ax.set_xlabel("ABX accuracy (across recordings, 95% CI)")
    ax.set_title("Held-out ABX by representation")
    ax.legend(loc="lower right")
    panel_label(ax, "A")

    ax = axes[0, 1]
    variants = {"MFCC baseline": SUITE_MFCC, "Raw encoder": SUITE_RAW, "Adapted": SUITE_ADAPTED}
    same_trip = SCORER.abx.triplets(CFG.abx_triplets, across_groups=False, seed=CFG.seed + 5)
    width = 0.36
    for j, (name, raw) in enumerate(variants.items()):
        Xh = EmbeddingSpace.from_config(CFG).fit(raw).transform(raw)[held]
        safe = SCORER.abx.score(Xh, SCORER.triplets, n_boot=200)
        unsafe = SCORER.abx.score(Xh, same_trip, n_boot=200)
        ax.bar(j - width / 2, safe[0], width, color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.4, label="different recordings" if j == 0 else None)
        ax.bar(j + width / 2, unsafe[0], width, color=Palette.ORANGE, edgecolor=Palette.INK, linewidth=1.4, label="same recording allowed" if j == 0 else None)
        ax.errorbar([j - width / 2, j + width / 2], [safe[0], unsafe[0]], yerr=[[max(safe[0] - safe[1], 0), max(unsafe[0] - unsafe[1], 0)], [max(safe[2] - safe[0], 0), max(unsafe[2] - unsafe[0], 0)]], fmt="none", ecolor=Palette.INK, capsize=4)
        ax.text(j, max(safe[0], unsafe[0]) + 0.03, f"+{unsafe[0] - safe[0]:.3f}", ha="center", fontsize=9, fontweight="bold", color=Palette.RED)
    ax.axhline(0.5, color=Palette.RED, linestyle="--", linewidth=2)
    ax.set_xticks(range(len(variants)))
    ax.set_xticklabels(list(variants))
    ax.set_ylabel("ABX accuracy")
    ax.set_ylim(0.4, 1.05)
    ax.set_title("Unsafe triplets inflate ABX")
    ax.legend(loc="lower right", fontsize=8.5)
    panel_label(ax, "B")

    ax = axes[1, 0]
    Xa = EmbeddingSpace.from_config(CFG).fit(SUITE_ADAPTED).transform(SUITE_ADAPTED)[held]
    rng = np.random.default_rng(CFG.seed)
    matrix = np.full((len(species_names), len(species_names)), np.nan)
    present = SCORER.abx.classes
    for ci in present:
        for cj in present:
            if ci == cj:
                continue
            trip = pair_triplets(SCORER.abx, ci, cj, 300, rng)
            if len(trip):
                matrix[ci, cj] = ABXEvaluator.correct(Xa, trip).mean()
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap=Palette.diverging(), center=0.5, vmin=0.2, vmax=0.8, ax=ax, linewidths=1.0, linecolor=Palette.INK,
                xticklabels=[s.split(" ")[0] for s in species_names], yticklabels=[s.split(" ")[0] for s in species_names], cbar_kws={"label": "ABX accuracy"})
    ax.set_xlabel("Species of B")
    ax.set_ylabel("Species of A and X")
    ax.set_title("Which species pairs are easy or hard? (adapted)")
    ax.grid(False)
    panel_label(ax, "C")

    ax = axes[1, 1]
    sizes = [n for n in [100, 200, 400, 800, 1600, 3200, 6400, 12800] if n <= CFG.abx_triplets * 2]
    big = SCORER.abx.triplets(max(sizes), across_groups=True, seed=CFG.seed + 9)
    for name, raw, color in [("MFCC", SUITE_MFCC, Palette.GREEN), ("Raw", SUITE_RAW, Palette.BLUE), ("Adapted", SUITE_ADAPTED, Palette.ORANGE)]:
        Xh = EmbeddingSpace.from_config(CFG).fit(raw).transform(raw)[held]
        means, lows, highs = [], [], []
        for n in sizes:
            m, lo, hi = SCORER.abx.score(Xh, big[:n], n_boot=150)
            means.append(m); lows.append(lo); highs.append(hi)
        ax.plot(sizes, means, marker="o", color=color, label=name)
        ax.fill_between(sizes, lows, highs, color=color, alpha=0.2)
    ax.set_xscale("log")
    ax.axhline(0.5, color=Palette.RED, linestyle="--", linewidth=2)
    ax.set_xlabel("Number of triplets")
    ax.set_ylabel("ABX accuracy")
    ax.set_title("Is the estimate stable?")
    ax.legend()
    panel_label(ax, "D")
    return VAULT.finish(fig, "abx", "ABX species discrimination on held-out recordings", section="ABX")


figure_abx()

# Part 15: Behaviour probes

## 15.1 The labelled set and the classifier comparison

The question: **can a simple classifier read alarm, song or contact from the frozen embeddings?** All numbers come from **recording-level cross-validation** (`StratifiedGroupKFold`, repeated), so no recording ever appears in both the training and the test fold. Confidence intervals resample recordings. The labelled set is balanced across the three classes and capped per recording, so long recordings cannot dominate.

Nine classifiers are compared on the same features and the same folds. The features are the primary features from Part 10 (adapted embeddings, or raw encoder features if the adapter collapsed).

In [ ]:
LAB = LabelledSetBuilder(seg_meta, CFG, "strict").build(seed=CFG.seed)
if len(LAB) < 12:
    raise RuntimeError("Too few weakly labelled segments. Increase recordings_per_species or targeted_recordings_per_label.")
LAB_POS = LAB["seg_pos"].to_numpy()
LAB_Y = LAB["label"].to_numpy()
LAB_G = LAB["recording_id"].to_numpy()
N_CLASSES = len(BEHAVIOURS)
X_LAB = FEATURES_FULL[LAB_POS]
CV = CrossValidator(CFG.cv_folds, CFG.cv_repeats, CFG.seed, True, CFG.bootstrap_iterations)
print(f"labelled set: {len(LAB)} segments | {len(np.unique(LAB_G))} recordings | classes: {dict(zip(BEHAVIOURS, np.bincount(LAB_Y)))}")

PROBE_RESULTS = {}
with TIMER.track("classifier comparison"):
    for name in available_probes():
        try:
            PROBE_RESULTS[name] = CACHE.get_or_compute(f"probe_{name}_{FEATURE_NAME.replace(' ', '_')}", lambda name=name: CV.run(X_LAB, LAB_Y, LAB_G, name, N_CLASSES))
        except Exception as err:
            print(f"probe {name} failed: {str(err)[:100]}")
classifier_frame = pd.DataFrame([{"classifier": n, **{k: r[k] for k in ["accuracy", "accuracy_lo", "accuracy_hi", "macro_f1", "macro_f1_lo", "macro_f1_hi", "balanced_accuracy", "kappa", "mcc"]}} for n, r in PROBE_RESULTS.items()])
classifier_frame["roc_auc_ovr"] = [roc_auc_score(LAB_Y, r["probabilities"], multi_class="ovr") if len(np.unique(LAB_Y)) == N_CLASSES and np.isfinite(r["probabilities"]).all() else np.nan for r in PROBE_RESULTS.values()]
classifier_frame = classifier_frame.sort_values("macro_f1", ascending=False).reset_index(drop=True)
BEST_PROBE = classifier_frame.loc[0, "classifier"]
TABLES.show(classifier_frame, "classifier_comparison", f"Table 10. Weak-label behaviour probes on {FEATURE_NAME} (recording-level CV, 95% CI over recordings; chance accuracy = {1 / N_CLASSES:.3f})", precision=3)
print("best classifier by macro-F1:", BEST_PROBE)
RESULTS["classifiers"] = classifier_frame


def figure_classifiers():
    fig, axes = VAULT.canvas(1, 3, panel=(6.6, 5.0))
    order = classifier_frame.sort_values("macro_f1")
    for ax, (col, title) in zip(axes[:2], [("accuracy", "Accuracy"), ("macro_f1", "Macro-F1")]):
        colors = [Palette.ORANGE if c == BEST_PROBE else Palette.BLUE for c in order["classifier"]]
        bars = ax.barh(order["classifier"], order[col], color=colors, edgecolor=Palette.INK, linewidth=1.4)
        ax.errorbar(order[col], range(len(order)), xerr=[np.clip(order[col] - order[f"{col}_lo"], 0, None), np.clip(order[f"{col}_hi"] - order[col], 0, None)], fmt="none", ecolor=Palette.INK, capsize=4, linewidth=1.6)
        ax.axvline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2.2, label="chance")
        annotate_bars(ax, bars, "{:.3f}", horizontal=True, fontsize=8.5, offset=12)
        ax.set_xlim(0, min(1.15, order[f"{col}_hi"].max() * 1.22))
        ax.set_title(f"{title} (95% CI over recordings)")
    axes[0].legend(loc="lower right")
    ax = axes[2]
    data = [np.asarray(PROBE_RESULTS[c]["fold_scores"]) for c in order["classifier"]]
    box = ax.boxplot(data, vert=False, patch_artist=True, labels=order["classifier"], medianprops=dict(color=Palette.INK, linewidth=2.2))
    for patch, c in zip(box["boxes"], order["classifier"]):
        patch.set_facecolor(Palette.ORANGE if c == BEST_PROBE else Palette.BLUE_LIGHT)
        patch.set_edgecolor(Palette.INK)
        patch.set_linewidth(1.5)
    ax.axvline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2.2)
    ax.set_xlabel("Fold accuracy")
    ax.set_title("Spread across folds")
    return VAULT.finish(fig, "classifiers", "Nine probes on the same features and folds", section="Behaviour")


figure_classifiers()

## 15.2 Which features carry behaviour information?

Seven feature sets are compared with the same classifier (logistic regression) and the same folds:

| Feature set | Level | What it uses |
|---|---|---|
| MFCC | segment | Hand-made features |
| Raw encoder | segment | Frozen encoder output |
| Adapted | segment | After the contrastive adapter |
| Token histogram | recording | How often each acoustic unit occurs in the whole recording |
| Token bigrams | recording | How often each pair of consecutive units occurs |
| LM embedding | recording | The hidden state of the language model averaged over the recording |
| Adapted + histogram | recording + segment | Segment embedding joined with the recording histogram |

The recording-level features use the token sequences of the **whole recording** but never its label, so this is fair, but it does mean these features see more context than a single segment. The table states this clearly.

Below the segment-level scores, a **recording-level** table is given as well: one row per recording (mean embedding over all its segments), one prediction per recording. This is the level at which a biologist would use the result.

In [ ]:
class RecordingFeatureFactory:
    def __init__(self, sequences, features_full, seg_meta, lm_model, lm_trainer, k):
        self.sequences = sequences
        self.features_full = features_full
        self.seg_meta = seg_meta
        self.lm_model = lm_model
        self.lm_trainer = lm_trainer
        self.k = k
        self.by_recording = seg_meta.groupby("recording_id")["seg_pos"].apply(np.array).to_dict()
        self.cache = {}

    def histogram(self, rid):
        v = self.sequences[rid]
        h = np.bincount(v, minlength=self.k + 1)[1:].astype(np.float32)
        return h / max(h.sum(), 1)

    def bigrams(self, rid, dim=128):
        v = self.sequences[rid]
        out = np.zeros(dim, dtype=np.float32)
        for a, b in zip(v[:-1], v[1:]):
            out[(a * (self.k + 1) + b) % dim] += 1
        return out / max(out.sum(), 1)

    @torch.no_grad()
    def lm_embedding(self, rid):
        v = self.sequences[rid][: self.lm_trainer.max_len]
        x = torch.tensor([[self.lm_model.bos] + list(v)], dtype=torch.long, device=DEVICE)
        h = self.lm_model.hidden(x)[0, 1:]
        return h.mean(0).cpu().numpy() if len(h) else np.zeros(self.lm_model.norm.normalized_shape[0], dtype=np.float32)

    def mean_embedding(self, rid):
        return self.features_full[self.by_recording[rid]].mean(0)

    def table(self, rids, kind):
        fn = {"hist": self.histogram, "bigram": self.bigrams, "lm": self.lm_embedding, "mean": self.mean_embedding}[kind]
        return np.stack([fn(r) for r in rids]).astype(np.float32)


ALL_SEQS = {**SEQ["train"], **SEQ["val"], **SEQ["test"]}
for rid in seg_meta["recording_id"].unique():
    if rid not in ALL_SEQS:
        ALL_SEQS[rid] = seg_meta.loc[seg_meta["recording_id"] == rid].sort_values("segment_index")["token"].to_numpy()[: CFG.max_segments_per_recording]
FACTORY = RecordingFeatureFactory(ALL_SEQS, FEATURES_FULL, seg_meta, LM_MODELS[0].eval(), LM_TRAINER, VOCAB["k"])
LAB_MFCC = CACHE.get_or_compute("lab_mfcc", lambda: MFCCBaseline(CFG.sample_rate).extract(store, LAB_POS))
unique_lab_recordings = list(dict.fromkeys(LAB_G))
row_of = {r: i for i, r in enumerate(unique_lab_recordings)}
row_index = np.array([row_of[g] for g in LAB_G])
hist_rows = FACTORY.table(unique_lab_recordings, "hist")[row_index]
bigram_rows = FACTORY.table(unique_lab_recordings, "bigram")[row_index]
lm_rows_feat = FACTORY.table(unique_lab_recordings, "lm")[row_index]

SEGMENT_FEATURES = {
    "MFCC": LAB_MFCC,
    "Raw encoder": RAW_FULL[LAB_POS],
    "Adapted": ADAPTED_FULL[LAB_POS],
    "Token histogram (recording)": hist_rows,
    "Token bigrams (recording)": bigram_rows,
    "LM embedding (recording)": lm_rows_feat,
    "Adapted + histogram": np.concatenate([ADAPTED_FULL[LAB_POS], hist_rows], axis=1),
}
FEATURE_RESULTS = {}
with TIMER.track("feature set comparison"):
    for name, X in SEGMENT_FEATURES.items():
        FEATURE_RESULTS[name] = CACHE.get_or_compute(f"feat_{name.replace(' ', '_')}", lambda X=X: CV.run(X, LAB_Y, LAB_G, "logreg", N_CLASSES))
feature_frame = pd.DataFrame([{"features": n, **{k: r[k] for k in ["accuracy", "accuracy_lo", "accuracy_hi", "macro_f1", "macro_f1_lo", "macro_f1_hi", "balanced_accuracy", "mcc"]}} for n, r in FEATURE_RESULTS.items()])
TABLES.show(feature_frame, "behaviour_features", "Table 11. Segment-level behaviour probe by feature set (logistic regression, recording-level CV)", precision=3)

rec_labels = LAB.groupby("recording_id")["label"].first().reindex(unique_lab_recordings).to_numpy()
REC_FEATURES = {"Mean raw encoder": None, "Mean adapted": None}
REC_FEATURES = {
    "Mean raw encoder": np.stack([RAW_FULL[FACTORY.by_recording[r]].mean(0) for r in unique_lab_recordings]),
    "Mean adapted": FACTORY.table(unique_lab_recordings, "mean"),
    "Token histogram": FACTORY.table(unique_lab_recordings, "hist"),
    "Token bigrams": FACTORY.table(unique_lab_recordings, "bigram"),
    "LM embedding": FACTORY.table(unique_lab_recordings, "lm"),
}
REC_FEATURES["Mean adapted + histogram"] = np.concatenate([REC_FEATURES["Mean adapted"], REC_FEATURES["Token histogram"]], axis=1)
rec_groups = np.asarray(unique_lab_recordings)
REC_CV = CrossValidator(CFG.cv_folds, max(CFG.cv_repeats, 2), CFG.seed, True, CFG.bootstrap_iterations)
REC_RESULTS = {}
if len(np.unique(rec_labels)) == N_CLASSES and np.bincount(rec_labels.astype(int)).min() >= 3:
    for name, X in REC_FEATURES.items():
        REC_RESULTS[name] = REC_CV.run(X, rec_labels.astype(int), rec_groups, "logreg", N_CLASSES)
    rec_frame = pd.DataFrame([{"features": n, "recordings": len(rec_labels), **{k: r[k] for k in ["accuracy", "accuracy_lo", "accuracy_hi", "macro_f1", "macro_f1_lo", "macro_f1_hi", "balanced_accuracy", "mcc"]}} for n, r in REC_RESULTS.items()])
    TABLES.show(rec_frame, "behaviour_recording_level", "Table 12. Recording-level behaviour probe (one prediction per recording)", precision=3)
    RESULTS["recording_level"] = rec_frame
else:
    rec_frame = pd.DataFrame()
    print("recording-level table skipped: too few recordings per class")
RESULTS["features"] = feature_frame


def figure_features():
    fig, axes = VAULT.canvas(1, 2, panel=(8.6, 5.4))
    for ax, frame, title, name_col in [(axes[0], feature_frame, "Segment-level rows", "features"), (axes[1], rec_frame, "Recording-level rows", "features")]:
        if frame is None or len(frame) == 0:
            ax.axis("off")
            continue
        order = frame.sort_values("macro_f1")
        colors = [Palette.ORANGE if ("histogram" in f.lower() or "bigram" in f.lower() or "LM" in f) else (Palette.GREEN if f == "MFCC" else Palette.BLUE) for f in order[name_col]]
        bars = ax.barh(order[name_col], order["macro_f1"], color=colors, edgecolor=Palette.INK, linewidth=1.4)
        ax.errorbar(order["macro_f1"], range(len(order)), xerr=[np.clip(order["macro_f1"] - order["macro_f1_lo"], 0, None), np.clip(order["macro_f1_hi"] - order["macro_f1"], 0, None)], fmt="none", ecolor=Palette.INK, capsize=4, linewidth=1.6)
        ax.axvline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2.2, label="chance")
        annotate_bars(ax, bars, "{:.3f}", horizontal=True, fontsize=8.5, offset=12)
        ax.set_xlim(0, min(1.2, float(order["macro_f1_hi"].max()) * 1.25))
        ax.set_xlabel("Macro-F1 (95% CI over recordings)")
        ax.set_title(title)
    axes[0].legend(loc="lower right")
    return VAULT.finish(fig, "behaviour_features", "Which representation carries behaviour information?", section="Behaviour")


figure_features()

## 15.3 Leakage audit: how much did random segment folds inflate the old results?

The old Run notebooks used random segment-level folds. Here the same classifiers on the same features are scored twice: with **naive** folds (segments shuffled freely) and with **recording-level** folds. The difference is the inflation caused by recording leakage. The figure also shows what fraction of naive test segments had a sibling segment from the same recording in the training data.

In [ ]:
class LeakageAudit:
    def __init__(self, cfg, X, y, groups, n_classes):
        self.cfg = cfg
        self.X = X
        self.y = y
        self.groups = groups
        self.n_classes = n_classes

    def sibling_share(self, seed=0):
        splitter = StratifiedKFold(n_splits=self.cfg.cv_folds, shuffle=True, random_state=seed)
        shares = []
        for train_idx, test_idx in splitter.split(self.X, self.y):
            train_groups = set(self.groups[train_idx])
            shares.append(np.mean([g in train_groups for g in self.groups[test_idx]]))
        return float(np.mean(shares))

    def run(self, probes):
        rows = []
        naive = CrossValidator(self.cfg.cv_folds, self.cfg.cv_repeats, self.cfg.seed, False, self.cfg.bootstrap_iterations)
        grouped = CrossValidator(self.cfg.cv_folds, self.cfg.cv_repeats, self.cfg.seed, True, self.cfg.bootstrap_iterations)
        for name in probes:
            a = naive.run(self.X, self.y, self.groups, name, self.n_classes)
            b = grouped.run(self.X, self.y, self.groups, name, self.n_classes)
            rows.append({"classifier": name, "naive_acc": a["accuracy"], "naive_lo": a["accuracy_lo"], "naive_hi": a["accuracy_hi"],
                         "grouped_acc": b["accuracy"], "grouped_lo": b["accuracy_lo"], "grouped_hi": b["accuracy_hi"],
                         "naive_f1": a["macro_f1"], "grouped_f1": b["macro_f1"]})
        frame = pd.DataFrame(rows)
        frame["inflation_acc"] = frame["naive_acc"] - frame["grouped_acc"]
        frame["inflation_f1"] = frame["naive_f1"] - frame["grouped_f1"]
        return frame


audit_probes = [p for p in ["logreg", "svm_rbf", "random_forest", "mlp", "knn"] if p in available_probes()]
with TIMER.track("leakage audit"):
    audit = LeakageAudit(CFG, X_LAB, LAB_Y, LAB_G, N_CLASSES)
    leak_frame = audit.run(audit_probes)
    sibling = audit.sibling_share(CFG.seed)
TABLES.show(leak_frame, "leakage_audit", f"Table 13. Naive segment folds against recording-level folds ({sibling * 100:.0f}% of naive test segments had a sibling from the same recording in training)", precision=3)
RESULTS["leakage"] = {"table": leak_frame, "sibling_share": sibling}


def figure_leakage():
    fig, axes = VAULT.canvas(1, 2, panel=(8.4, 5.2))
    ax = axes[0]
    x = np.arange(len(leak_frame))
    ax.bar(x - 0.19, leak_frame["naive_acc"], 0.38, color=Palette.ORANGE, edgecolor=Palette.INK, linewidth=1.4, label="naive segment folds")
    ax.bar(x + 0.19, leak_frame["grouped_acc"], 0.38, color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.4, label="recording-level folds")
    ax.errorbar(x + 0.19, leak_frame["grouped_acc"], yerr=[np.clip(leak_frame["grouped_acc"] - leak_frame["grouped_lo"], 0, None), np.clip(leak_frame["grouped_hi"] - leak_frame["grouped_acc"], 0, None)], fmt="none", ecolor=Palette.INK, capsize=4)
    for i, row in leak_frame.iterrows():
        ax.text(i, max(row["naive_acc"], row["grouped_acc"]) + 0.03, f"+{row['inflation_acc']:.2f}", ha="center", fontweight="bold", color=Palette.RED, fontsize=10)
    ax.axhline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2, label="chance")
    ax.set_xticks(x)
    ax.set_xticklabels(leak_frame["classifier"], rotation=15)
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.12)
    ax.set_title("Same features, same classifiers, two split rules")
    ax.legend(loc="lower right", fontsize=8.5)
    panel_label(ax, "A")
    ax = axes[1]
    sizes = [sibling, 1 - sibling]
    wedges, texts, autotexts = ax.pie(sizes, labels=["has a sibling in\ntraining data", "no sibling"], colors=[Palette.ORANGE, Palette.BLUE], autopct="%1.0f%%", startangle=90,
                                      wedgeprops=dict(edgecolor=Palette.INK, linewidth=2), textprops=dict(fontsize=11, fontweight="bold"))
    for t in autotexts:
        t.set_color("white")
        t.set_fontsize(14)
    ax.set_title("Naive test segments")
    panel_label(ax, "B", dx=0.0)
    return VAULT.finish(fig, "leakage_audit", "Recording leakage audit", section="Behaviour")


figure_leakage()

## 15.4 Confusion, ROC, precision-recall and calibration

These four plots use the out-of-fold probabilities of the best classifier, so every point was predicted by a model that never saw its recording.

In [ ]:
def oof_confusion(y, predictions):
    cm = np.zeros((N_CLASSES, N_CLASSES))
    for pred in predictions:
        cm += confusion_matrix(y, pred, labels=range(N_CLASSES)) / len(predictions)
    return cm


def expected_calibration_error(confidence, correct, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (confidence > lo) & (confidence <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - confidence[m].mean())
    return float(ece)


def figure_curves(result, name):
    proba = result["probabilities"]
    cm = oof_confusion(LAB_Y, result["predictions"])
    fig, axes = VAULT.canvas(2, 2, panel=(7.0, 5.4))
    ax = axes[0, 0]
    normed = cm / np.maximum(cm.sum(1, keepdims=True), 1e-9)
    sns.heatmap(normed, annot=np.array([[f"{normed[i, j]:.2f}\n(n={cm[i, j]:.0f})" for j in range(N_CLASSES)] for i in range(N_CLASSES)]), fmt="", cmap=Palette.blues(), vmin=0, vmax=1, ax=ax,
                xticklabels=BEHAVIOURS, yticklabels=BEHAVIOURS, linewidths=1.5, linecolor=Palette.INK, cbar_kws={"label": "row share"})
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix ({name})")
    ax.grid(False)
    panel_label(ax, "A")
    onehot = label_binarize(LAB_Y, classes=list(range(N_CLASSES)))
    ax = axes[0, 1]
    for c, b in enumerate(BEHAVIOURS):
        fpr, tpr, _ = roc_curve(onehot[:, c], proba[:, c])
        ax.plot(fpr, tpr, color=Palette.BEHAVIOUR[b], label=f"{b} (AUC = {auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], color=Palette.INK, linestyle="--", linewidth=1.6)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("One-vs-rest ROC")
    ax.legend(loc="lower right")
    panel_label(ax, "B")
    ax = axes[1, 0]
    for c, b in enumerate(BEHAVIOURS):
        precision, recall, _ = precision_recall_curve(onehot[:, c], proba[:, c])
        ax.plot(recall, precision, color=Palette.BEHAVIOUR[b], label=f"{b} (AP = {average_precision_score(onehot[:, c], proba[:, c]):.3f})")
        ax.axhline(onehot[:, c].mean(), color=Palette.BEHAVIOUR[b], linestyle=":", linewidth=1.4)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("One-vs-rest precision-recall (dotted = prevalence)")
    ax.legend(loc="lower left")
    panel_label(ax, "C")
    ax = axes[1, 1]
    confidence = proba.max(1)
    correct = (proba.argmax(1) == LAB_Y).astype(float)
    edges = np.linspace(0, 1, 9)
    mids, accs, counts = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (confidence > lo) & (confidence <= hi)
        if m.sum() >= 3:
            mids.append(confidence[m].mean()); accs.append(correct[m].mean()); counts.append(m.sum())
    ax.plot([0, 1], [0, 1], color=Palette.INK, linestyle="--", linewidth=1.6, label="perfect calibration")
    ax.plot(mids, accs, marker="o", color=Palette.ORANGE, label=f"model (ECE = {expected_calibration_error(confidence, correct):.3f})")
    for m, a, n in zip(mids, accs, counts):
        ax.annotate(str(n), (m, a), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8, color=Palette.BLUE_DARK, fontweight="bold")
    ax.set_xlabel("Predicted confidence")
    ax.set_ylabel("Observed accuracy")
    ax.set_title("Calibration")
    ax.legend(loc="upper left")
    panel_label(ax, "D")
    return VAULT.finish(fig, "behaviour_curves", f"Behaviour probe diagnostics ({name}, out-of-fold)", section="Behaviour")


figure_curves(PROBE_RESULTS[BEST_PROBE], BEST_PROBE)


def figure_species_breakdown(result):
    pred = result["predictions"][0]
    rows = []
    for si, species in enumerate(species_names):
        m = (LAB["species_index"].to_numpy() == si)
        for c, b in enumerate(BEHAVIOURS):
            mm = m & (LAB_Y == c)
            rows.append({"species": species, "behaviour": b, "n": int(mm.sum()), "recall": float((pred[mm] == c).mean()) if mm.sum() >= 3 else np.nan})
    frame = pd.DataFrame(rows)
    table = frame.pivot(index="species", columns="behaviour", values="recall").reindex(species_names)[BEHAVIOURS]
    counts = frame.pivot(index="species", columns="behaviour", values="n").reindex(species_names)[BEHAVIOURS]
    fig, ax = VAULT.canvas(1, 1, figsize=(9.0, 5.6))
    labels = np.array([[("n/a" if np.isnan(table.iloc[i, j]) else f"{table.iloc[i, j]:.2f}\n(n={counts.iloc[i, j]:.0f})") for j in range(N_CLASSES)] for i in range(len(species_names))])
    sns.heatmap(table.astype(float), annot=labels, fmt="", cmap=Palette.diverging(), center=1 / N_CLASSES, vmin=0, vmax=1, ax=ax, linewidths=1.5, linecolor=Palette.INK,
                yticklabels=[s for s in species_names], cbar_kws={"label": "recall"})
    ax.set_xlabel("True weak behaviour")
    ax.set_ylabel("")
    ax.set_title("Where does the probe work? Recall per species and behaviour")
    ax.grid(False)
    VAULT.finish(fig, "species_breakdown", "Behaviour probe per species", section="Behaviour")
    return frame


species_breakdown = figure_species_breakdown(PROBE_RESULTS[BEST_PROBE])
TABLES.show(species_breakdown, "species_breakdown", "Table 14. Recall of each behaviour class within each species (out-of-fold)", precision=3, render_png=False)

## 15.5 Semi-supervised pseudo-labelling (done without leakage)

Most recordings have no behaviour label. The idea from Runs 9 and 10 was to let a teacher classifier label the unlabelled segments and to add the confident ones to the training data. Two fixes make it honest:

- The teacher is trained **only on the training fold**, and the unlabelled pool comes only from recordings that never have a strict label, so no test recording is ever pseudo-labelled.
- The confidence threshold is **not chosen by looking at the results**. All thresholds are reported, and the headline uses a threshold that is fixed before the run (0.8).

The paired difference to the supervised baseline has its own bootstrap interval over recordings.

In [ ]:
class PseudoLabelHook:
    def __init__(self, X_lab, y, pool_X, threshold, cap_factor=2.0, teacher="logreg"):
        self.X_lab = X_lab
        self.y = y
        self.pool_X = pool_X
        self.threshold = threshold
        self.cap_factor = cap_factor
        self.teacher = teacher
        self.added = []

    def __call__(self, probe_name, train_idx, seed):
        teacher = make_probe(self.teacher, seed).fit(self.X_lab[train_idx], self.y[train_idx])
        proba = teacher.predict_proba(self.pool_X)
        confidence, guess = proba.max(1), proba.argmax(1)
        keep = []
        for c in range(N_CLASSES):
            cap = int(self.cap_factor * max((self.y[train_idx] == c).sum(), 1))
            candidates = np.where((guess == c) & (confidence >= self.threshold))[0]
            candidates = candidates[np.argsort(-confidence[candidates])][:cap]
            keep.append(candidates)
        keep = np.concatenate(keep) if keep else np.zeros(0, dtype=int)
        self.added.append(len(keep))
        Xtr = np.concatenate([self.X_lab[train_idx], self.pool_X[keep]]) if len(keep) else self.X_lab[train_idx]
        ytr = np.concatenate([self.y[train_idx], guess[keep]]) if len(keep) else self.y[train_idx]
        return make_probe(probe_name, seed), Xtr, ytr


unlabelled_pool = seg_meta[seg_meta["behaviour_strict"].isna() & seg_meta["recording_id"].map(lambda r: r not in set(LAB_G))]
pool_rng = np.random.default_rng(CFG.seed)
pool_pos = np.sort(pool_rng.choice(unlabelled_pool["seg_pos"].to_numpy(), size=min(6000, len(unlabelled_pool)), replace=False)) if len(unlabelled_pool) else np.zeros(0, dtype=int)
POOL_X = FEATURES_FULL[pool_pos]
print("unlabelled pool:", len(pool_pos), "segments")

pseudo_rows, PSEUDO_RESULTS = [], {}
with TIMER.track("pseudo-labelling"):
    for student in [p for p in ["logreg", "mlp"] if p in available_probes()]:
        base = PROBE_RESULTS[student]
        pseudo_rows.append({"student": student, "threshold": np.nan, "added_mean": 0.0, "macro_f1": base["macro_f1"], "macro_f1_lo": base["macro_f1_lo"], "macro_f1_hi": base["macro_f1_hi"],
                            "accuracy": base["accuracy"], "diff_f1": 0.0, "diff_lo": 0.0, "diff_hi": 0.0, "diff_p": np.nan})
        if len(pool_pos) == 0:
            continue
        for threshold in CFG.pseudo_thresholds:
            hook = PseudoLabelHook(X_LAB, LAB_Y, POOL_X, threshold)
            result = CV.run(X_LAB, LAB_Y, LAB_G, student, N_CLASSES, fit_hook=hook)
            PSEUDO_RESULTS[(student, threshold)] = result
            diff = paired_difference(LAB_Y, result["predictions"], base["predictions"], LAB_G, N_CLASSES, "macro_f1", CFG.bootstrap_iterations, CFG.seed)
            pseudo_rows.append({"student": student, "threshold": threshold, "added_mean": float(np.mean(hook.added)), "macro_f1": result["macro_f1"], "macro_f1_lo": result["macro_f1_lo"],
                                "macro_f1_hi": result["macro_f1_hi"], "accuracy": result["accuracy"], "diff_f1": diff["diff"], "diff_lo": diff["lo"], "diff_hi": diff["hi"], "diff_p": diff["p"]})
pseudo_frame = pd.DataFrame(pseudo_rows)
TABLES.show(pseudo_frame, "pseudo_labelling", "Table 15. Pseudo-labelling sweep (no threshold selection; diff = change in macro-F1 against the supervised baseline)", precision=3)
RESULTS["pseudo"] = pseudo_frame


def figure_pseudo():
    fig, axes = VAULT.canvas(1, 2, panel=(8.0, 5.0))
    for student, color in zip(pseudo_frame["student"].unique(), [Palette.BLUE, Palette.ORANGE]):
        part = pseudo_frame[(pseudo_frame["student"] == student) & pseudo_frame["threshold"].notna()].sort_values("threshold")
        base = pseudo_frame[(pseudo_frame["student"] == student) & pseudo_frame["threshold"].isna()].iloc[0]
        if part.empty:
            continue
        axes[0].plot(part["threshold"], part["macro_f1"], marker="o", color=color, label=student)
        axes[0].fill_between(part["threshold"], part["macro_f1_lo"], part["macro_f1_hi"], color=color, alpha=0.16)
        axes[0].axhline(base["macro_f1"], color=color, linestyle="--", linewidth=1.8)
        axes[1].errorbar(part["threshold"], part["diff_f1"], yerr=[np.clip(part["diff_f1"] - part["diff_lo"], 0, None), np.clip(part["diff_hi"] - part["diff_f1"], 0, None)], fmt="o-", color=color, capsize=5, label=student)
    axes[0].axhline(1 / N_CLASSES, color=Palette.RED, linestyle=":", linewidth=2)
    axes[0].set_xlabel("Confidence threshold")
    axes[0].set_ylabel("Macro-F1 (dashed = supervised only)")
    axes[0].set_title("Pseudo-labelling sweep")
    axes[0].legend()
    axes[0].invert_xaxis()
    axes[1].axhline(0, color=Palette.INK, linewidth=2)
    axes[1].set_xlabel("Confidence threshold")
    axes[1].set_ylabel("Change in macro-F1 (95% CI)")
    axes[1].set_title("Is the gain different from zero?")
    axes[1].legend()
    axes[1].invert_xaxis()
    return VAULT.finish(fig, "pseudo_labelling", "Semi-supervised pseudo-labelling without leakage", section="Behaviour")


if len(pseudo_frame) > 2:
    figure_pseudo()

## 15.6 Frame-level BiLSTM probe

Pooling squeezes a whole segment into one vector and loses timing. The Run 8 idea was a small **bidirectional LSTM** that reads the encoder's frame sequence (average-pooled to a fixed number of steps) with attention pooling. It is scored with exactly the same recording-level folds as everything else, and it is compared with the pooled logistic regression using a paired bootstrap.

In [ ]:
class FrameLSTMClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, steps=25, hidden=64, epochs=60, lr=2e-3, batch_size=32, patience=10, dropout=0.3, seed=42):
        self.steps = steps
        self.hidden = hidden
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.patience = patience
        self.dropout = dropout
        self.seed = seed

    def _net(self, d_in, n_out):
        class Net(nn.Module):
            def __init__(inner):
                super().__init__()
                inner.proj = nn.Sequential(nn.Linear(d_in, 128), nn.GELU(), nn.Dropout(self.dropout))
                inner.rnn = nn.LSTM(128, self.hidden, batch_first=True, bidirectional=True)
                inner.attn = nn.Linear(2 * self.hidden, 1)
                inner.out = nn.Sequential(nn.Dropout(self.dropout), nn.Linear(2 * self.hidden, n_out))

            def forward(inner, x):
                h, _ = inner.rnn(inner.proj(x))
                w = torch.softmax(inner.attn(h), dim=1)
                return inner.out((w * h).sum(1))
        return Net()

    def _shape(self, X):
        return np.asarray(X, dtype=np.float32).reshape(len(X), self.steps, -1)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        X = self._shape(X)
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        self.mu_ = X.mean((0, 1), keepdims=True)
        self.sd_ = X.std((0, 1), keepdims=True) + 1e-4
        X = (X - self.mu_) / self.sd_
        try:
            Xtr, Xva, ytr, yva = train_test_split(X, y_idx, test_size=0.15, stratify=y_idx, random_state=self.seed)
        except ValueError:
            Xtr, Xva, ytr, yva = X, X, y_idx, y_idx
        self.net_ = self._net(X.shape[2], len(self.classes_)).to(DEVICE)
        optimizer = torch.optim.AdamW(self.net_.parameters(), lr=self.lr, weight_decay=1e-3)
        weights = torch.tensor(len(ytr) / (len(self.classes_) * np.maximum(np.bincount(ytr, minlength=len(self.classes_)), 1)), dtype=torch.float32, device=DEVICE)
        loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)), batch_size=self.batch_size, shuffle=True)
        Xva_t, yva_t = torch.tensor(Xva).to(DEVICE), torch.tensor(yva).to(DEVICE)
        best, state, wait = float("inf"), None, 0
        for _ in range(self.epochs):
            self.net_.train()
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                xb = xb + 0.05 * torch.randn_like(xb)
                optimizer.zero_grad()
                F.cross_entropy(self.net_(xb), yb, weight=weights).backward()
                torch.nn.utils.clip_grad_norm_(self.net_.parameters(), 1.0)
                optimizer.step()
            self.net_.eval()
            with torch.no_grad():
                val = F.cross_entropy(self.net_(Xva_t), yva_t).item()
            if val < best - 1e-4:
                best, wait = val, 0
                state = {k: v.detach().clone() for k, v in self.net_.state_dict().items()}
            else:
                wait += 1
                if wait >= self.patience:
                    break
        if state is not None:
            self.net_.load_state_dict(state)
        return self

    def predict_proba(self, X):
        self.net_.eval()
        X = (self._shape(X) - self.mu_) / self.sd_
        with torch.no_grad():
            return torch.softmax(self.net_(torch.tensor(X, dtype=torch.float32).to(DEVICE)), dim=1).cpu().numpy()

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]


with TIMER.track("frame-level BiLSTM"):
    FRAME_GRID = CACHE.get_or_compute("lab_frame_grid", lambda: get_primary_extractor().frame_grid(store, LAB_POS, layer=PRIMARY["layer"], steps=CFG.frame_steps))
    FRAME_FLAT = FRAME_GRID.reshape(len(FRAME_GRID), -1).astype(np.float32)
    lstm_hook = lambda name, train_idx, seed: (FrameLSTMClassifier(steps=CFG.frame_steps, seed=seed), FRAME_FLAT[train_idx], LAB_Y[train_idx])
    LSTM_RESULT = CV.run(FRAME_FLAT, LAB_Y, LAB_G, "bilstm", N_CLASSES, fit_hook=lstm_hook)
lstm_vs_logreg = paired_difference(LAB_Y, LSTM_RESULT["predictions"], FEATURE_RESULTS["Raw encoder"]["predictions"], LAB_G, N_CLASSES, "macro_f1", CFG.bootstrap_iterations, CFG.seed)
print(f"BiLSTM macro-F1 = {LSTM_RESULT['macro_f1']:.3f} [{LSTM_RESULT['macro_f1_lo']:.3f}, {LSTM_RESULT['macro_f1_hi']:.3f}] | difference to pooled raw logreg = {lstm_vs_logreg['diff']:+.3f} (95% CI {lstm_vs_logreg['lo']:+.3f} to {lstm_vs_logreg['hi']:+.3f})")

master_rows = []
def add_master(label, result, family):
    master_rows.append({"method": label, "family": family, "accuracy": result["accuracy"], "accuracy_lo": result["accuracy_lo"], "accuracy_hi": result["accuracy_hi"],
                        "macro_f1": result["macro_f1"], "macro_f1_lo": result["macro_f1_lo"], "macro_f1_hi": result["macro_f1_hi"], "mcc": result["mcc"]})


add_master("MFCC + logreg", FEATURE_RESULTS["MFCC"], "baseline")
add_master("Raw encoder + logreg", FEATURE_RESULTS["Raw encoder"], "supervised")
add_master("Adapted + logreg", FEATURE_RESULTS["Adapted"], "supervised")
add_master(f"Primary features + {BEST_PROBE}", PROBE_RESULTS[BEST_PROBE], "supervised")
if "ensemble" in PROBE_RESULTS:
    add_master("Primary features + soft-vote ensemble", PROBE_RESULTS["ensemble"], "ensemble")
for key in [(s, 0.8) for s in ["logreg", "mlp"]]:
    if key in PSEUDO_RESULTS:
        add_master(f"Pseudo-labels (thr 0.8) + {key[0]}", PSEUDO_RESULTS[key], "semi-supervised")
add_master("Frame-level BiLSTM", LSTM_RESULT, "sequence model")
add_master("Token histogram + logreg", FEATURE_RESULTS["Token histogram (recording)"], "tokens")
add_master("LM embedding + logreg", FEATURE_RESULTS["LM embedding (recording)"], "tokens")
behaviour_master = pd.DataFrame(master_rows)
TABLES.show(behaviour_master, "behaviour_master", "Table 16. All behaviour methods on one scale (recording-level CV, 95% CI over recordings)", precision=3)
RESULTS["behaviour_master"] = behaviour_master


def figure_behaviour_master():
    family_color = {"baseline": Palette.GREEN, "supervised": Palette.BLUE, "ensemble": Palette.PURPLE, "semi-supervised": Palette.ORANGE, "sequence model": Palette.RED, "tokens": "#7FB8F0"}
    order = behaviour_master.sort_values("macro_f1")
    fig, ax = VAULT.canvas(1, 1, figsize=(11, 6.2))
    bars = ax.barh(order["method"], order["macro_f1"], color=[family_color[f] for f in order["family"]], edgecolor=Palette.INK, linewidth=1.5)
    ax.errorbar(order["macro_f1"], range(len(order)), xerr=[np.clip(order["macro_f1"] - order["macro_f1_lo"], 0, None), np.clip(order["macro_f1_hi"] - order["macro_f1"], 0, None)], fmt="none", ecolor=Palette.INK, capsize=4, linewidth=1.7)
    ax.axvline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2.4, label=f"chance = {1 / N_CLASSES:.2f}")
    annotate_bars(ax, bars, "{:.3f}", horizontal=True, fontsize=9, offset=14)
    ax.set_xlabel("Macro-F1 (95% CI over recordings)")
    ax.set_xlim(0, min(1.2, float(order["macro_f1_hi"].max()) * 1.2))
    handles = [mpatches.Patch(facecolor=c, edgecolor=Palette.INK, label=f) for f, c in family_color.items()]
    ax.legend(handles=handles + [Line2D([0], [0], color=Palette.RED, linestyle="--", linewidth=2.4, label="chance")], loc="lower right", fontsize=8.5)
    return VAULT.finish(fig, "behaviour_master", "Weak-label behaviour: every method on one scale", section="Behaviour")


figure_behaviour_master()

# Part 16: Syllable discovery at frame level

## 16.1 Units inside a segment

Segment-level tokens treat 3 seconds as one unit. Birds, however, sing **syllables** that are much shorter. This study repeats the discovery at **frame level** (one frame is about 20 ms) on a sample of labelled segments:

1. Take the encoder frames of a sample of labelled segments.
2. Reduce them with PCA and cluster the frames with K-Means (a sweep over K), fitted only on frames from **training** recordings.
3. Turn each frame sequence into simple **rhythm and repertoire features**: the histogram of frame tokens, the histogram of token pairs, the number of token runs per second, and the mean run length.
4. Test how well these features predict weak behaviour with the same recording-level folds as before.

This is the honest version of the old "syllable discovery" pilot, which used 44 files and tested on 14 samples.

In [ ]:
class FrameUnitStudy:
    def __init__(self, cfg, store, extractor_fn, primary, seg_meta, lab):
        self.cfg = cfg
        self.store = store
        self.extractor_fn = extractor_fn
        self.primary = primary
        rng = np.random.default_rng(cfg.seed)
        n = min(cfg.frame_study_segments, len(lab))
        picks = rng.choice(len(lab), size=n, replace=False)
        self.sample = lab.iloc[np.sort(picks)].reset_index(drop=True)

    def load_frames(self):
        extractor = self.extractor_fn()
        return extractor.frames_raw(self.store, self.sample["seg_pos"].to_numpy(), layer=self.primary["layer"])

    def fit_frame_space(self, frames, train_mask, n_components=32, max_frames=60000, seed=0):
        pool = np.concatenate([f for f, m in zip(frames, train_mask) if m]).astype(np.float32)
        if len(pool) > max_frames:
            pool = pool[np.random.default_rng(seed).choice(len(pool), max_frames, replace=False)]
        scaler = StandardScaler().fit(pool)
        pca = PCA(n_components=min(n_components, pool.shape[1], len(pool) - 1), random_state=seed).fit(scaler.transform(pool))
        return scaler, pca, pca.transform(scaler.transform(pool))

    @staticmethod
    def features_from_tokens(token_seqs, k, seconds_per_frame):
        rows = []
        for seq in token_seqs:
            hist = np.bincount(seq, minlength=k).astype(np.float32)
            hist /= max(hist.sum(), 1)
            pair = np.zeros(128, dtype=np.float32)
            for a, b in zip(seq[:-1], seq[1:]):
                pair[(a * k + b) % 128] += 1
            pair /= max(pair.sum(), 1)
            change = np.concatenate([[True], seq[1:] != seq[:-1]]) if len(seq) else np.zeros(0, dtype=bool)
            runs = int(change.sum())
            run_rate = runs / max(len(seq) * seconds_per_frame, 1e-6)
            mean_run = len(seq) / max(runs, 1)
            rows.append(np.concatenate([hist, pair, [run_rate, mean_run]]))
        return np.asarray(rows, dtype=np.float32)


study = FrameUnitStudy(CFG, store, get_primary_extractor, PRIMARY, seg_meta, LAB)
with TIMER.track("frame extraction"):
    FRAMES = CACHE.get_or_compute("frame_study_frames", study.load_frames)
train_flags = (study.sample["split"] == "train").to_numpy()
if train_flags.sum() < 5:
    train_flags = np.ones(len(train_flags), dtype=bool)
scaler_f, pca_f, pool_projected = study.fit_frame_space(FRAMES, train_flags, seed=CFG.seed)
frame_projected = [pca_f.transform(scaler_f.transform(f.astype(np.float32))) for f in FRAMES]
y_frame = study.sample["label"].to_numpy()
g_frame = study.sample["recording_id"].to_numpy()
frame_k_grid = [k for k in (8, 16, 32, 64) if k * 8 < len(pool_projected)] or [4]
frame_cv = CrossValidator(CFG.cv_folds, 1, CFG.seed, True, min(CFG.bootstrap_iterations, 300))
frame_rows, FRAME_TOKENS = [], {}
seconds_per_frame = FRAME_STRIDE / CFG.sample_rate
with TIMER.track("frame unit sweep"):
    for k in frame_k_grid:
        model = KMeansUnits().fit(pool_projected, k, CFG.seed)
        seqs = [model.predict(p) for p in frame_projected]
        FRAME_TOKENS[("kmeans", k)] = seqs
        X = FrameUnitStudy.features_from_tokens(seqs, k, seconds_per_frame)
        if len(np.unique(y_frame)) == N_CLASSES and min(len(np.unique(g_frame[y_frame == c])) for c in range(N_CLASSES)) >= 2:
            res = frame_cv.run(X, y_frame, g_frame, "logreg", N_CLASSES)
            dominant = np.array([np.bincount(s, minlength=k).argmax() for s in seqs])
            nmi = normalized_mutual_info_score(y_frame, dominant)
            ari = adjusted_rand_score(y_frame, dominant)
            frame_rows.append({"method": "kmeans", "K": k, "macro_f1": res["macro_f1"], "macro_f1_lo": res["macro_f1_lo"], "macro_f1_hi": res["macro_f1_hi"], "accuracy": res["accuracy"], "nmi_dominant": nmi, "ari_dominant": ari,
                               "runs_per_second": float(X[:, -2].mean())})
    best_k = frame_k_grid[len(frame_k_grid) // 2]
    vq_model = VQUnits(CFG.vq_epochs).fit(pool_projected, best_k, CFG.seed)
    vq_seqs = [vq_model.predict(p) for p in frame_projected]
    FRAME_TOKENS[("vq", best_k)] = vq_seqs
    X = FrameUnitStudy.features_from_tokens(vq_seqs, best_k, seconds_per_frame)
    if len(np.unique(y_frame)) == N_CLASSES and min(len(np.unique(g_frame[y_frame == c])) for c in range(N_CLASSES)) >= 2:
        res = frame_cv.run(X, y_frame, g_frame, "logreg", N_CLASSES)
        dominant = np.array([np.bincount(s, minlength=best_k).argmax() for s in vq_seqs])
        frame_rows.append({"method": "vq", "K": best_k, "macro_f1": res["macro_f1"], "macro_f1_lo": res["macro_f1_lo"], "macro_f1_hi": res["macro_f1_hi"], "accuracy": res["accuracy"],
                           "nmi_dominant": normalized_mutual_info_score(y_frame, dominant), "ari_dominant": adjusted_rand_score(y_frame, dominant), "runs_per_second": float(X[:, -2].mean())})
frame_frame = pd.DataFrame(frame_rows)
TABLES.show(frame_frame, "frame_units", "Table 17. Frame-level unit discovery: behaviour information in rhythm and repertoire features", precision=3)
RESULTS["frame_units"] = frame_frame


def figure_frame_units():
    fig, axes = VAULT.canvas(1, 3, panel=(6.8, 4.9))
    ax = axes[0]
    km = frame_frame[frame_frame["method"] == "kmeans"]
    ax.plot(km["K"], km["macro_f1"], marker="o", color=Palette.BLUE, label="K-Means")
    ax.fill_between(km["K"], km["macro_f1_lo"], km["macro_f1_hi"], color=Palette.BLUE, alpha=0.18)
    vq = frame_frame[frame_frame["method"] == "vq"]
    if len(vq):
        ax.scatter(vq["K"], vq["macro_f1"], s=160, color=Palette.ORANGE, edgecolors=Palette.INK, linewidths=1.6, zorder=5, label="VQ autoencoder")
    ax.axhline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2, label="chance")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Number of frame tokens (K)")
    ax.set_ylabel("Macro-F1")
    ax.set_title("Behaviour from frame-token features")
    ax.legend()
    panel_label(ax, "A")
    ax = axes[1]
    key = ("kmeans", best_k)
    rates = []
    for seq, label in zip(FRAME_TOKENS[key], y_frame):
        change = np.concatenate([[True], seq[1:] != seq[:-1]]) if len(seq) else np.zeros(0, dtype=bool)
        rates.append((BEHAVIOURS[label], change.sum() / max(len(seq) * seconds_per_frame, 1e-6)))
    rate_frame = pd.DataFrame(rates, columns=["behaviour", "runs_per_second"])
    present = [b for b in BEHAVIOURS if b in set(rate_frame["behaviour"])]
    sns.violinplot(data=rate_frame, x="behaviour", y="runs_per_second", order=present, palette=[Palette.BEHAVIOUR[b] for b in present], ax=ax, inner="quartile", linewidth=1.4, cut=0)
    ax.set_xlabel("")
    ax.set_ylabel("Token changes per second")
    ax.set_title("Rhythm by behaviour")
    panel_label(ax, "B")
    ax = axes[2]
    picks = {}
    for c, b in enumerate(BEHAVIOURS):
        idx = np.where(y_frame == c)[0]
        if len(idx):
            picks[b] = int(idx[0])
    colors = Palette.token_colors(best_k)
    for row, (b, i) in enumerate(picks.items()):
        seq = FRAME_TOKENS[key][i]
        ax.imshow(np.array([seq]), aspect="auto", cmap=mcolors.ListedColormap(colors), vmin=0, vmax=best_k - 1, extent=[0, len(seq) * seconds_per_frame, row, row + 0.9], interpolation="nearest")
    ax.set_ylim(0, len(picks))
    ax.set_yticks([i + 0.45 for i in range(len(picks))])
    ax.set_yticklabels(list(picks))
    ax.set_xlabel("Time (s)")
    ax.set_title("Frame-token timelines (one example each)")
    ax.grid(False)
    panel_label(ax, "C")
    return VAULT.finish(fig, "frame_units", "Syllable-level unit discovery", section="Frame units")


if len(frame_frame):
    figure_frame_units()

# Part 17: The controlled experiment suite

## 17.1 How the suite works

This part holds the **ablation experiments** that a reviewer normally asks for: change one thing, keep everything else fixed, and measure the effect. Every experiment is compared with a **baseline** on the same held-out subset, with a confidence interval for the difference and a p-value. Because we test many variants, the p-values are corrected with the **Benjamini-Hochberg** false discovery rate procedure.

`SuiteLedger` keeps one row per (variant, metric). The sign of every difference is flipped where lower is better (Davies-Bouldin, perplexity), so a **positive improvement always means better than the baseline**.

| Id | Experiment | What changes |
|---|---|---|
| E01 | Baselines | MFCC, raw encoder, adapted embeddings |
| E02 | Layer choice | Which encoder layer is used |
| E03 | Pooling | Mean, mean + std, mean + max |
| E04 | Adapter capacity | Depth and output size of the projection head |
| E05 | Adapter training length | Number of epochs |
| E06 | Segment duration | 2, 3 or 4 seconds |
| E07 | Silence trimming and denoising | Trim strength, noise reduction on or off |
| E08 | Augmentation diversity | Which augmentations create the two views |
| E09 | Number of units | K in K-Means |
| E10 | Clustering method and covariance | K-Means, GMM diagonal, GMM full, Ward |
| E11 | HDBSCAN grid | Minimum cluster size |
| E12 | PCA choice | No PCA, less variance, whitening |
| E13 | Language model size | Small, base, large |
| E14 | Language model context | Context window of 4, 8, 16 tokens |
| E15 | Classifier | Nine probes (from Part 15) |
| E16 | Token features | Histogram, bigrams, LM embedding (from Part 15) |
| E17 | ABX triplet rule | Across recordings against same recording |
| E18 | Representation checkpoints | Same rows as E01 (MFCC, raw, adapted) |
| E19 | Random seeds | Five adapter seeds |
| E20 | Per-species results | Descriptive table |
| E21 | Encoder family | All encoders on held-out data |
| E22 | Token method | K-Means, GMM, Ward, VQ vocabularies compared by language model perplexity |
| E23 | Pseudo-labelling | Threshold sweep (from Part 15) |
| E24 | Data scaling | Part 18 |
| E25 | Leakage audit | Part 15 |

In [ ]:
def bh_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    values = p[ok]
    order = np.argsort(values)
    ranked = values[order] * len(values) / (np.arange(len(values)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted = np.empty_like(values)
    adjusted[order] = np.minimum(ranked, 1.0)
    out[ok] = adjusted
    return out


LOWER_IS_BETTER = {"davies_bouldin", "ppl"}
METRIC_LABEL = {"abx": "ABX accuracy", "silhouette": "Silhouette", "davies_bouldin": "Davies-Bouldin", "behaviour_f1": "Behaviour macro-F1", "ppl": "Test perplexity"}


def approximate_difference(value, lo, hi, base, base_lo, base_hi):
    se = math.sqrt(((hi - lo) / 3.92) ** 2 + ((base_hi - base_lo) / 3.92) ** 2)
    delta = value - base
    if not np.isfinite(se) or se <= 0:
        return delta, np.nan, np.nan, np.nan
    z = delta / se
    return delta, delta - 1.96 * se, delta + 1.96 * se, float(2 * (1 - sps.norm.cdf(abs(z))))


def paired_hit_difference(hits_a, hits_b, n_boot=500, seed=0):
    d = hits_a - hits_b
    rng = np.random.default_rng(seed)
    draws = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)])
    p = float(min(1.0, 2 * min((draws <= 0).mean(), (draws >= 0).mean()) + 1.0 / n_boot))
    return float(d.mean()), float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975)), p


class SuiteLedger:
    def __init__(self):
        self.rows = []
        self.variants = []

    def add_variant(self, exp_id, experiment, row, base_row, scorer=None, tag=None, base_tag=None, is_baseline=False, comparison_note="", metrics=None):
        self.variants.append({"exp": exp_id, "experiment": experiment, **row})
        for metric in (metrics or ["abx", "silhouette", "davies_bouldin", "behaviour_f1"]):
            if metric not in row or not np.isfinite(row.get(metric, np.nan)):
                continue
            value, lo, hi = row[metric], row.get(f"{metric}_lo", np.nan), row.get(f"{metric}_hi", np.nan)
            base = base_row.get(metric, np.nan) if base_row else np.nan
            delta = dlo = dhi = p = np.nan
            if base_row is not None and np.isfinite(base) and not is_baseline:
                if scorer is not None and tag in scorer.details and base_tag in scorer.details:
                    if metric == "abx":
                        delta, dlo, dhi, p = paired_hit_difference(scorer.details[tag]["hits"], scorer.details[base_tag]["hits"], seed=CFG.seed)
                    elif metric == "behaviour_f1" and scorer.details[tag]["predictions"] is not None and scorer.details[base_tag]["predictions"] is not None:
                        d = paired_difference(SUITE.behaviour[scorer.lab_idx], scorer.details[tag]["predictions"], scorer.details[base_tag]["predictions"], SUITE.groups[scorer.lab_idx], len(BEHAVIOURS), "macro_f1", CFG.bootstrap_iterations, CFG.seed)
                        delta, dlo, dhi, p = d["diff"], d["lo"], d["hi"], d["p"]
                if not np.isfinite(delta):
                    delta, dlo, dhi, p = approximate_difference(value, lo, hi, base, base_row.get(f"{metric}_lo", np.nan), base_row.get(f"{metric}_hi", np.nan))
            sign = -1.0 if metric in LOWER_IS_BETTER else 1.0
            self.rows.append({"exp": exp_id, "experiment": experiment, "variant": row["variant"], "metric": metric, "value": value, "lo": lo, "hi": hi,
                              "baseline": base, "improvement": sign * delta if np.isfinite(delta) else np.nan,
                              "imp_lo": (sign * dlo if sign > 0 else sign * dhi) if np.isfinite(dlo) else np.nan,
                              "imp_hi": (sign * dhi if sign > 0 else sign * dlo) if np.isfinite(dhi) else np.nan,
                              "p": p, "is_baseline": is_baseline, "note": comparison_note})

    def add_ppl(self, exp_id, experiment, variant, ppl, lo, hi, base_ppl, delta, dlo, dhi, p, is_baseline=False):
        self.variants.append({"exp": exp_id, "experiment": experiment, "variant": variant, "ppl": ppl, "ppl_lo": lo, "ppl_hi": hi})
        self.rows.append({"exp": exp_id, "experiment": experiment, "variant": variant, "metric": "ppl", "value": ppl, "lo": lo, "hi": hi, "baseline": base_ppl,
                          "improvement": -delta if np.isfinite(delta) else np.nan, "imp_lo": -dhi if np.isfinite(dhi) else np.nan, "imp_hi": -dlo if np.isfinite(dlo) else np.nan,
                          "p": p, "is_baseline": is_baseline, "note": ""})

    def frame(self):
        frame = pd.DataFrame(self.rows)
        if frame.empty:
            return frame
        frame["p_adj"] = np.nan
        for metric, idx in frame.groupby("metric").groups.items():
            mask = frame.index.isin(idx) & ~frame["is_baseline"]
            frame.loc[mask, "p_adj"] = bh_adjust(frame.loc[mask, "p"].to_numpy())
        frame["significant"] = frame["p_adj"] < 0.05
        return frame


LEDGER = SuiteLedger()
SUITE_TIMER = {}
E01_rows = {}
with TIMER.track("suite E01 baselines"):
    E01_rows["adapted"] = SCORER.evaluate(SUITE_ADAPTED, "E01 adapted (baseline)")
    E01_rows["raw"] = SCORER.evaluate(SUITE_RAW, "E01 raw encoder")
    E01_rows["mfcc"] = SCORER.evaluate(SUITE_MFCC, "E01 MFCC")
LEDGER.add_variant("E01", "Baselines", E01_rows["adapted"], None, is_baseline=True)
LEDGER.add_variant("E01", "Baselines", E01_rows["raw"], E01_rows["adapted"], SCORER, "E01 raw encoder", "E01 adapted (baseline)")
LEDGER.add_variant("E01", "Baselines", E01_rows["mfcc"], E01_rows["adapted"], SCORER, "E01 MFCC", "E01 adapted (baseline)")
BASE_ADAPTED, BASE_RAW = E01_rows["adapted"], E01_rows["raw"]
BASE_ADAPTED_TAG, BASE_RAW_TAG = "E01 adapted (baseline)", "E01 raw encoder"
print("baseline ABX / silhouette / behaviour F1:", round(BASE_ADAPTED["abx"], 3), round(BASE_ADAPTED["silhouette"], 3), round(BASE_ADAPTED.get("behaviour_f1", np.nan), 3))

## 17.2 Experiments on layers, pooling, adapter and clustering choices (E02 to E05, E08 to E12, E19, E21)

In [ ]:
FEATURE_METRICS = ["abx", "silhouette", "davies_bouldin", "behaviour_f1"]
CLUSTER_METRICS = ["silhouette", "davies_bouldin"]
POOLINGS_ALL = ("mean", "mean_std", "mean_max")


def train_adapter_variant(tag, depth=2, out_dim=None, hidden=None, epochs=None, seed=None, views=None, patience=None, config=None):
    cfg = replace(config or CFG, adapter_patience=patience or (config or CFG).adapter_patience)
    trainer = ContrastiveAdapterTrainer(RAW_FULL.shape[1], cfg, depth=depth, out_dim=out_dim, hidden=hidden, epochs=epochs, seed=seed or CFG.seed)
    train_v, val_v = views if views is not None else (TRAIN_VIEWS, VAL_VIEWS)
    history = trainer.fit(train_v, val_v, verbose=False)
    return trainer.transform(RAW_FULL[SUITE.positions]), history


with TIMER.track("suite E02 E03 layers pooling"):
    primary_layers = sorted(set(ENCODERS.layer_ids(PRIMARY["encoder"], CFG.layer_fractions)) | {PRIMARY["layer"]})
    suite_layer_bank = CACHE.get_or_compute("suite_layer_bank", lambda: get_primary_extractor().extract(store, SUITE.positions, layers=tuple(primary_layers), poolings=POOLINGS_ALL, desc="layers"))
    for layer in primary_layers:
        if layer == PRIMARY["layer"]:
            continue
        tag = f"E02 layer {layer}"
        row = SCORER.evaluate(suite_layer_bank[(layer, PRIMARY["pooling"])], tag)
        LEDGER.add_variant("E02", "Layer choice", row, BASE_RAW, SCORER, tag, BASE_RAW_TAG, metrics=FEATURE_METRICS)
    LEDGER.add_variant("E02", "Layer choice", {**BASE_RAW, "variant": f"E02 layer {PRIMARY['layer']} (primary)"}, BASE_RAW, is_baseline=True, metrics=FEATURE_METRICS)
    for pooling in POOLINGS_ALL:
        if pooling == PRIMARY["pooling"]:
            LEDGER.add_variant("E03", "Pooling", {**BASE_RAW, "variant": f"E03 {pooling} (primary)"}, BASE_RAW, is_baseline=True, metrics=FEATURE_METRICS)
            continue
        tag = f"E03 {pooling}"
        row = SCORER.evaluate(suite_layer_bank[(PRIMARY["layer"], pooling)], tag)
        LEDGER.add_variant("E03", "Pooling", row, BASE_RAW, SCORER, tag, BASE_RAW_TAG, metrics=FEATURE_METRICS)

with TIMER.track("suite E04 E05 adapter"):
    e = CFG.adapter_epochs
    capacity = [("depth 1 (linear)", dict(depth=1)), ("depth 3", dict(depth=3)), ("output 32", dict(out_dim=32)), ("output 64", dict(out_dim=64)), ("output 256", dict(out_dim=256)), ("hidden 512", dict(hidden=512))]
    for name, kw in capacity:
        tag = f"E04 {name}"
        feats, _ = train_adapter_variant(tag, **kw)
        LEDGER.add_variant("E04", "Adapter capacity", SCORER.evaluate(feats, tag), BASE_ADAPTED, SCORER, tag, BASE_ADAPTED_TAG, metrics=FEATURE_METRICS)
    LEDGER.add_variant("E04", "Adapter capacity", {**BASE_ADAPTED, "variant": "E04 depth 2, output 128 (primary)"}, BASE_ADAPTED, is_baseline=True, metrics=FEATURE_METRICS)
    for n_epochs in sorted({max(2, e // 8), max(2, e // 4), max(2, e // 2), e * 2}):
        tag = f"E05 {n_epochs} epochs"
        feats, _ = train_adapter_variant(tag, epochs=n_epochs, patience=999)
        LEDGER.add_variant("E05", "Adapter training length", SCORER.evaluate(feats, tag), BASE_ADAPTED, SCORER, tag, BASE_ADAPTED_TAG, metrics=FEATURE_METRICS)
    LEDGER.add_variant("E05", "Adapter training length", {**BASE_ADAPTED, "variant": f"E05 {e} epochs, early stopping (primary)"}, BASE_ADAPTED, is_baseline=True, metrics=FEATURE_METRICS)

with TIMER.track("suite E08 augmentation"):
    aug_sets = {"noise and gain": ("noise", "gain"), "speed only": ("speed",), "masking and shift": ("timemask", "shift"), "spectral (band, background)": ("bandpass", "mixbg"), "all seven": AugmentationBank.ALL}
    n_small_train, n_small_val = min(len(TRAIN_POS), 2000), min(len(VAL_POS), 500)
    aug_rows = {}
    for name, ops in aug_sets.items():
        tv = CACHE.get_or_compute(f"aug_train_{name}", lambda ops=ops: fresh_view_builder().build(TRAIN_POS[:n_small_train], 2, ops, CFG.seed))
        vv = CACHE.get_or_compute(f"aug_val_{name}", lambda ops=ops: fresh_view_builder().build(VAL_POS[:n_small_val], 2, ops, CFG.seed + 1))
        tag = f"E08 {name}"
        feats, _ = train_adapter_variant(tag, views=(tv, vv))
        aug_rows[name] = SCORER.evaluate(feats, tag)
    base_aug = aug_rows["all seven"]
    for name, row in aug_rows.items():
        LEDGER.add_variant("E08", "Augmentation diversity", row, base_aug, SCORER, row["variant"], "E08 all seven", is_baseline=(name == "all seven"), metrics=FEATURE_METRICS)

with TIMER.track("suite E09 to E12 clustering"):
    for k in CFG.k_grid:
        tag = f"E09 K = {k}"
        row = SCORER.evaluate(SUITE_ADAPTED, tag, k=k, with_behaviour=False)
        LEDGER.add_variant("E09", "Number of units", row, BASE_ADAPTED, None, is_baseline=False, metrics=CLUSTER_METRICS)
    for method in ["gmm_diag", "gmm_full", "ward"]:
        tag = f"E10 {method}"
        row = SCORER.evaluate(SUITE_ADAPTED, tag, cluster_method=method, with_behaviour=False)
        LEDGER.add_variant("E10", "Clustering method", row, BASE_ADAPTED, None, metrics=CLUSTER_METRICS)
    LEDGER.add_variant("E10", "Clustering method", {**BASE_ADAPTED, "variant": "E10 kmeans (primary)"}, BASE_ADAPTED, is_baseline=True, metrics=CLUSTER_METRICS)
    n_held = len(SCORER.held_idx)
    for size in sorted({max(5, n_held // 150), max(8, n_held // 75), max(12, n_held // 40)}):
        tag = f"E11 min size {size}"
        row = SCORER.evaluate(SUITE_ADAPTED, tag, cluster_method="hdbscan", cluster_param=size, with_behaviour=False)
        LEDGER.add_variant("E11", "HDBSCAN grid", row, BASE_ADAPTED, None, metrics=CLUSTER_METRICS, comparison_note=f"noise {row.get('noise_fraction', np.nan):.2f}")
    for name, kw in [("no PCA", dict(use_pca=False)), ("PCA 80% variance", dict(variance=0.80)), ("PCA 99% variance", dict(variance=0.99)), ("PCA + whitening", dict(whiten=True))]:
        tag = f"E12 {name}"
        LEDGER.add_variant("E12", "PCA choice", SCORER.evaluate(SUITE_ADAPTED, tag, space_kwargs=kw), BASE_ADAPTED, SCORER, tag, BASE_ADAPTED_TAG, metrics=FEATURE_METRICS)

with TIMER.track("suite E19 seeds"):
    seed_rows = [BASE_ADAPTED]
    for seed in CFG.cluster_seeds:
        if seed == CFG.seed:
            continue
        tag = f"E19 adapter seed {seed}"
        feats, _ = train_adapter_variant(tag, seed=seed)
        row = SCORER.evaluate(feats, tag)
        seed_rows.append(row)
        LEDGER.add_variant("E19", "Random seeds", row, BASE_ADAPTED, SCORER, tag, BASE_ADAPTED_TAG, metrics=FEATURE_METRICS)
    seed_frame = pd.DataFrame(seed_rows)
    seed_summary = seed_frame[["abx", "silhouette", "davies_bouldin", "behaviour_f1"]].agg(["mean", "std", "min", "max"]).T.reset_index().rename(columns={"index": "metric"})
TABLES.show(seed_summary, "seed_variance", "Table 18. Variation across adapter seeds (E19)", precision=4)
RESULTS["seed_summary"] = seed_summary

with TIMER.track("suite E21 encoders"):
    for name, feats in SUITE_BANK.items():
        if name in ("Adapted (primary + adapter)",):
            continue
        tag = f"E21 {name}"
        LEDGER.add_variant("E21", "Encoder family", SCORER.evaluate(feats, tag), BASE_RAW, SCORER, tag, BASE_RAW_TAG, metrics=FEATURE_METRICS)
print("ledger rows so far:", len(LEDGER.rows))

## 17.3 Preprocessing, language model and token method experiments (E06, E07, E13, E14, E22)

Preprocessing experiments need to cut the raw audio again, so they run on a **sample of validation and test recordings** (`ablation_recordings`, at most 60 seconds each and 8 segments per recording). For each setting the primary encoder embeds the new segments and the same ABX and clustering scores are computed. Because the segments differ between settings, the comparison uses the approximate difference from the two confidence intervals (this is stated in the table notes).

The language model experiments compare **paired** perplexities: the same test recordings, the same seed, and a bootstrap over recordings.

In [ ]:
class AblationSubset:
    def __init__(self, species, groups):
        self.species = np.asarray(species)
        self.groups = np.asarray(groups)
        self.is_held = np.ones(len(species), dtype=bool)
        self.is_labelled = np.zeros(len(species), dtype=bool)
        self.behaviour = np.full(len(species), -1)


class PreprocessAblation:
    def __init__(self, cfg, catalog, max_seconds=60.0):
        self.cfg = cfg
        pool = catalog[catalog["split"].isin(["val", "test"])]
        per_species = max(2, cfg.ablation_recordings // len(cfg.species))
        parts = [g.sample(n=min(per_species, len(g)), random_state=cfg.seed) for _, g in pool.groupby("species")]
        self.recordings = pd.concat(parts).reset_index(drop=True)
        self.audio = {}
        for row in tqdm(self.recordings.itertuples(), total=len(self.recordings), desc="loading ablation audio", leave=False):
            try:
                wave, _ = librosa.load(row.file, sr=cfg.sample_rate, mono=True, duration=max_seconds)
                if len(wave) > cfg.sample_rate:
                    self.audio[row.recording_id] = wave
            except Exception:
                continue

    def segments(self, **overrides):
        pre = AudioPreprocessor.from_config(self.cfg, max_segments=8, **overrides)
        waves, valid, species, groups = [], [], [], []
        for row in self.recordings.itertuples():
            if row.recording_id not in self.audio:
                continue
            seg, v = pre.cut(pre.clean(self.audio[row.recording_id]))
            for s, n in zip(seg, v):
                waves.append(s.astype(np.float32) / 32768.0)
                valid.append(int(n))
                species.append(species_to_index[row.species])
                groups.append(row.recording_id)
        length = max(len(w) for w in waves)
        waves = np.stack([np.pad(w, (0, length - len(w))) for w in waves])
        return waves, np.asarray(valid), np.asarray(species), np.asarray(groups)

    def evaluate(self, tag, **overrides):
        waves, valid, species, groups = self.segments(**overrides)
        extractor = get_primary_extractor()
        feats = extractor.extract(ArrayWaveSource(waves, valid), np.arange(len(waves)), layers=(PRIMARY["layer"],), poolings=(PRIMARY["pooling"],), progress=False)[(PRIMARY["layer"], PRIMARY["pooling"])]
        scorer = SuiteScorer(AblationSubset(species, groups), CFG)
        row = scorer.evaluate(feats, tag, with_behaviour=False)
        row["segments"] = len(waves)
        return row


with TIMER.track("suite E06 E07 preprocessing"):
    ABLATION = PreprocessAblation(CFG, catalog)
    base_cfg_kwargs = dict(segment_seconds=CFG.segment_seconds, trim_db=CFG.trim_top_db, denoise=CFG.denoise)
    pre_base = ABLATION.evaluate("E06 3.0 s (primary)", **base_cfg_kwargs)
    LEDGER.add_variant("E06", "Segment duration", pre_base, pre_base, is_baseline=True, metrics=["abx", "silhouette", "davies_bouldin"])
    for seconds in (2.0, 4.0):
        row = ABLATION.evaluate(f"E06 {seconds:.1f} s", **{**base_cfg_kwargs, "segment_seconds": seconds})
        LEDGER.add_variant("E06", "Segment duration", row, pre_base, metrics=["abx", "silhouette", "davies_bouldin"], comparison_note="approximate: different segments")
    LEDGER.add_variant("E07", "Trimming and denoising", {**pre_base, "variant": f"E07 trim {CFG.trim_top_db:.0f} dB, denoise {CFG.denoise} (primary)"}, pre_base, is_baseline=True, metrics=["abx", "silhouette", "davies_bouldin"])
    for db in (20.0, 40.0, None):
        if db == CFG.trim_top_db:
            continue
        row = ABLATION.evaluate(f"E07 trim {db if db is not None else 'off'}", **{**base_cfg_kwargs, "trim_db": db})
        LEDGER.add_variant("E07", "Trimming and denoising", row, pre_base, metrics=["abx", "silhouette", "davies_bouldin"], comparison_note="approximate: different segments")
    row = ABLATION.evaluate(f"E07 denoise {not CFG.denoise}", **{**base_cfg_kwargs, "denoise": not CFG.denoise})
    LEDGER.add_variant("E07", "Trimming and denoising", row, pre_base, metrics=["abx", "silhouette", "davies_bouldin"], comparison_note="approximate: different segments")
    del ABLATION.audio

In [ ]:
def recording_sums(trainer, model, seqs, chunk=None):
    keys, nll, mask = trainer.per_recording_nll(model, seqs)
    totals = defaultdict(lambda: [0.0, 0.0])
    for k, n, m in zip(keys, nll, mask):
        rid = str(k).split("|")[0]
        totals[rid][0] += float(n.sum())
        totals[rid][1] += float(m.sum())
    ids = sorted(totals)
    return ids, np.array([totals[i][0] for i in ids]), np.array([totals[i][1] for i in ids])


def paired_ppl_difference(sums_a, counts_a, sums_b, counts_b, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    point = math.exp(sums_a.sum() / counts_a.sum()) - math.exp(sums_b.sum() / counts_b.sum())
    draws = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(sums_a), len(sums_a))
        draws.append(math.exp(sums_a[pick].sum() / counts_a[pick].sum()) - math.exp(sums_b[pick].sum() / counts_b[pick].sum()))
    draws = np.asarray(draws)
    p = float(min(1.0, 2 * min((draws <= 0).mean(), (draws >= 0).mean()) + 1.0 / n_boot))
    return float(point), float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975)), p


def chunk_sequences(seqs, size):
    out = {}
    for rid, v in seqs.items():
        for i in range(0, len(v), size):
            piece = v[i:i + size]
            if len(piece) >= 2:
                out[f"{rid}|{i // size}"] = piece
    return out


def ppl_from_sums(sums, counts, n_boot, seed):
    return bootstrap_perplexity(sums, counts, n_boot, seed)


def run_lm_variant(train, val, test, trainer_kwargs, max_len):
    trainer = LMTrainer(CFG, K_TOKENS, max_len, **trainer_kwargs)
    model, history = trainer.fit(train, val, CFG.seed)
    return trainer, model, recording_sums(trainer, model, test)


with TIMER.track("suite E13 E14 E22 language model"):
    base_ids, base_sums, base_counts = recording_sums(LM_TRAINER, LM_MODELS[0], SEQ["test"])
    base_ppl = ppl_from_sums(base_sums, base_counts, CFG.bootstrap_iterations, CFG.seed)
    LEDGER.add_ppl("E13", "Language model size", "E13 base (primary)", *base_ppl, base_ppl[0], 0.0, 0.0, 0.0, np.nan, is_baseline=True)
    sizes = {"small (d=64, 1 layer)": dict(d_model=64, heads=2, layers=1, ff=128), "large (d=256, 4 layers)": dict(d_model=256, heads=8, layers=4, ff=1024)}
    for name, kw in sizes.items():
        trainer, model, (ids, sums, counts) = run_lm_variant(SEQ["train"], SEQ["val"], SEQ["test"], kw, CFG.max_segments_per_recording)
        ppl = ppl_from_sums(sums, counts, CFG.bootstrap_iterations, CFG.seed)
        d = paired_ppl_difference(sums, counts, base_sums, base_counts, CFG.bootstrap_iterations, CFG.seed)
        LEDGER.add_ppl("E13", "Language model size", f"E13 {name}", *ppl, base_ppl[0], *d)
    LEDGER.add_ppl("E14", "Language model context", f"E14 context {CFG.max_segments_per_recording} (primary)", *base_ppl, base_ppl[0], 0.0, 0.0, 0.0, np.nan, is_baseline=True)
    for size in [s for s in (4, 8, 16) if s < CFG.max_segments_per_recording]:
        tr, va, te = (chunk_sequences(SEQ[s], size) for s in ["train", "val", "test"])
        if min(len(tr), len(va), len(te)) < 3:
            continue
        trainer, model, (ids, sums, counts) = run_lm_variant(tr, va, te, {}, size)
        common = [i for i in ids if i in set(base_ids)]
        if not common:
            continue
        pos_a = [ids.index(i) for i in common]
        pos_b = [base_ids.index(i) for i in common]
        ppl = ppl_from_sums(sums[pos_a], counts[pos_a], CFG.bootstrap_iterations, CFG.seed)
        d = paired_ppl_difference(sums[pos_a], counts[pos_a], base_sums[pos_b], base_counts[pos_b], CFG.bootstrap_iterations, CFG.seed)
        LEDGER.add_ppl("E14", "Language model context", f"E14 context {size}", *ppl, base_ppl[0], *d)

    token_ppl = {}
    LEDGER.add_ppl("E22", "Token method", f"E22 {VOCAB['method']} (primary)", *base_ppl, base_ppl[0], 0.0, 0.0, 0.0, np.nan, is_baseline=True)
    for method in [m for m in ["kmeans", "gmm_diag", "ward", "vq"] if m != VOCAB["method"]]:
        try:
            model_units = make_units(method, CFG.vq_epochs).fit(Z_ALL[TRAIN_MASK], VOCAB["k"], CFG.seed)
        except Exception as err:
            print("token method failed:", method, str(err)[:80])
            continue
        alt_tokens = model_units.predict(Z_ALL).astype(int) + 1
        alt_builder = SequenceBuilder(seg_meta, alt_tokens, CFG.max_segments_per_recording)
        alt = {s: SequenceBuilder.usable(alt_builder.sequences(s)) for s in ["train", "val", "test"]}
        trainer, model, (ids, sums, counts) = run_lm_variant(alt["train"], alt["val"], alt["test"], {}, CFG.max_segments_per_recording)
        common = [i for i in ids if i in set(base_ids)]
        pa, pb = [ids.index(i) for i in common], [base_ids.index(i) for i in common]
        ppl = ppl_from_sums(sums[pa], counts[pa], CFG.bootstrap_iterations, CFG.seed)
        d = paired_ppl_difference(sums[pa], counts[pa], base_sums[pb], base_counts[pb], CFG.bootstrap_iterations, CFG.seed)
        LEDGER.add_ppl("E22", "Token method", f"E22 {method}", *ppl, base_ppl[0], *d)
print("ledger rows so far:", len(LEDGER.rows))

## 17.5 Experiments taken from earlier parts (E15, E16, E17, E23) and per-species results (E20)

The classifier, token-feature and pseudo-labelling experiments were already run in Part 15 with recording-level folds. Here their results are copied into the same ledger so that every experiment is judged in the same way. E17 compares ABX with and without the "different recordings" rule.

In [ ]:
def add_behaviour_row(exp_id, experiment, variant, result, base, is_baseline=False):
    d = {"diff": 0.0, "lo": 0.0, "hi": 0.0, "p": np.nan}
    if not is_baseline:
        d = paired_difference(LAB_Y, result["predictions"], base["predictions"], LAB_G, N_CLASSES, "macro_f1", CFG.bootstrap_iterations, CFG.seed)
    LEDGER.variants.append({"exp": exp_id, "experiment": experiment, "variant": variant, "behaviour_f1": result["macro_f1"], "behaviour_f1_lo": result["macro_f1_lo"], "behaviour_f1_hi": result["macro_f1_hi"]})
    LEDGER.rows.append({"exp": exp_id, "experiment": experiment, "variant": variant, "metric": "behaviour_f1", "value": result["macro_f1"], "lo": result["macro_f1_lo"], "hi": result["macro_f1_hi"],
                        "baseline": base["macro_f1"], "improvement": d["diff"], "imp_lo": d["lo"], "imp_hi": d["hi"], "p": d["p"], "is_baseline": is_baseline, "note": "recording-level CV, paired"})


for name, result in PROBE_RESULTS.items():
    add_behaviour_row("E15", "Classifier", f"E15 {name}", result, PROBE_RESULTS["logreg"], is_baseline=(name == "logreg"))
for name, result in FEATURE_RESULTS.items():
    add_behaviour_row("E16", "Token and embedding features", f"E16 {name}", result, FEATURE_RESULTS["Adapted"], is_baseline=(name == "Adapted"))
for _, row in pseudo_frame[pseudo_frame["threshold"].notna()].iterrows():
    base_result = PROBE_RESULTS[row["student"]]
    result = PSEUDO_RESULTS[(row["student"], row["threshold"])]
    add_behaviour_row("E23", "Pseudo-labelling", f"E23 {row['student']} thr {row['threshold']}", result, base_result)

held_adapted = EmbeddingSpace.from_config(CFG).fit(SUITE_ADAPTED).transform(SUITE_ADAPTED)[SCORER.held_idx]
safe_score = SCORER.abx.score(held_adapted, SCORER.triplets, n_boot=300)
same_triplets = SCORER.abx.triplets(CFG.abx_triplets, across_groups=False, seed=CFG.seed + 5)
same_score = SCORER.abx.score(held_adapted, same_triplets, n_boot=300)
safe_row = {"variant": "E17 different recordings (primary)", "abx": safe_score[0], "abx_lo": safe_score[1], "abx_hi": safe_score[2]}
same_row = {"variant": "E17 same recording allowed", "abx": same_score[0], "abx_lo": same_score[1], "abx_hi": same_score[2]}
LEDGER.add_variant("E17", "ABX triplet rule", safe_row, safe_row, is_baseline=True, metrics=["abx"])
LEDGER.add_variant("E17", "ABX triplet rule", same_row, safe_row, metrics=["abx"], comparison_note="approximate: different triplets")


def per_species_table():
    held_species = SUITE.species[SCORER.held_idx]
    rng = np.random.default_rng(CFG.seed)
    labels = KMeans(n_clusters=min(SCORER.k, max(2, len(held_adapted) // 10)), n_init=3, random_state=CFG.seed).fit_predict(held_adapted)
    rows = []
    for si, species in enumerate(species_names):
        m = held_species == si
        if m.sum() < 5:
            continue
        pool_a = np.where(m)[0]
        trip = []
        for _ in range(int(CFG.abx_triplets)):
            a = int(rng.choice(pool_a))
            options = pool_a[SCORER.abx.groups[pool_a] != SCORER.abx.groups[a]]
            others = np.where(~m)[0]
            if len(options) and len(others):
                trip.append((a, int(rng.choice(others)), int(rng.choice(options))))
        trip = np.asarray(trip, dtype=np.int64)
        if len(trip) == 0:
            continue
        hits = ABXEvaluator.correct(held_adapted, trip)
        boots = [hits[rng.integers(0, len(hits), len(hits))].mean() for _ in range(200)]
        table = pd.crosstab(labels[m], np.ones(m.sum()))
        rows.append({"species": species, "segments": int(m.sum()), "recordings": int(len(np.unique(SCORER.abx.groups[m]))), "abx_vs_others": float(hits.mean()),
                     "abx_lo": float(np.quantile(boots, 0.025)), "abx_hi": float(np.quantile(boots, 0.975)),
                     "clusters_used": int(len(np.unique(labels[m]))), "top_cluster_share": float(table.max().iloc[0] / m.sum())})
    return pd.DataFrame(rows)


species_table = per_species_table()
TABLES.show(species_table, "per_species", "Table 19. Per-species results on held-out recordings (E20)", precision=3)
RESULTS["per_species"] = species_table

## 17.6 The master results table

The ledger is turned into one table. Only comparisons with a paired or approximate p-value are corrected. The full table with every row is saved as CSV and LaTeX. The table shown here keeps only the rows where the adjusted p-value is below 0.05, so the reader sees at a glance which changes really matter.

In [ ]:
suite_frame = LEDGER.frame()
suite_variants = pd.DataFrame(LEDGER.variants)
full_columns = ["exp", "experiment", "variant", "metric", "value", "lo", "hi", "baseline", "improvement", "imp_lo", "imp_hi", "p", "p_adj", "significant", "note"]
TABLES.show(suite_frame[full_columns], "suite_full", "Table 20. Every experiment, every metric (positive improvement = better than the baseline)", precision=4, render_png=False, display_table=False)
significant_rows = suite_frame[(suite_frame["significant"] == True) & (~suite_frame["is_baseline"])].sort_values(["exp", "metric"])
if len(significant_rows):
    TABLES.show(significant_rows[["exp", "variant", "metric", "value", "improvement", "imp_lo", "imp_hi", "p_adj"]], "suite_significant", "Table 21. Changes that survive the false discovery rate correction (adjusted p < 0.05)", precision=4)
else:
    note_box("No single change survived the false discovery rate correction on this data. This is a finding, not a failure: it says the pipeline is stable against these design choices, or that the test set is too small to detect differences.", "warn")
count_table = (suite_frame[~suite_frame["is_baseline"]].assign(better=lambda d: (d["significant"] == True) & (d["improvement"] > 0), worse=lambda d: (d["significant"] == True) & (d["improvement"] < 0))
               .groupby("exp").agg(comparisons=("variant", "count"), significantly_better=("better", "sum"), significantly_worse=("worse", "sum")).reset_index())
TABLES.show(count_table, "suite_counts", "Table 22. Experiments summarised: how many comparisons were significantly better or worse?", precision=0)
RESULTS["suite"] = suite_frame

## 17.7 Suite figures

The **forest plot** shows the improvement of every variant over its baseline with a 95% interval. Green means significantly better after correction, red means significantly worse, grey means no reliable difference. The **heatmap** shows all feature variants on one colour scale. The **summary bars** count significant changes per experiment. The last figures show seed variation and per-species results.

In [ ]:
def figure_forest():
    metrics = [m for m in ["abx", "silhouette", "behaviour_f1", "ppl"] if ((suite_frame["metric"] == m) & (~suite_frame["is_baseline"]) & suite_frame["improvement"].notna()).any()]
    counts = [int(((suite_frame["metric"] == m) & (~suite_frame["is_baseline"]) & suite_frame["improvement"].notna()).sum()) for m in metrics]
    height = max(6.0, 0.30 * max(counts))
    fig, axes = plt.subplots(1, len(metrics), figsize=(6.6 * len(metrics), height), squeeze=False)
    for ax, metric in zip(axes[0], metrics):
        part = suite_frame[(suite_frame["metric"] == metric) & (~suite_frame["is_baseline"]) & suite_frame["improvement"].notna()].reset_index(drop=True)
        y = np.arange(len(part))[::-1]
        colors = []
        for _, r in part.iterrows():
            if r["significant"] == True:
                colors.append(Palette.GREEN if r["improvement"] > 0 else Palette.RED)
            else:
                colors.append("#9AA0A6")
        lo = np.where(part["imp_lo"].notna(), part["improvement"] - part["imp_lo"], 0).clip(min=0)
        hi = np.where(part["imp_hi"].notna(), part["imp_hi"] - part["improvement"], 0).clip(min=0)
        ax.errorbar(part["improvement"], y, xerr=[lo, hi], fmt="none", ecolor=Palette.INK, capsize=3, linewidth=1.4, zorder=2)
        ax.scatter(part["improvement"], y, s=90, c=colors, edgecolors=Palette.INK, linewidths=1.2, zorder=3)
        ax.axvline(0, color=Palette.ORANGE, linewidth=2.6, zorder=1)
        ax.set_yticks(y)
        ax.set_yticklabels([v[:34] for v in part["variant"]], fontsize=8)
        ax.set_xlabel("Improvement over baseline (95% CI)")
        ax.set_title(METRIC_LABEL[metric])
    handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markeredgecolor=Palette.INK, markersize=11, label=l) for c, l in [(Palette.GREEN, "significantly better"), (Palette.RED, "significantly worse"), ("#9AA0A6", "no reliable difference")]]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0))
    return VAULT.finish(fig, "suite_forest", "Ablation forest plot (Benjamini-Hochberg corrected)", section="Suite", bottom=0.05)


def figure_suite_heatmap():
    frame = suite_variants[suite_variants["abx"].notna()].copy() if "abx" in suite_variants else pd.DataFrame()
    if frame.empty:
        return None
    frame = frame.drop_duplicates("variant").reset_index(drop=True)
    columns = [c for c in ["abx", "silhouette", "davies_bouldin", "behaviour_f1", "species_nmi", "effective_rank"] if c in frame]
    values = frame[columns].astype(float)
    z = (values - values.mean()) / (values.std(ddof=0) + 1e-9)
    if "davies_bouldin" in z:
        z["davies_bouldin"] = -z["davies_bouldin"]
    labels = values.copy().map(lambda v: "" if pd.isna(v) else f"{v:.3f}")
    fig, ax = plt.subplots(figsize=(11, max(6.0, 0.34 * len(frame))))
    sns.heatmap(z.fillna(0), annot=labels, fmt="", cmap=Palette.diverging(), center=0, ax=ax, linewidths=0.8, linecolor=Palette.INK, cbar_kws={"label": "z-score (blue = worse, orange = better)"},
                xticklabels=[METRIC_LABEL.get(c, c.replace("_", " ")) for c in columns], yticklabels=[v[:38] for v in frame["variant"]])
    ax.tick_params(axis="x", rotation=25)
    ax.set_title("All feature variants on one colour scale")
    ax.grid(False)
    return VAULT.finish(fig, "suite_heatmap", "Experiment heatmap", section="Suite")


def figure_suite_counts():
    fig, ax = VAULT.canvas(1, 1, figsize=(11, 5.6))
    x = np.arange(len(count_table))
    neutral = count_table["comparisons"] - count_table["significantly_better"] - count_table["significantly_worse"]
    ax.bar(x, count_table["significantly_better"], color=Palette.GREEN, edgecolor=Palette.INK, linewidth=1.4, label="significantly better")
    ax.bar(x, neutral, bottom=count_table["significantly_better"], color=Palette.BLUE_LIGHT, edgecolor=Palette.INK, linewidth=1.4, label="no reliable difference")
    ax.bar(x, count_table["significantly_worse"], bottom=count_table["significantly_better"] + neutral, color=Palette.RED, edgecolor=Palette.INK, linewidth=1.4, label="significantly worse")
    ax.set_xticks(x)
    ax.set_xticklabels(count_table["exp"])
    ax.set_ylabel("Comparisons")
    ax.set_xlabel("Experiment")
    ax.set_title("Outcome of every comparison per experiment")
    ax.legend()
    return VAULT.finish(fig, "suite_counts", "Suite overview", section="Suite")


def figure_seeds_and_species():
    fig, axes = VAULT.canvas(1, 3, panel=(7.0, 5.0))
    ax = axes[0]
    metrics = ["abx", "silhouette", "behaviour_f1"]
    for j, m in enumerate(metrics):
        vals = seed_frame[m].dropna().to_numpy()
        ax.scatter(np.full(len(vals), j) + np.random.default_rng(j).uniform(-0.08, 0.08, len(vals)), vals, s=130, color=[Palette.ORANGE, Palette.BLUE, Palette.GREEN][j], edgecolors=Palette.INK, linewidths=1.3, zorder=3)
        ax.hlines(vals.mean(), j - 0.25, j + 0.25, color=Palette.INK, linewidth=2.6, zorder=4)
    ax.set_xticks(range(len(metrics)))
    ax.set_xticklabels([METRIC_LABEL[m] for m in metrics], fontsize=9)
    ax.set_ylabel("Value")
    ax.set_title("Adapter seeds (black line = mean)")
    panel_label(ax, "A")
    ax = axes[1]
    order = species_table.sort_values("abx_vs_others")
    bars = ax.barh([s.split(" ")[0] for s in order["species"]], order["abx_vs_others"], color=[SPECIES_COLORS[s] for s in order["species"]], edgecolor=Palette.INK, linewidth=1.4)
    ax.errorbar(order["abx_vs_others"], range(len(order)), xerr=[np.clip(order["abx_vs_others"] - order["abx_lo"], 0, None), np.clip(order["abx_hi"] - order["abx_vs_others"], 0, None)], fmt="none", ecolor=Palette.INK, capsize=4)
    ax.axvline(0.5, color=Palette.RED, linestyle="--", linewidth=2)
    annotate_bars(ax, bars, "{:.3f}", horizontal=True, offset=12)
    ax.set_xlabel("ABX: species against all others")
    ax.set_title("Species distinctness (E20)")
    panel_label(ax, "B")
    ax = axes[2]
    bars = ax.bar([s.split(" ")[0] for s in species_table["species"]], species_table["top_cluster_share"], color=[SPECIES_COLORS[s] for s in species_table["species"]], edgecolor=Palette.INK, linewidth=1.4)
    annotate_bars(ax, bars, "{:.2f}")
    ax.set_ylabel("Share in the most used cluster")
    ax.set_title("Cluster concentration per species")
    ax.tick_params(axis="x", rotation=20)
    panel_label(ax, "C")
    return VAULT.finish(fig, "seeds_species", "Seed stability and per-species results", section="Suite")


figure_forest()
figure_suite_heatmap()
figure_suite_counts()
figure_seeds_and_species()

# Part 18: Data scaling

## 18.1 What does more data buy?

Two scaling curves are measured on the test set, which never changes:

1. **Unsupervised scaling**: the adapter, the vocabulary and the language model are re-trained on a random fraction of the **training recordings** (stratified by species, several random draws per fraction). This shows if collecting more recordings really improves the embedding (ABX, silhouette) and the language model (perplexity).
2. **Label scaling**: the behaviour probe is trained with a fraction of the **labelled training recordings**. This shows how many labels are needed and whether the curve is still rising.

The frozen encoder is not re-trained, only the parts that learn from your recordings.

In [ ]:
class ScalingStudy:
    def __init__(self, cfg):
        self.cfg = cfg
        self.train_recordings = catalog[catalog["split"] == "train"]
        self.train_pos_recordings = seg_meta.loc[TRAIN_POS, "recording_id"].to_numpy()
        self.repeats = cfg.cv_repeats + 1

    def draw(self, fraction, rep):
        rng = np.random.default_rng(self.cfg.seed + 17 * rep + int(fraction * 1000))
        chosen = []
        for _, group in self.train_recordings.groupby("species"):
            n = max(2, int(math.ceil(fraction * len(group))))
            chosen += list(rng.choice(group["recording_id"].to_numpy(), size=min(n, len(group)), replace=False))
        return set(chosen)

    def run_unsupervised(self):
        rows = []
        for fraction in self.cfg.scaling_fractions:
            for rep in range(1 if fraction >= 1.0 else self.repeats):
                rids = self.draw(fraction, rep)
                mask = np.array([r in rids for r in self.train_pos_recordings])
                if mask.sum() < 24:
                    continue
                trainer = ContrastiveAdapterTrainer(RAW_FULL.shape[1], self.cfg, seed=self.cfg.seed + rep)
                trainer.fit(TRAIN_VIEWS[mask], VAL_VIEWS, verbose=False)
                feats = trainer.transform(RAW_FULL)
                tag = f"scale {fraction:.2f} rep {rep}"
                row = SCORER.evaluate(feats[SUITE.positions], tag)
                seg_mask = (seg_meta["split"] == "train").to_numpy() & seg_meta["recording_id"].isin(rids).to_numpy()
                space = EmbeddingSpace.from_config(self.cfg).fit(feats[seg_mask])
                Z = space.transform(feats)
                units = KMeansUnits().fit(Z[seg_mask], VOCAB["k"], self.cfg.seed)
                tokens = units.predict(Z).astype(int) + 1
                builder = SequenceBuilder(seg_meta, tokens, self.cfg.max_segments_per_recording)
                train_seq = {k: v for k, v in SequenceBuilder.usable(builder.sequences("train")).items() if k in rids}
                val_seq, test_seq = (SequenceBuilder.usable(builder.sequences(s)) for s in ["val", "test"])
                ppl = np.nan
                if len(train_seq) >= 5 and len(val_seq) and len(test_seq):
                    lm_trainer = LMTrainer(self.cfg, VOCAB["k"], self.cfg.max_segments_per_recording)
                    model, _ = lm_trainer.fit(train_seq, val_seq, self.cfg.seed)
                    _, nll, msk = lm_trainer.per_recording_nll(model, test_seq)
                    ppl = math.exp(nll.sum() / max(msk.sum(), 1))
                rows.append({"fraction": fraction, "rep": rep, "train_recordings": len(rids), "train_segments": int(seg_mask.sum()), "abx": row["abx"], "silhouette": row["silhouette"],
                             "behaviour_f1": row.get("behaviour_f1", np.nan), "lm_ppl": ppl})
        return pd.DataFrame(rows)

    def run_labels(self):
        rows = []
        for fraction in self.cfg.scaling_fractions:
            def hook(probe_name, train_idx, seed, fraction=fraction):
                rng = np.random.default_rng(seed * 7 + int(fraction * 100))
                keep = np.zeros(len(train_idx), dtype=bool)
                for c in range(N_CLASSES):
                    cls = np.where(LAB_Y[train_idx] == c)[0]
                    groups = np.unique(LAB_G[train_idx][cls])
                    take = rng.choice(groups, size=max(2, int(math.ceil(fraction * len(groups)))), replace=False) if len(groups) else groups
                    keep[cls[np.isin(LAB_G[train_idx][cls], take)]] = True
                return make_probe(probe_name, seed), X_LAB[train_idx][keep], LAB_Y[train_idx][keep]
            result = CV.run(X_LAB, LAB_Y, LAB_G, "logreg", N_CLASSES, fit_hook=hook)
            n_rec = int(round(fraction * len(np.unique(LAB_G))))
            rows.append({"fraction": fraction, "labelled_recordings": n_rec, "macro_f1": result["macro_f1"], "macro_f1_lo": result["macro_f1_lo"], "macro_f1_hi": result["macro_f1_hi"], "accuracy": result["accuracy"]})
        return pd.DataFrame(rows)


scaler_study = ScalingStudy(CFG)
with TIMER.track("scaling study"):
    scale_unsup = CACHE.get_or_compute("scaling_unsup", scaler_study.run_unsupervised)
    scale_labels = CACHE.get_or_compute("scaling_labels", scaler_study.run_labels)
scale_mean = scale_unsup.groupby("fraction").agg(train_recordings=("train_recordings", "mean"), train_segments=("train_segments", "mean"), abx=("abx", "mean"), abx_sd=("abx", "std"),
                                                  silhouette=("silhouette", "mean"), silhouette_sd=("silhouette", "std"), lm_ppl=("lm_ppl", "mean"), lm_ppl_sd=("lm_ppl", "std"),
                                                  behaviour_f1=("behaviour_f1", "mean"), behaviour_f1_sd=("behaviour_f1", "std")).reset_index()
TABLES.show(scale_mean, "scaling_unsupervised", "Table 23. Unsupervised scaling: metrics against the fraction of training recordings (mean and SD over random draws)", precision=3)
TABLES.show(scale_labels, "scaling_labels", "Table 24. Label scaling: behaviour macro-F1 against the fraction of labelled recordings", precision=3)
RESULTS["scaling"] = {"unsupervised": scale_mean, "labels": scale_labels}


def loglinear_fit(x, y):
    ok = np.isfinite(x) & np.isfinite(y) & (x > 0)
    if ok.sum() < 3:
        return None
    slope, intercept, r, p, se = sps.linregress(np.log10(x[ok]), y[ok])
    return slope, intercept, r ** 2, p


def figure_scaling():
    fig, axes = VAULT.canvas(2, 3, panel=(6.6, 4.6))
    specs = [("abx", "abx_sd", "ABX accuracy", 0.5), ("silhouette", "silhouette_sd", "Silhouette", None), ("lm_ppl", "lm_ppl_sd", "Test perplexity (lower is better)", None),
             ("behaviour_f1", "behaviour_f1_sd", "Behaviour macro-F1 (unsupervised parts scaled)", 1 / N_CLASSES)]
    for ax, (col, sd, title, chance), letter in zip(axes.ravel()[:4], specs, "ABCD"):
        x = scale_mean["train_recordings"].to_numpy()
        y = scale_mean[col].to_numpy()
        e = scale_mean[sd].fillna(0).to_numpy()
        ax.errorbar(x, y, yerr=e, marker="o", color=Palette.BLUE, ecolor=Palette.INK, capsize=4, linewidth=2.6)
        ax.fill_between(x, y - e, y + e, color=Palette.BLUE, alpha=0.18)
        fit = loglinear_fit(x, y)
        if fit is not None:
            grid = np.linspace(x.min(), x.max() * 1.6, 60)
            ax.plot(grid, fit[0] * np.log10(grid) + fit[1], color=Palette.ORANGE, linestyle="--", linewidth=2.2, label=f"log-linear: {fit[0]:+.4f} per 10x (R2 = {fit[2]:.2f})")
            ax.legend(fontsize=8)
        if chance is not None:
            ax.axhline(chance, color=Palette.RED, linestyle=":", linewidth=2)
        ax.set_xscale("log")
        ax.set_xlabel("Training recordings")
        ax.set_title(title, fontsize=11.5)
        panel_label(ax, letter)
    ax = axes[1, 1]
    ax.plot(scale_labels["labelled_recordings"], scale_labels["macro_f1"], marker="o", color=Palette.ORANGE)
    ax.fill_between(scale_labels["labelled_recordings"], scale_labels["macro_f1_lo"], scale_labels["macro_f1_hi"], color=Palette.ORANGE, alpha=0.2)
    ax.axhline(1 / N_CLASSES, color=Palette.RED, linestyle=":", linewidth=2)
    ax.set_xscale("log")
    ax.set_xlabel("Labelled recordings (all folds)")
    ax.set_ylabel("Macro-F1")
    ax.set_title("Label scaling curve")
    panel_label(ax, "E")
    ax = axes[1, 2]
    ax.plot(scale_mean["train_segments"], scale_mean["lm_ppl"], marker="s", color=Palette.PURPLE)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Training segments")
    ax.set_ylabel("Test perplexity (log scale)")
    ax.set_title("Language model against corpus size")
    panel_label(ax, "F")
    return VAULT.finish(fig, "scaling", "Data scaling study (fixed test recordings)", section="Scaling")


figure_scaling()

## 18.2 Registry of all earlier runs

Every number below was **copied from the saved outputs of the earlier notebooks** (Run 1 to Run 10 and the fine-tuned HuBERT notebook). They are shown next to this notebook's results so the whole history is in one table and one figure.

Please read the last column. Behaviour accuracy in the old runs came from **random segment-level folds** and, in Runs 1 to 9, ABX and behaviour values were pasted into the notebook as fallback constants. So the old accuracies are **not comparable** with the recording-level results of this notebook, and they were reported without confidence intervals.

In [ ]:
REGISTRY_COLUMNS = ["run", "recordings", "segments", "silhouette", "davies_bouldin", "calinski_harabasz", "lm_perplexity", "abx_encoder", "abx_mfcc", "behaviour_acc", "behaviour_n", "behaviour_method", "protocol"]
REGISTRY_ROWS = [
    ("Reference run", 150, 4234, 0.0339, np.nan, np.nan, 25.99, 0.580, 0.599, 0.6327, np.nan, "not recorded", "segment folds"),
    ("Run 1", 150, 3832, 0.0806, 2.341, 212.9, 7.639, 0.572, 0.5733, 0.4444, 45, "pooled probe", "segment folds, fallback constants"),
    ("Run 2", 300, 9408, 0.0809, 2.406, 605.8, 6.452, 0.5533, 0.5607, 0.5333, 45, "pooled probe", "segment folds, fallback constants"),
    ("Run 3", 900, 27597, 0.0733, 2.408, 1827.7, 6.338, 0.5747, 0.5607, 0.3778, 45, "pooled probe", "segment folds, fallback constants"),
    ("Run 4", 900, 27597, 0.0733, 2.408, 1827.7, 6.338, 0.5747, 0.5607, 0.4778, 180, "MLP 128-64", "segment folds, fallback constants"),
    ("Run 5", 900, 27597, 0.0523, 3.129, 1418.6, 5.450, 0.566, 0.5607, 0.6100, 300, "supervised adapter MLP", "segment folds, fallback constants"),
    ("Run 6", 900, 27597, 0.0523, 3.129, 1418.6, 5.418, 0.566, 0.5607, 0.6489, 450, "deep adapter", "segment folds, fallback constants"),
    ("Run 7", 900, 27597, 0.0440, 3.247, 1054.8, 8.081, 0.566, 0.5607, 0.6267, 450, "ensemble", "segment folds, fallback constants"),
    ("Run 8", 900, 27597, 0.0523, 3.129, 1418.6, 5.419, 0.566, 0.5607, 0.6821, 475, "LSTM + pseudo-labels", "segment folds, early stopping on test fold"),
    ("Run 9", 1200, 37401, 0.0519, 3.035, 1453.6, 7.122, 0.566, 0.5607, 0.6358, np.nan, "XGBoost", "segment folds"),
    ("Run 10", 1500, 45961, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.7052, np.nan, "MLP + pseudo-labels", "segment folds, threshold chosen on pseudo-labelled CV"),
    ("HuBERT notebook", np.nan, np.nan, 0.0548, np.nan, np.nan, 16.35, 0.5888, 0.5502, 0.545, 66, "pooled probe", "segment folds"),
]
registry = pd.DataFrame(REGISTRY_ROWS, columns=REGISTRY_COLUMNS)
TABLES.show(registry, "registry", "Table 25. Logged results of all earlier notebooks (copied from saved outputs; not recomputed)", precision=4)
RESULTS["registry"] = registry


def figure_registry():
    fig, axes = VAULT.canvas(1, 3, panel=(7.2, 5.0))
    old = registry.dropna(subset=["recordings"])
    ax = axes[0]
    ax.plot(old["recordings"], old["silhouette"], marker="o", color=Palette.ORANGE, label="earlier runs")
    ax.scatter([RESULTS["dataset"]["recordings"]], [E01_rows["adapted"]["silhouette"]], s=220, marker="*", color=Palette.BLUE, edgecolors=Palette.INK, linewidths=1.6, zorder=5, label="this notebook (adapted, held-out)")
    ax.set_xscale("log")
    ax.set_xlabel("Recordings used")
    ax.set_ylabel("Silhouette")
    ax.set_title("Silhouette across runs")
    ax.legend(fontsize=8.5)
    panel_label(ax, "A")
    ax = axes[1]
    gaps = registry.dropna(subset=["abx_encoder", "abx_mfcc"]).copy()
    gaps["gap"] = gaps["abx_encoder"] - gaps["abx_mfcc"]
    colors = [Palette.ORANGE if g >= 0 else Palette.RED for g in gaps["gap"]]
    bars = ax.barh(gaps["run"], gaps["gap"], color=colors, edgecolor=Palette.INK, linewidth=1.4)
    own_gap = E01_rows["adapted"]["abx"] - E01_rows["mfcc"]["abx"]
    ax.barh(["this notebook"], [own_gap], color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.4)
    ax.axvline(0, color=Palette.INK, linewidth=2)
    ax.set_xlabel("ABX(encoder) minus ABX(MFCC)")
    ax.set_title("Did the encoder beat MFCC?")
    ax.invert_yaxis()
    panel_label(ax, "B")
    ax = axes[2]
    acc = registry.dropna(subset=["behaviour_acc"])
    ax.bar(range(len(acc)), acc["behaviour_acc"], color=Palette.ORANGE, edgecolor=Palette.INK, linewidth=1.4, hatch="//", label="earlier runs (segment folds)")
    best = behaviour_master.sort_values("accuracy", ascending=False).iloc[0]
    ax.bar([len(acc)], [best["accuracy"]], color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.5, label="this notebook (recording-level folds)")
    ax.errorbar([len(acc)], [best["accuracy"]], yerr=[[max(best["accuracy"] - best["accuracy_lo"], 0)], [max(best["accuracy_hi"] - best["accuracy"], 0)]], fmt="none", ecolor=Palette.INK, capsize=6, linewidth=2)
    ax.axhline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2.2, label="chance")
    ax.set_xticks(range(len(acc) + 1))
    ax.set_xticklabels(list(acc["run"]) + ["this run"], rotation=60, ha="right", fontsize=8.5)
    ax.set_ylabel("Behaviour accuracy")
    ax.set_title("Behaviour accuracy (different protocols)")
    ax.legend(fontsize=8, loc="upper left")
    panel_label(ax, "C")
    return VAULT.finish(fig, "registry", "Earlier runs against this notebook", section="History")


figure_registry()

# Part 19: Statistics, claim checker and exports

## 19.1 Pairwise significance between classifiers

A bar chart with overlapping intervals does not say which classifiers really differ. Here every pair of classifiers is compared with a **paired bootstrap over recordings**, and the p-values are corrected with Benjamini-Hochberg. Orange cells mean the row classifier is better than the column classifier.

In [ ]:
def pairwise_matrix(results, y, groups, n_classes, metric="macro_f1"):
    names = list(results)
    diff = np.zeros((len(names), len(names)))
    pvals = np.full((len(names), len(names)), np.nan)
    pairs = []
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if i < j:
                d = paired_difference(y, results[a]["predictions"], results[b]["predictions"], groups, n_classes, metric, min(CFG.bootstrap_iterations, 300), CFG.seed)
                diff[i, j], diff[j, i] = d["diff"], -d["diff"]
                pairs.append((i, j, d["p"]))
    adjusted = bh_adjust([p for _, _, p in pairs])
    for (i, j, _), p in zip(pairs, adjusted):
        pvals[i, j] = pvals[j, i] = p
    return names, diff, pvals


with TIMER.track("pairwise significance"):
    pair_names, pair_diff, pair_p = pairwise_matrix(PROBE_RESULTS, LAB_Y, LAB_G, N_CLASSES)


def figure_pairwise():
    stars = np.array([["" if (i == j or not np.isfinite(pair_p[i, j])) else ("***" if pair_p[i, j] < 0.001 else "**" if pair_p[i, j] < 0.01 else "*" if pair_p[i, j] < 0.05 else "ns") for j in range(len(pair_names))] for i in range(len(pair_names))])
    labels = np.array([[("" if i == j else f"{pair_diff[i, j]:+.2f}\n{stars[i, j]}") for j in range(len(pair_names))] for i in range(len(pair_names))])
    fig, ax = plt.subplots(figsize=(10.5, 8.2))
    limit = max(0.05, np.abs(pair_diff).max())
    sns.heatmap(pair_diff, annot=labels, fmt="", cmap=Palette.diverging(), center=0, vmin=-limit, vmax=limit, ax=ax, linewidths=1.2, linecolor=Palette.INK, xticklabels=pair_names, yticklabels=pair_names,
                cbar_kws={"label": "macro-F1 difference (row minus column)"})
    ax.tick_params(axis="x", rotation=35)
    ax.set_title("Paired bootstrap between classifiers (BH-corrected: * < .05, ** < .01, *** < .001)")
    ax.grid(False)
    return VAULT.finish(fig, "pairwise_significance", "Which classifiers really differ?", section="Statistics")


figure_pairwise()

## 19.2 Claim checker

A paper should only claim what the numbers can carry. `ClaimChecker` turns the results of this run into a short list of claims and gives each one a verdict using fixed rules:

- **Supported**: the 95% interval excludes the null value, or the corrected p-value is below 0.05.
- **Not supported**: the interval is entirely on the wrong side.
- **Inconclusive**: the interval includes the null value. With small test sets this is common, and it is not the same as "no effect".

In [ ]:
class ClaimChecker:
    def __init__(self):
        self.rows = []

    def add(self, claim, evidence, verdict):
        self.rows.append({"claim": claim, "evidence": evidence, "verdict": verdict})

    @staticmethod
    def interval_verdict(lo, hi, null):
        if not (np.isfinite(lo) and np.isfinite(hi)):
            return "inconclusive"
        if lo > null:
            return "supported"
        if hi < null:
            return "not supported"
        return "inconclusive"

    def frame(self):
        return pd.DataFrame(self.rows)


checker = ClaimChecker()
best_learned = heldout_frame[~heldout_frame["variant"].isin(["MFCC baseline", "Adapted (primary + adapter)"])].sort_values("abx", ascending=False).iloc[0]
mfcc_row = heldout_frame[heldout_frame["variant"] == "MFCC baseline"].iloc[0]
checker.add("Species can be told apart above chance by the primary representation (ABX > 0.5)", f"ABX = {E01_rows['adapted']['abx']:.3f} [{E01_rows['adapted']['abx_lo']:.3f}, {E01_rows['adapted']['abx_hi']:.3f}]",
            ClaimChecker.interval_verdict(E01_rows["adapted"]["abx_lo"], E01_rows["adapted"]["abx_hi"], 0.5))
mfcc_gap = suite_frame[(suite_frame["variant"] == "E01 MFCC") & (suite_frame["metric"] == "abx")]
if len(mfcc_gap):
    imp = -float(mfcc_gap["improvement"].iloc[0])
    checker.add("The adapted representation beats MFCC on ABX", f"ABX difference (adapted minus MFCC) = {imp:+.3f}, adjusted p = {float(mfcc_gap['p_adj'].iloc[0]):.3g}",
                "supported" if (imp > 0 and float(mfcc_gap["p_adj"].iloc[0]) < 0.05) else ("not supported" if (imp < 0 and float(mfcc_gap["p_adj"].iloc[0]) < 0.05) else "inconclusive"))
raw_gap = suite_frame[(suite_frame["variant"] == "E01 raw encoder") & (suite_frame["metric"] == "abx")]
if len(raw_gap):
    imp = -float(raw_gap["improvement"].iloc[0])
    checker.add("The contrastive adapter improves ABX over the raw encoder", f"ABX difference (adapted minus raw) = {imp:+.3f}, adjusted p = {float(raw_gap['p_adj'].iloc[0]):.3g}",
                "supported" if (imp > 0 and float(raw_gap["p_adj"].iloc[0]) < 0.05) else ("not supported" if (imp < 0 and float(raw_gap["p_adj"].iloc[0]) < 0.05) else "inconclusive"))
order_effect = RESULTS.get("lm_order_effect")
if order_effect:
    checker.add("Token order carries information (real order beats shuffled order)", f"perplexity gain = {order_effect['ppl_gain_from_order']:.3f}, paired t p = {order_effect['paired_t_p']:.3g} over {len(CFG.lm_seeds)} seeds",
                "supported" if (order_effect["ppl_gain_from_order"] > 0 and order_effect["paired_t_p"] < 0.05) else "inconclusive")
best_beh = behaviour_master.sort_values("macro_f1", ascending=False).iloc[0]
checker.add("Weak behaviour labels are predictable above chance with recording-level folds", f"best macro-F1 = {best_beh['macro_f1']:.3f} [{best_beh['macro_f1_lo']:.3f}, {best_beh['macro_f1_hi']:.3f}] ({best_beh['method']})",
            ClaimChecker.interval_verdict(best_beh["macro_f1_lo"], best_beh["macro_f1_hi"], 1 / N_CLASSES))
assoc = RESULTS.get("association")
if assoc:
    checker.add("Token usage is associated with weak behaviour (recording-level permutation test)", f"Cramer's V = {assoc['cramers_v']:.3f}, p = {assoc['permutation_p']:.4f}", "supported" if assoc["permutation_p"] < 0.05 else "inconclusive")
pseudo_sig = suite_frame[(suite_frame["exp"] == "E23") & (suite_frame["significant"] == True)]
checker.add("Pseudo-labelling improves behaviour classification", "no threshold gave a corrected significant gain" if pseudo_sig.empty else f"{len(pseudo_sig)} thresholds significant", "inconclusive" if pseudo_sig.empty else ("supported" if (pseudo_sig["improvement"] > 0).any() else "not supported"))
inflation = leak_frame["inflation_acc"].mean()
checker.add("Random segment folds inflate accuracy compared with recording-level folds", f"mean accuracy inflation = {inflation:+.3f} across {len(leak_frame)} classifiers ({sibling * 100:.0f}% of naive test segments had a sibling in training)", "supported" if inflation > 0.03 else "inconclusive")
aves = heldout_frame[heldout_frame["variant"].str.contains("birdaves", case=False, na=False)]
speech = heldout_frame[heldout_frame["variant"].isin(["hubert-base", "wav2vec2-base", "wav2vec2-large-lv60"])]
if len(aves) and len(speech):
    a, s = aves.sort_values("abx", ascending=False).iloc[0], speech.sort_values("abx", ascending=False).iloc[0]
    verdict = "supported" if a["abx_lo"] > s["abx_hi"] else ("not supported" if a["abx_hi"] < s["abx_lo"] else "inconclusive")
    checker.add("A bird-trained encoder beats speech encoders on ABX", f"{a['variant']} {a['abx']:.3f} [{a['abx_lo']:.3f}, {a['abx_hi']:.3f}] against {s['variant']} {s['abx']:.3f} [{s['abx_lo']:.3f}, {s['abx_hi']:.3f}] (non-overlapping intervals are required)", verdict)
claims = checker.frame()
TABLES.show(claims, "claim_checker", "Table 26. Claims and the evidence behind them", precision=3)
RESULTS["claims"] = claims
supported = int((claims["verdict"] == "supported").sum())
note_box(f"<b>{supported} of {len(claims)} claims are supported</b> by this run. Anything marked inconclusive should be written in the paper as a trend, not as a result. Weak labels come from uploader text, so behaviour results are weak-label results.", "good" if supported >= len(claims) / 2 else "warn")

## 19.3 Results dashboard

One page with the headline numbers and six mini charts. This is the figure to put at the top of a talk or a poster.

In [ ]:
def figure_dashboard():
    fig = plt.figure(figsize=(20, 12.2))
    grid = GridSpec(3, 6, figure=fig, height_ratios=[0.55, 1.0, 1.0], hspace=0.55, wspace=0.62)
    lm_real = lm_table.loc[lm_table["model"] == "Transformer (real order)", "test_ppl"].iloc[0]
    lm_shuf = lm_table.loc[lm_table["model"] == "Transformer (shuffled order)", "test_ppl"].iloc[0]
    tiles = [("Recordings", f"{RESULTS['dataset']['recordings']}", Palette.ORANGE), ("Segments", f"{RESULTS['dataset']['segments']:,}", Palette.BLUE),
             ("Primary encoder", PRIMARY["encoder"], Palette.ORANGE), ("Vocabulary", f"{VOCAB['method']} K={VOCAB['k']}", Palette.BLUE),
             ("Held-out ABX", f"{E01_rows['adapted']['abx']:.3f}", Palette.ORANGE), ("Best behaviour F1", f"{best_beh['macro_f1']:.3f}", Palette.BLUE)]
    for i, (title, value, color) in enumerate(tiles):
        ax = fig.add_subplot(grid[0, i])
        ax.set_facecolor(color)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        ax.text(0.5, 0.68, value, ha="center", va="center", fontsize=21 if len(value) < 12 else 14, fontweight="bold", color="white")
        ax.text(0.5, 0.2, title, ha="center", va="center", fontsize=12, color="white")
        ax.grid(False)
    ax = fig.add_subplot(grid[1, 0:2])
    order = heldout_frame.sort_values("abx")
    ax.barh(order["variant"], order["abx"], color=[Palette.GREEN if v == "MFCC baseline" else (Palette.ORANGE if "Adapted" in v else Palette.BLUE) for v in order["variant"]], edgecolor=Palette.INK, linewidth=1.3)
    ax.axvline(0.5, color=Palette.RED, linestyle="--", linewidth=2)
    ax.set_title("ABX by representation")
    ax.tick_params(axis="y", labelsize=8)
    ax = fig.add_subplot(grid[1, 2:4])
    top = behaviour_master.sort_values("macro_f1").tail(6)
    ax.barh(top["method"], top["macro_f1"], color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.3)
    ax.errorbar(top["macro_f1"], range(len(top)), xerr=[np.clip(top["macro_f1"] - top["macro_f1_lo"], 0, None), np.clip(top["macro_f1_hi"] - top["macro_f1"], 0, None)], fmt="none", ecolor=Palette.INK, capsize=3)
    ax.axvline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2)
    ax.set_title("Behaviour macro-F1 (top 6)")
    ax.tick_params(axis="y", labelsize=7.5)
    ax = fig.add_subplot(grid[1, 4:6])
    names = ["uniform", "unigram", "bigram", "transformer\nshuffled", "transformer\nreal"]
    values = [K_TOKENS]
    for label in ["Unigram", "Bigram"]:
        values.append(lm_table.loc[lm_table["model"] == label, "test_ppl"].iloc[0])
    values += [lm_shuf, lm_real]
    ax.bar(names, values, color=[Palette.RED, Palette.PURPLE, Palette.PURPLE, Palette.ORANGE, Palette.BLUE], edgecolor=Palette.INK, linewidth=1.3)
    ax.set_title("Test perplexity")
    ax.tick_params(axis="x", labelsize=8)
    ax = fig.add_subplot(grid[2, 0:2])
    x = np.arange(len(leak_frame))
    ax.bar(x - 0.2, leak_frame["naive_acc"], 0.4, color=Palette.ORANGE, edgecolor=Palette.INK, linewidth=1.3, label="naive folds")
    ax.bar(x + 0.2, leak_frame["grouped_acc"], 0.4, color=Palette.BLUE, edgecolor=Palette.INK, linewidth=1.3, label="recording folds")
    ax.axhline(1 / N_CLASSES, color=Palette.RED, linestyle="--", linewidth=2)
    ax.set_xticks(x)
    ax.set_xticklabels(leak_frame["classifier"], rotation=25, fontsize=8)
    ax.set_title("Leakage audit (accuracy)")
    ax.legend(fontsize=8)
    ax = fig.add_subplot(grid[2, 2:4])
    ax.errorbar(scale_mean["train_recordings"], scale_mean["abx"], yerr=scale_mean["abx_sd"].fillna(0), marker="o", color=Palette.BLUE, capsize=4, ecolor=Palette.INK)
    ax.axhline(0.5, color=Palette.RED, linestyle="--", linewidth=2)
    ax.set_xscale("log")
    ax.set_xlabel("Training recordings")
    ax.set_title("ABX against corpus size")
    ax = fig.add_subplot(grid[2, 4:6])
    verdicts = claims["verdict"].value_counts().reindex(["supported", "inconclusive", "not supported"]).fillna(0)
    ax.bar(verdicts.index, verdicts.values, color=[Palette.GREEN, Palette.ORANGE, Palette.RED], edgecolor=Palette.INK, linewidth=1.3)
    ax.set_title("Claim checker")
    fig.suptitle(f"BioLingua results dashboard ({CFG.preset} preset)", fontsize=20, fontweight="bold", color=Palette.ORANGE_DARK, y=0.98)
    fig.subplots_adjust(top=0.9, bottom=0.07, left=0.06, right=0.97)
    border = mpatches.Rectangle((0.002, 0.002), 0.996, 0.996, transform=fig.transFigure, fill=False, edgecolor=Palette.INK, linewidth=3.0)
    fig.add_artist(border)
    index = len(VAULT.records) + 1
    saved = []
    for fmt in VAULT.formats:
        path = VAULT.out_dir / f"fig{index:02d}_dashboard.{fmt}"
        fig.savefig(path, dpi=VAULT.export_dpi if fmt == "png" else None, facecolor="white")
        saved.append(str(path))
    VAULT.records.append({"figure": index, "name": "dashboard", "title": "Results dashboard", "caption": "", "section": "Summary", "files": saved})
    plt.show()
    plt.close(fig)


figure_dashboard()

## 19.4 Runtime, manifest and export

The manifest records everything needed to repeat the run: configuration, package versions, seeds, the primary encoder configuration, timings and the main results. All figures, tables and result files are packed into one zip file in the output folder.

In [ ]:
timing = TIMER.frame()
fig, ax = VAULT.canvas(1, 1, figsize=(10, max(4.5, 0.36 * len(timing))))
ordered = timing.sort_values("seconds")
bars = ax.barh(ordered["stage"], ordered["seconds"] / 60.0, color=Palette.token_colors(len(ordered)), edgecolor=Palette.INK, linewidth=1.3)
annotate_bars(ax, bars, "{:.1f}", horizontal=True, fontsize=8.5)
ax.set_xlabel("Minutes")
ax.set_title("Where the time went")
VAULT.finish(fig, "runtime", "Runtime per stage", section="Summary")
TABLES.show(timing.assign(minutes=lambda d: d["seconds"] / 60.0)[["stage", "seconds", "minutes"]], "runtime", "Table 27. Runtime per stage", precision=1, render_png=False)

manifest = {
    "project": "BioLingua",
    "preset": CFG.preset,
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(CFG).items()},
    "primary_encoder": PRIMARY,
    "vocabulary": VOCAB,
    "device": str(DEVICE),
    "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__, "sklearn": __import__("sklearn").__version__,
                 "librosa": librosa.__version__, "matplotlib": matplotlib.__version__},
    "dataset": RESULTS["dataset"],
    "headline": {"abx": E01_rows["adapted"]["abx"], "abx_ci": [E01_rows["adapted"]["abx_lo"], E01_rows["adapted"]["abx_hi"]], "best_behaviour_method": best_beh["method"],
                 "best_behaviour_macro_f1": best_beh["macro_f1"], "best_behaviour_ci": [best_beh["macro_f1_lo"], best_beh["macro_f1_hi"]], "leakage_inflation_accuracy": float(inflation),
                 "lm_perplexity_real": float(lm_table.loc[lm_table["model"] == "Transformer (real order)", "test_ppl"].iloc[0])},
    "claims": claims.to_dict(orient="records"),
    "figures": [{"figure": r["figure"], "name": r["name"], "title": r["title"], "section": r["section"]} for r in VAULT.records],
    "tables": TABLES.records,
}
with open(PATHS.results / "run_manifest.json", "w") as handle:
    json.dump(manifest, handle, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o))
VAULT.index_frame().to_csv(PATHS.results / "figure_index.csv", index=False)
TABLES.index_frame().to_csv(PATHS.results / "table_index.csv", index=False)
suite_frame.to_csv(PATHS.results / "suite_ledger.csv", index=False)
with open(PATHS.results / "results.pkl", "wb") as handle:
    pickle.dump({k: v for k, v in RESULTS.items() if k != "suite"}, handle)
np.save(PATHS.results / "token_ids.npy", TOKENS)
seg_meta[["segment_id", "recording_id", "species", "segment_index", "split", "token"]].to_csv(PATHS.results / "segment_tokens.csv", index=False)

archive = PATHS.root.parent / f"biolingua_{CFG.preset}_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [PATHS.figures, PATHS.tables, PATHS.results]:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                zf.write(path, path.relative_to(PATHS.root))
size_mb = archive.stat().st_size / 1e6
print(f"figures: {len(VAULT.records)} | tables: {len(TABLES.records)} | archive: {archive} ({size_mb:.1f} MB)")
display(VAULT.index_frame().head(80))

## 19.5 Where each result goes in the paper

| Paper section | Use these outputs |
|---|---|
| Dataset | corpus overview, label coverage, acoustic gallery, Tables 2 to 4 |
| Method: representation | encoder comparison, layer-wise, pooling, adapter effect |
| Method: unit discovery | vocabulary sweep, method comparison, token diagnostics, unit gallery |
| Results: species structure | held-out encoder table, ABX figure, embedding maps |
| Results: sequence structure | language model figure with controls, transitions, barcodes |
| Results: behaviour | classifier table, feature comparison, master behaviour figure, pairwise significance, association test |
| Methodology and honesty | leakage audit, claim checker, registry of earlier runs |
| Ablations | suite forest plot, master ledger (CSV and LaTeX), seeds and per-species figure |
| Scaling | scaling figure, label scaling table |
| Supplement | full configuration table, runtime, manifest |

**How to describe the behaviour results honestly**

- Say "weak-label behaviour", because the labels come from uploader text.
- Report the recording-level numbers with their intervals, and say how many labelled recordings each class has.
- Report inconclusive results as trends.
- State that the earlier runs used segment-level folds and that this notebook shows how much that inflated accuracy.

**Limits that a reviewer will point out**

- Only five species from one source.
- The labelled test set is small, so intervals can be wide.
- ABX and clustering scores are related to species, not to a manual annotation of units.
- Recordings from the same recordist or place can share background noise; a location-level split would be stricter than the recording-level split used here.

In [ ]:
import shutil, base64
from pathlib import Path
from IPython.display import HTML, display

existing_zip = PATHS.root.parent / f"biolingua_{CFG.preset}_results.zip"
if existing_zip.exists():
    final_zip = existing_zip
else:
    final_zip = Path(shutil.make_archive(str(PATHS.root.parent / f"biolingua_{CFG.preset}_all_outputs"), "zip", root_dir=str(PATHS.root)))

size_mb = final_zip.stat().st_size / 1e6
print(f"zip ready: {final_zip.name} ({size_mb:.1f} MB)")

if size_mb > 150:
    print("File is large; auto-download may fail in the browser. Use the Output tab to download instead.")
else:
    with open(final_zip, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    html = f"""
    <a id="auto_dl_link" href="data:application/zip;base64,{b64}" download="{final_zip.name}"></a>
    <script>document.getElementById('auto_dl_link').click();</script>
    """
    display(HTML(html))
    print("Download should start automatically in your browser.")